In [ ]:
from pathlib import Path
import os
import sys
_release_root = Path(os.environ.get('AVATAR_RELEASE_ROOT', Path.cwd())).resolve()
while not (_release_root / 'avatar_paths.py').is_file():
    if _release_root == _release_root.parent:
        raise RuntimeError('Open this notebook from the code package directory.')
    _release_root = _release_root.parent
if str(_release_root) not in sys.path:
    sys.path.insert(0, str(_release_root))
from avatar_paths import project_path, get_value, get_column, get_path, get_label


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'Arial', 'font.size': 12.0, 'axes.titlesize': 14.5, 'axes.labelsize': 12.5, 'xtick.labelsize': 11.5, 'ytick.labelsize': 11.5, 'legend.fontsize': 11.0, 'figure.titlesize': 15.0, 'text.color': '#1F1F1F', 'axes.labelcolor': '#1F1F1F', 'axes.titlecolor': '#1F1F1F', 'xtick.color': '#1F1F1F', 'ytick.color': '#1F1F1F', 'axes.edgecolor': '#2B2B2B', 'axes.grid': False, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none'})
from scipy.stats import f_oneway
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
project_dir = str(project_path(get_value('science.paths.directory_008', 'directory_008')))
input_dir = os.path.join(project_dir, get_value('science.paths.directory_009', 'directory_009'))
legacy_high_dim_dir = str(project_path(get_value('science.paths.directory_052', 'directory_052')))
output_root = os.path.join(project_dir, get_value('science.paths.directory_053', 'directory_053'))
LABEL_FILENAME = get_value('science.tables.outcomes_001', 'outcomes_001.csv')
PREFERRED_RESPONSE_COLUMN = 'Response'
label_path = os.path.join(input_dir, LABEL_FILENAME)
clinical_path = str(project_path(get_value('science.tables.clinical_006', 'clinical_006.xlsx')))
files = {'Metagenomics': {'path': os.path.join(input_dir, get_value('science.tables.features_001', 'features_001.csv.gz')), 'method': 'pca_distance', 'already_scaled': True}, 'Species': {'path': os.path.join(input_dir, get_value('science.tables.result_050', 'result_050.csv.gz')), 'method': 'braycurtis', 'already_scaled': False}, 'Radiomics': {'path': os.path.join(input_dir, get_value('science.tables.features_002', 'features_002.csv.gz')), 'method': 'pca_distance', 'already_scaled': True}, 'Blood': {'path': os.path.join(input_dir, get_value('science.tables.features_003', 'features_003.csv.gz')), 'method': 'pca_distance', 'already_scaled': True}, 'Urine': {'path': os.path.join(input_dir, get_value('science.tables.features_004', 'features_004.csv.gz')), 'method': 'pca_distance', 'already_scaled': True}}
FUSION_WEIGHTS = {'Metagenomics': 0.5, 'Species': 0.5, 'Radiomics': 1.0, 'Blood': 1.0, 'Urine': 1.0}
ADJUSTMENT_COVARIATES = ['TBS_z', 'PVTT', 'ALBI_grade2', 'NLR_z']
PFS_LANDMARK_MONTHS = 6.0
OS_LANDMARK_MONTHS = 12.0
SURVIVAL_TIME_UNIT = 'months'

def months_to_survival_time(months):
    if SURVIVAL_TIME_UNIT == 'months':
        return float(months)
    if SURVIVAL_TIME_UNIT == 'days':
        return float(months) * (365.25 / 12.0)
    raise ValueError(f"Unsupported SURVIVAL_TIME_UNIT={SURVIVAL_TIME_UNIT!r}; use 'days' or 'months'.")
PFS_LANDMARK_VALUE = months_to_survival_time(PFS_LANDMARK_MONTHS)
OS_LANDMARK_VALUE = months_to_survival_time(OS_LANDMARK_MONTHS)
PCA_VARIANCE_TARGET = 0.7
PCA_MAX_COMPONENTS = None
PCA_MIN_COMPONENTS = 2
N_PERM = 4999
RANDOM_STATE = 42
out_dir = os.path.join(output_root, get_value('science.paths.directory_054', 'directory_054'))
os.makedirs(out_dir, exist_ok=True)
required_paths = [label_path, clinical_path] + [v['path'] for v in files.values()]
missing_paths = [p for p in required_paths if not os.path.exists(p)]
if missing_paths:
    print('\nMissing required files:')
    for path in missing_paths:
        print(' -', path)
    raise FileNotFoundError('Please correct the file paths above.')

def normalize_sample_id(x):
    if pd.isna(x):
        return np.nan
    value = str(x).strip().lower()
    if value.endswith('.0') and value[:-2].isdigit():
        value = value[:-2]
    return value

def read_matrix(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    df.columns = [str(c).strip() for c in df.columns]
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    sample_col = None
    for candidate in ['sample', 'sample_id', 'sample id', 'id']:
        if candidate in lower_map:
            sample_col = lower_map[candidate]
            break
    if sample_col is None:
        sample_col = df.columns[0]
    df = df.rename(columns={sample_col: 'sample'})
    df['sample'] = df['sample'].map(normalize_sample_id)
    df = df.dropna(subset=['sample']).drop_duplicates(subset='sample', keep='first').set_index('sample')
    df = df.apply(pd.to_numeric, errors='coerce')
    return df

def read_labels(path):
    raw = pd.read_csv(path, encoding='utf-8-sig')
    raw.columns = [str(c).strip() for c in raw.columns]
    lower_map = {str(c).strip().lower(): c for c in raw.columns}
    sample_col = None
    for candidate in ['sample id', 'sample_id', 'sample', 'id']:
        if candidate in lower_map:
            sample_col = lower_map[candidate]
            break
    if sample_col is None:
        raise ValueError('Cannot find the sample ID column.')
    response_col = None
    preferred_response = str(PREFERRED_RESPONSE_COLUMN).strip().lower()
    response_candidates = list(dict.fromkeys([preferred_response, 'response', 'response', 'group']))
    for candidate in response_candidates:
        if candidate in lower_map:
            response_col = lower_map[candidate]
            break
    if response_col is None:
        raise ValueError('Cannot find a Response column. Checked: ' + ', '.join(response_candidates))
    
    print(f'[Outcome audit] Response column used: {response_col}')
    pass
    labels = pd.DataFrame({'sample': raw[sample_col].map(normalize_sample_id), 'Response': pd.to_numeric(raw[response_col], errors='coerce'), 'PFS_time': pd.to_numeric(raw['PFS_time'], errors='coerce'), 'PFS_status': pd.to_numeric(raw['PFS_status'], errors='coerce'), 'OS_time': pd.to_numeric(raw['OS_time'], errors='coerce'), 'OS_status': pd.to_numeric(raw['OS_status'], errors='coerce')})
    labels = labels.dropna(subset=['sample']).drop_duplicates(subset='sample', keep='first').set_index('sample')
    required = ['Response', 'PFS_time', 'PFS_status', 'OS_time', 'OS_status']
    if labels[required].isna().any().any():
        raise ValueError('Missing values remain in the required outcomes.')
    labels['Response'] = labels['Response'].astype(int)
    labels['PFS_status'] = labels['PFS_status'].astype(int)
    labels['OS_status'] = labels['OS_status'].astype(int)
    print(f"[Outcome audit] n={len(labels)}, Response={labels['Response'].value_counts().sort_index().to_dict()}, PFS events={int(labels['PFS_status'].sum())}, OS events={int(labels['OS_status'].sum())}")
    return labels

def bh_fdr(pvals):
    values = np.asarray(pvals, dtype=float)
    qvals = np.full(values.shape, np.nan)
    valid = np.isfinite(values)
    if valid.sum() == 0:
        return qvals
    p = values[valid]
    order = np.argsort(p)
    ranked = p[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    restored = np.empty_like(adjusted)
    restored[order] = adjusted
    qvals[valid] = restored
    return qvals

def star_from_q(q):
    if not np.isfinite(q):
        return ''
    if q < 0.001:
        return '***'
    if q < 0.01:
        return '**'
    if q < 0.05:
        return '*'
    return ''

def normalize_distance(distance_matrix):
    D = np.asarray(distance_matrix, dtype=float)
    D = np.nan_to_num(D, nan=0.0, posinf=0.0, neginf=0.0)
    upper = D[np.triu_indices_from(D, k=1)]
    positive = upper[upper > 0]
    scale = np.median(positive) if positive.size else 1.0
    if not np.isfinite(scale) or scale <= 0:
        scale = 1.0
    return D / scale

def deterministic_axis_orientation(coords):
    oriented = np.asarray(coords, dtype=float).copy()
    for column in range(oriented.shape[1]):
        values = oriented[:, column]
        if not np.isfinite(values).any():
            continue
        idx = np.nanargmax(np.abs(values))
        if values[idx] < 0:
            oriented[:, column] *= -1
    return oriented

def run_pcoa(distance_matrix, n_components=2):
    D = np.asarray(distance_matrix, dtype=float)
    n = D.shape[0]
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ D ** 2 @ J
    eigvals, eigvecs = np.linalg.eigh(B)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    positive = eigvals > 1e-12
    eigvals_pos = eigvals[positive]
    eigvecs_pos = eigvecs[:, positive]
    if len(eigvals_pos) < n_components:
        raise ValueError('Too few positive PCoA eigenvalues.')
    coords = eigvecs_pos[:, :n_components] * np.sqrt(eigvals_pos[:n_components])
    coords = deterministic_axis_orientation(coords)
    variance = eigvals_pos[:n_components] / eigvals_pos.sum() * 100
    return (coords, variance)

def pca_distance(X, already_scaled, variance_target=PCA_VARIANCE_TARGET, max_components=PCA_MAX_COMPONENTS):
    X = X.dropna(axis=1, how='all')
    X = X.loc[:, X.var(axis=0, skipna=True, ddof=0) > 0]
    X = X.fillna(X.median())
    if already_scaled:
        X_processed = X
    else:
        X_processed = pd.DataFrame(StandardScaler().fit_transform(X), index=X.index, columns=X.columns)
    max_allowed = min(X_processed.shape[0] - 1, X_processed.shape[1])
    if max_components is not None:
        max_allowed = min(max_allowed, int(max_components))
    if max_allowed < 1:
        raise ValueError('Too few dimensions for PCA.')
    pca = PCA(n_components=max_allowed, svd_solver='full')
    scores = pca.fit_transform(X_processed)
    cumulative = np.cumsum(pca.explained_variance_ratio_)
    n_keep = int(np.searchsorted(cumulative, variance_target) + 1)
    n_keep = max(min(PCA_MIN_COMPONENTS, max_allowed), n_keep)
    n_keep = min(n_keep, max_allowed)
    selected_scores = scores[:, :n_keep]
    distance = pairwise_distances(selected_scores, metric='euclidean')
    metadata = {'n_features': int(X_processed.shape[1]), 'n_components': int(n_keep), 'variance_explained': float(cumulative[n_keep - 1])}
    return (normalize_distance(distance), metadata)

def species_braycurtis_distance(X):
    X = X.dropna(axis=1, how='all')
    X = X.fillna(0)
    X = X.loc[:, X.sum(axis=0) > 0]
    X_relative = X.div(X.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    distance = pairwise_distances(X_relative, metric='braycurtis')
    metadata = {'n_features': int(X_relative.shape[1]), 'n_components': np.nan, 'variance_explained': np.nan}
    return (normalize_distance(distance), metadata)

def pseudo_f_from_distance(distance_matrix, groups):
    D = np.asarray(distance_matrix, dtype=float)
    groups = np.asarray(groups)
    n = D.shape[0]
    unique_groups = pd.unique(groups)
    k = len(unique_groups)
    if k < 2 or k >= n:
        return (np.nan, np.nan)
    total_ss = np.sum(D ** 2) / n
    within_ss = 0.0
    for group in unique_groups:
        idx = np.where(groups == group)[0]
        n_group = len(idx)
        if n_group <= 1:
            continue
        sub = D[np.ix_(idx, idx)]
        within_ss += np.sum(sub ** 2) / n_group
    between_ss = total_ss - within_ss
    if total_ss <= 0 or within_ss <= 0:
        return (np.nan, np.nan)
    r2 = between_ss / total_ss
    pseudo_f = between_ss / (k - 1) / (within_ss / (n - k))
    return (float(pseudo_f), float(r2))

def permanova(distance_matrix, groups, n_perm=N_PERM, random_state=RANDOM_STATE):
    groups = np.asarray(groups)
    observed_f, observed_r2 = pseudo_f_from_distance(distance_matrix, groups)
    if not np.isfinite(observed_f):
        return (observed_f, observed_r2, np.nan)
    rng = np.random.default_rng(random_state)
    exceed = 0
    for _ in range(n_perm):
        permuted = rng.permutation(groups)
        perm_f, _ = pseudo_f_from_distance(distance_matrix, permuted)
        if np.isfinite(perm_f) and perm_f >= observed_f:
            exceed += 1
    p_value = (exceed + 1) / (n_perm + 1)
    return (observed_f, observed_r2, p_value)

def dispersion_f_stat(coords, groups):
    groups = np.asarray(groups)
    unique_groups = pd.unique(groups)
    distances_by_group = []
    for group in unique_groups:
        idx = np.where(groups == group)[0]
        if len(idx) < 2:
            return np.nan
        centroid = coords[idx].mean(axis=0)
        distances = np.sqrt(np.sum((coords[idx] - centroid) ** 2, axis=1))
        distances_by_group.append(distances)
    return float(f_oneway(*distances_by_group).statistic)

def permdisp(distance_matrix, groups, n_perm=N_PERM, random_state=RANDOM_STATE):
    D = np.asarray(distance_matrix, dtype=float)
    n = D.shape[0]
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ D ** 2 @ J
    eigvals, eigvecs = np.linalg.eigh(B)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    positive = eigvals > 1e-12
    coords = eigvecs[:, positive] * np.sqrt(eigvals[positive])
    observed_f = dispersion_f_stat(coords, groups)
    if not np.isfinite(observed_f):
        return (observed_f, np.nan)
    rng = np.random.default_rng(random_state)
    exceed = 0
    groups = np.asarray(groups)
    for _ in range(n_perm):
        permuted = rng.permutation(groups)
        perm_f = dispersion_f_stat(coords, permuted)
        if np.isfinite(perm_f) and perm_f >= observed_f:
            exceed += 1
    p_value = (exceed + 1) / (n_perm + 1)
    return (observed_f, p_value)

def make_landmark_group(time, status, landmark):
    time = pd.to_numeric(time, errors='coerce')
    status = pd.to_numeric(status, errors='coerce')
    output = pd.Series(index=time.index, dtype=object)
    output[(status == 1) & (time <= landmark)] = 'Early event'
    output[time > landmark] = 'No early event'
    output[(status == 0) & (time <= landmark)] = 'Censored before landmark'
    return output

def add_cov_ellipse(ax, x, y, edgecolor, facecolor, alpha=0.14, n_std=1.5):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]
    if len(x) < 3:
        return
    covariance = np.cov(x, y)
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
    width, height = 2 * n_std * np.sqrt(np.maximum(eigenvalues, 0))
    ellipse = Ellipse(xy=(np.mean(x), np.mean(y)), width=width, height=height, angle=angle, facecolor=facecolor, edgecolor=edgecolor, linewidth=1.6, linestyle='--', alpha=alpha, zorder=1)
    ax.add_patch(ellipse)

def normalize_colname(x):
    value = str(x).strip().lower()
    for token in [' ', '_', '-', get_value('science.translation.open_parenthesis', '('), get_value('science.translation.close_parenthesis', ')'), '(', ')']:
        value = value.replace(token, '')
    return value

def find_column(df, candidates, required=False):
    mapping = {normalize_colname(column): column for column in df.columns}
    for candidate in candidates:
        key = normalize_colname(candidate)
        if key in mapping:
            return mapping[key]
    if required:
        raise ValueError(f'Cannot find any of columns: {candidates}')
    return None

def binary_value(x):
    if pd.isna(x):
        return np.nan
    value = str(x).strip().lower()
    yes_values = {'1', 'yes', 'y', 'true', 'present', 'positive', get_value('science.translation.analysis_001', 'present'), get_value('science.translation.analysis_002', 'yes'), 'a', 'b', 'pvtt'}
    no_values = {'0', 'no', 'n', 'false', 'absent', 'negative', get_value('science.translation.analysis_005', 'absent'), get_value('science.translation.analysis_006', 'no'), 'none'}
    if value in yes_values:
        return 1.0
    if value in no_values:
        return 0.0
    try:
        return float(value)
    except Exception:
        return np.nan

def numeric_column(df, candidates, required=False):
    column = find_column(df, candidates, required=required)
    if column is None:
        return pd.Series(np.nan, index=df.index, dtype=float)
    return pd.to_numeric(df[column], errors='coerce')

def zscore_series(series):
    values = pd.to_numeric(series, errors='coerce').astype(float)
    standard_deviation = values.std(ddof=0)
    if not np.isfinite(standard_deviation) or standard_deviation <= 0:
        return pd.Series(0.0, index=values.index)
    return (values - values.mean()) / standard_deviation

def read_adjustment_covariates(path):
    raw = pd.read_excel(path, engine='openpyxl')
    raw.columns = [str(column).strip() for column in raw.columns]
    sample_column = find_column(raw, ['sample_id', 'Sample ID', 'sample', 'ID', 'PatientID'], required=True)
    raw[sample_column] = raw[sample_column].map(normalize_sample_id)
    raw = raw.dropna(subset=[sample_column]).drop_duplicates(subset=sample_column, keep='first').set_index(sample_column)
    cohort_column = find_column(raw, ['Cohort'], required=False)
    if cohort_column is not None:
        raw = raw[raw[cohort_column].astype(str).str.strip().str.lower() == 'internal'].copy()
    tbs = numeric_column(raw, ['TBS'], required=False)
    if tbs.isna().all():
        size = numeric_column(raw, ['Size', 'Tumor size'], required=False)
        number = numeric_column(raw, ['Number', 'Tumor number'], required=False)
        tbs = np.sqrt(size ** 2 + number ** 2)
    pvtt_column = find_column(raw, ['PVTT'], required=True)
    pvtt = raw[pvtt_column].apply(binary_value)
    albi_grade = numeric_column(raw, ['ALBI_grade', 'ALBI grade'], required=True)
    invalid_albi_levels = sorted(set(albi_grade.dropna().unique()) - {1, 2})
    if invalid_albi_levels:
        raise ValueError(f'ALBI_grade contains values other than 1 and 2: {invalid_albi_levels}')
    if albi_grade.dropna().empty:
        raise ValueError('ALBI_grade contains no valid values.')
    albi_grade = albi_grade.fillna(albi_grade.mode(dropna=True).iloc[0])
    albi_grade2 = (albi_grade.astype(float) == 2).astype(float)
    nlr = numeric_column(raw, ['NLR'], required=False)
    if nlr.isna().all():
        neutrophil = numeric_column(raw, ['NEU', 'Neutrophil'], required=False)
        lymphocyte = numeric_column(raw, ['LYM', 'Lymphocyte'], required=False)
        nlr = neutrophil / lymphocyte.replace(0, np.nan)
    adjustment = pd.DataFrame({'TBS': tbs, 'PVTT': pvtt, 'ALBI_grade2': albi_grade2, 'NLR': nlr}, index=raw.index)
    adjustment['PVTT'] = adjustment['PVTT'].fillna(adjustment['PVTT'].mode(dropna=True).iloc[0] if len(adjustment['PVTT'].mode(dropna=True)) else 0)
    for column in ['TBS', 'NLR']:
        adjustment[column] = adjustment[column].fillna(adjustment[column].median())
    adjustment['TBS_z'] = zscore_series(np.log1p(adjustment['TBS'].clip(lower=0)))
    adjustment['NLR_z'] = zscore_series(np.log1p(adjustment['NLR'].clip(lower=0)))
    adjustment['PVTT'] = adjustment['PVTT'].astype(float)
    return adjustment[ADJUSTMENT_COVARIATES]

def positive_pcoa_coordinates(distance_matrix):
    distance = np.asarray(distance_matrix, dtype=float)
    sample_count = distance.shape[0]
    centering = np.eye(sample_count) - np.ones((sample_count, sample_count)) / sample_count
    gower = -0.5 * centering @ distance ** 2 @ centering
    eigenvalues, eigenvectors = np.linalg.eigh(gower)
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    positive = eigenvalues > 1e-12
    if positive.sum() == 0:
        raise ValueError('No positive PCoA coordinates are available.')
    return eigenvectors[:, positive] * np.sqrt(eigenvalues[positive])

def projection_matrix(design):
    X = np.asarray(design, dtype=float)
    return X @ np.linalg.pinv(X)

def group_design_matrix(groups):
    group_series = pd.Series(groups, dtype='category')
    dummies = pd.get_dummies(group_series, drop_first=True, dtype=float)
    if dummies.shape[1] == 0:
        raise ValueError('The endpoint contains fewer than two groups.')
    return dummies.to_numpy(dtype=float)

def partial_permanova_freedman_lane(distance_matrix, groups, covariates, n_perm=N_PERM, random_state=RANDOM_STATE):
    distance = np.asarray(distance_matrix, dtype=float)
    groups = np.asarray(groups)
    covariates = np.asarray(covariates, dtype=float)
    coordinates = positive_pcoa_coordinates(distance)
    sample_count = coordinates.shape[0]
    reduced_design = np.column_stack([np.ones(sample_count), covariates])
    outcome_design = group_design_matrix(groups)
    full_design = np.column_stack([reduced_design, outcome_design])
    reduced_rank = np.linalg.matrix_rank(reduced_design)
    full_rank = np.linalg.matrix_rank(full_design)
    outcome_df = full_rank - reduced_rank
    residual_df = sample_count - full_rank
    if outcome_df <= 0 or residual_df <= 0:
        return (np.nan, np.nan, np.nan, np.nan)
    H_reduced = projection_matrix(reduced_design)
    H_full = projection_matrix(full_design)
    identity = np.eye(sample_count)

    def calculate_statistics(response_coordinates):
        outcome_ss = float(np.sum(((H_full - H_reduced) @ response_coordinates) ** 2))
        residual_ss = float(np.sum(((identity - H_full) @ response_coordinates) ** 2))
        total_ss = float(np.sum(response_coordinates ** 2))
        if residual_ss <= 0 or total_ss <= 0:
            return (np.nan, np.nan, np.nan)
        pseudo_f = outcome_ss / outcome_df / (residual_ss / residual_df)
        incremental_r2 = outcome_ss / total_ss
        partial_r2 = outcome_ss / (outcome_ss + residual_ss)
        return (pseudo_f, incremental_r2, partial_r2)
    observed_f, incremental_r2, partial_r2 = calculate_statistics(coordinates)
    if not np.isfinite(observed_f):
        return (observed_f, incremental_r2, partial_r2, np.nan)
    fitted_reduced = H_reduced @ coordinates
    residual_reduced = (identity - H_reduced) @ coordinates
    rng = np.random.default_rng(random_state)
    exceed = 0
    for _ in range(n_perm):
        permutation = rng.permutation(sample_count)
        permuted_coordinates = fitted_reduced + residual_reduced[permutation, :]
        permuted_f, _, _ = calculate_statistics(permuted_coordinates)
        if np.isfinite(permuted_f) and permuted_f >= observed_f:
            exceed += 1
    p_value = (exceed + 1) / (n_perm + 1)
    return (observed_f, incremental_r2, partial_r2, p_value)
labels = read_labels(label_path)
adjustment_covariates = read_adjustment_covariates(clinical_path)
missing_adjustment = labels.index.difference(adjustment_covariates.index)
if len(missing_adjustment) > 0:
    raise ValueError('Clinical adjustment data are missing for samples: ' + str(missing_adjustment[:10].tolist()))
adjustment_covariates = adjustment_covariates.loc[labels.index].copy()
matrices = {}
for modality, info in files.items():
    matrix = read_matrix(info['path'])
    missing = labels.index.difference(matrix.index)
    if len(missing) > 0:
        raise ValueError(f'{modality} is missing labeled samples: {missing[:10].tolist()}')
    matrices[modality] = matrix.loc[labels.index].copy()
print('Samples used:', len(labels))
distance_matrices = {}
distance_metadata = []
for modality, info in files.items():
    X = matrices[modality]
    if info['method'] == 'pca_distance':
        distance, metadata = pca_distance(X, already_scaled=info['already_scaled'])
    elif info['method'] == 'braycurtis':
        distance, metadata = species_braycurtis_distance(X)
    else:
        raise ValueError(f'Unknown distance method for {modality}.')
    distance_matrices[modality] = distance
    distance_metadata.append({'Modality': modality, 'Distance_method': info['method'], **metadata})
    print(modality, '| features:', metadata['n_features'], '| retained PCs:', metadata['n_components'], '| retained variance:', metadata['variance_explained'])
total_weight = float(sum(FUSION_WEIGHTS.values()))
fused_distance = np.sum([FUSION_WEIGHTS[modality] * distance_matrices[modality] for modality in FUSION_WEIGHTS], axis=0) / total_weight
fused_distance = normalize_distance(fused_distance)
distance_matrices['Fused'] = fused_distance
distance_metadata.append({'Modality': 'Fused', 'Distance_method': 'weighted normalized distance of ' + '+'.join(FUSION_WEIGHTS.keys()), 'n_features': np.nan, 'n_components': np.nan, 'variance_explained': np.nan})
for _endpoint, _time_col in [('PFS', 'PFS_time'), ('OS', 'OS_time')]:
    _time_values = pd.to_numeric(labels[_time_col], errors='coerce')
    print(f'[Survival time audit] {_endpoint}: unit={SURVIVAL_TIME_UNIT}; range={_time_values.min():.2f}–{_time_values.max():.2f}; median={_time_values.median():.2f}')
labels['PFS_landmark'] = make_landmark_group(labels['PFS_time'], labels['PFS_status'], PFS_LANDMARK_VALUE)
labels['OS_landmark'] = make_landmark_group(labels['OS_time'], labels['OS_status'], OS_LANDMARK_VALUE)
print(f'[Landmark audit] PFS={PFS_LANDMARK_MONTHS:g} months -> {PFS_LANDMARK_VALUE:.2f} {SURVIVAL_TIME_UNIT}; OS={OS_LANDMARK_MONTHS:g} months -> {OS_LANDMARK_VALUE:.2f} {SURVIVAL_TIME_UNIT}')
landmark_audit_records = []
for endpoint_name, group_col, time_col, status_col, landmark_value in [('Early PFS', 'PFS_landmark', 'PFS_time', 'PFS_status', PFS_LANDMARK_VALUE), ('Early OS', 'OS_landmark', 'OS_time', 'OS_status', OS_LANDMARK_VALUE)]:
    counts = labels[group_col].value_counts(dropna=False).to_dict()
    usable = labels[group_col].isin(['Early event', 'No early event'])
    usable_counts = labels.loc[usable, group_col].value_counts().to_dict()
    landmark_audit_records.append({'Endpoint': endpoint_name, 'Landmark_value': float(landmark_value), 'Landmark_unit': SURVIVAL_TIME_UNIT, 'Early_event_n': int(counts.get('Early event', 0)), 'No_early_event_n': int(counts.get('No early event', 0)), 'Censored_before_landmark_n': int(counts.get('Censored before landmark', 0)), 'Usable_n': int(usable.sum()), 'Usable_group_n': int(len(usable_counts)), 'Time_min': float(labels[time_col].min()), 'Time_median': float(labels[time_col].median()), 'Time_max': float(labels[time_col].max()), 'Total_events': int(labels[status_col].sum())})
    print(f"[Landmark groups] {endpoint_name}: Early event={counts.get('Early event', 0)}, No early event={counts.get('No early event', 0)}, censored before landmark={counts.get('Censored before landmark', 0)}, usable={int(usable.sum())}; time range={labels[time_col].min():.2f}–{labels[time_col].max():.2f} {SURVIVAL_TIME_UNIT}")
landmark_audit = pd.DataFrame(landmark_audit_records)
landmark_audit.to_csv(os.path.join(out_dir, get_value('science.tables.result_051', 'result_051.csv')), index=False, encoding='utf-8-sig')
endpoint_groups = {'Response': labels['Response'].map({1: 'Responder', 0: 'Non-Responder'}), 'Early PFS': labels['PFS_landmark'], 'Early OS': labels['OS_landmark']}
records = []
for modality, distance in distance_matrices.items():
    for endpoint_index, (endpoint, groups) in enumerate(endpoint_groups.items()):
        valid = groups.notna().values
        if endpoint in ['Early PFS', 'Early OS']:
            valid = valid & (groups.values != 'Censored before landmark')
        D = distance[np.ix_(valid, valid)]
        g = groups.iloc[np.where(valid)[0]].astype(str).values
        covariates = adjustment_covariates.iloc[np.where(valid)[0]][ADJUSTMENT_COVARIATES].to_numpy(dtype=float)
        group_counts = pd.Series(g).value_counts().to_dict()
        estimable = len(group_counts) >= 2 and valid.sum() >= 3
        if not estimable:
            print(f'[PERMANOVA WARNING] {modality} / {endpoint} is not estimable after landmark censoring: n={int(valid.sum())}, groups={group_counts}. Statistics are recorded as NaN; the prespecified landmark is NOT changed automatically.')
            records.append({'Modality': modality, 'Endpoint': endpoint, 'N': int(valid.sum()), 'Estimable': False, 'Group_counts': str(group_counts), 'Unadjusted_PERMANOVA_F': np.nan, 'Unadjusted_PERMANOVA_R2': np.nan, 'Unadjusted_PERMANOVA_P': np.nan, 'Adjusted_partial_PERMANOVA_F': np.nan, 'Adjusted_incremental_R2': np.nan, 'Adjusted_partial_R2': np.nan, 'Adjusted_partial_PERMANOVA_P': np.nan, 'PERMDISP_F': np.nan, 'PERMDISP_P': np.nan})
            continue
        unadjusted_f, unadjusted_r2, unadjusted_p = permanova(D, g, n_perm=N_PERM, random_state=RANDOM_STATE + endpoint_index)
        adjusted_f, incremental_r2, partial_r2, adjusted_p = partial_permanova_freedman_lane(D, g, covariates, n_perm=N_PERM, random_state=RANDOM_STATE + 1000 + endpoint_index)
        permdisp_f, permdisp_p = permdisp(D, g, n_perm=N_PERM, random_state=RANDOM_STATE + 2000 + endpoint_index)
        records.append({'Modality': modality, 'Endpoint': endpoint, 'N': int(valid.sum()), 'Estimable': True, 'Group_counts': str(group_counts), 'Unadjusted_PERMANOVA_F': unadjusted_f, 'Unadjusted_PERMANOVA_R2': unadjusted_r2, 'Unadjusted_PERMANOVA_P': unadjusted_p, 'Adjusted_partial_PERMANOVA_F': adjusted_f, 'Adjusted_incremental_R2': incremental_r2, 'Adjusted_partial_R2': partial_r2, 'Adjusted_partial_PERMANOVA_P': adjusted_p, 'PERMDISP_F': permdisp_f, 'PERMDISP_P': permdisp_p})
association_results = pd.DataFrame(records)
association_results['Adjusted_partial_PERMANOVA_FDR'] = bh_fdr(association_results['Adjusted_partial_PERMANOVA_P'])
association_results['Unadjusted_PERMANOVA_FDR'] = bh_fdr(association_results['Unadjusted_PERMANOVA_P'])
association_results['PERMDISP_FDR'] = bh_fdr(association_results['PERMDISP_P'])
association_results.to_csv(os.path.join(out_dir, get_value('science.tables.metrics_025', 'metrics_025.csv')), index=False, encoding='utf-8-sig')
pd.DataFrame(distance_metadata).to_csv(os.path.join(out_dir, get_value('science.tables.result_052', 'result_052.csv')), index=False, encoding='utf-8-sig')
fused_coords, fused_variance = run_pcoa(fused_distance, n_components=2)
plot_df = labels.copy()
plot_df['PCoA1'] = fused_coords[:, 0]
plot_df['PCoA2'] = fused_coords[:, 1]
response_colors = {1: '#4C78A8', 0: '#E39C37'}
pfs_markers = {'No early event': 'o', 'Early event': '^', 'Censored before landmark': 'X'}
fig = plt.figure(figsize=(8.4, 4.6))
grid = GridSpec(nrows=1, ncols=2, width_ratios=[4.7, 1.75], wspace=0.025)
ax = fig.add_subplot(grid[0, 0])
legend_ax = fig.add_subplot(grid[0, 1])
legend_ax.axis('off')
for response_value in [1, 0]:
    subset = plot_df[plot_df['Response'] == response_value]
    color = response_colors[response_value]
    add_cov_ellipse(ax, subset['PCoA1'], subset['PCoA2'], edgecolor=color, facecolor=color)
for _, row in plot_df.iterrows():
    color = response_colors[int(row['Response'])]
    marker = pfs_markers.get(row['PFS_landmark'], 'o')
    early_os = row['OS_landmark'] == 'Early event'
    edgecolor = '#D84A3A' if early_os else '#333333'
    linewidth = 2.0 if early_os else 0.8
    ax.scatter(row['PCoA1'], row['PCoA2'], s=88, marker=marker, facecolor=color, edgecolor=edgecolor, linewidth=linewidth, alpha=0.98, zorder=3)
ax.axhline(0, color='#CCCCCC', linestyle='--', linewidth=0.8)
ax.axvline(0, color='#CCCCCC', linestyle='--', linewidth=0.8)
ax.set_xlabel(f'Fused PCoA1 ({fused_variance[0]:.1f}%)', fontsize=13)
ax.set_ylabel(f'Fused PCoA2 ({fused_variance[1]:.1f}%)', fontsize=13)
fused_stats = association_results[association_results['Modality'] == 'Fused'].set_index('Endpoint')
annotation_lines = ['Unsupervised fused distance']
for endpoint in ['Response', 'Early PFS', 'Early OS']:
    row = fused_stats.loc[endpoint]
    annotation_lines.append(f"{endpoint}: adjusted ΔR²={row['Adjusted_incremental_R2']:.3f}, FDR={row['Adjusted_partial_PERMANOVA_FDR']:.3g}; dispersion FDR={row['PERMDISP_FDR']:.3g}")
ax.text(0.02, 0.98, '\n'.join(annotation_lines), transform=ax.transAxes, ha='left', va='top', fontsize=11.0, bbox={'boxstyle': 'round,pad=0.35', 'facecolor': 'white', 'edgecolor': '#CCCCCC', 'alpha': 0.92})
y = 0.92
legend_ax.text(0.03, y, 'Response', fontsize=13.0, fontweight='bold', va='top')
y -= 0.1
for response_value, label in [(1, 'Responder'), (0, 'Non-responder')]:
    legend_ax.scatter(0.06, y, s=72, marker='o', facecolor=response_colors[response_value], edgecolor='#333333', linewidth=0.8)
    legend_ax.text(0.15, y, label, fontsize=11.5, va='center')
    y -= 0.075
y -= 0.055
legend_ax.text(0.03, y, f'PFS · {PFS_LANDMARK_MONTHS:g} mo', fontsize=13.0, fontweight='bold', va='top')
y -= 0.105
for group_name, marker, short_label in [('No early event', 'o', 'No event'), ('Early event', '^', 'Early event'), ('Censored before landmark', 'X', 'Censored')]:
    legend_ax.scatter(0.06, y, s=68, marker=marker, facecolor='white', edgecolor='#333333', linewidth=1.1)
    legend_ax.text(0.15, y, short_label, fontsize=11.5, va='center')
    y -= 0.075
y -= 0.055
legend_ax.text(0.03, y, f'OS · {OS_LANDMARK_MONTHS:g} mo', fontsize=13.0, fontweight='bold', va='top')
y -= 0.11
for edgecolor, linewidth, label in [('#333333', 0.8, 'No event / censored'), ('#D84A3A', 2.0, 'Early event')]:
    legend_ax.scatter(0.06, y, s=68, marker='o', facecolor='white', edgecolor=edgecolor, linewidth=linewidth)
    legend_ax.text(0.15, y, label, fontsize=11.5, va='center')
    y -= 0.075
legend_ax.set_xlim(0, 1)
legend_ax.set_ylim(0, 1)
plt.tight_layout()
fig3a_png = os.path.join(out_dir, get_value('science.figures.plot_072', 'plot_072.png'))
fig3a_pdf = os.path.join(out_dir, get_value('science.figures.plot_073', 'plot_073.pdf'))
plt.savefig(fig3a_png, dpi=600, bbox_inches='tight')
plt.savefig(fig3a_pdf, bbox_inches='tight')
plt.show()



row_order = ['Metagenomics', 'Species', 'Radiomics', 'Blood', 'Urine', 'Fused']
column_order = ['Response', 'Early PFS', 'Early OS']
r2_matrix = association_results.pivot(index='Modality', columns='Endpoint', values='Adjusted_incremental_R2').loc[row_order, column_order]
q_matrix = association_results.pivot(index='Modality', columns='Endpoint', values='Adjusted_partial_PERMANOVA_FDR').loc[row_order, column_order]
dispersion_q_matrix = association_results.pivot(index='Modality', columns='Endpoint', values='PERMDISP_FDR').loc[row_order, column_order]
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
DISPLAY_MAX_PERCENT = 6.0
PRIMARY_FDR_THRESHOLD = 0.05
PRIMARY_R2_THRESHOLD = 0.03
SHOW_EVIDENCE_OUTLINE = True
SHOW_DISPERSION_DAGGER = True
SHOW_ANY_SIGNIFICANCE_MARK = True
plot_matrix = r2_matrix.loc[row_order, column_order].copy() * 100
plot_q_matrix = q_matrix.loc[row_order, column_order].copy()
plot_dispersion_q = dispersion_q_matrix.loc[row_order, column_order].copy()
display_row_labels = {'Metagenomics': 'Metagenomics (KO)', 'Species': 'Species', 'Radiomics': 'Radiomics', 'Blood': 'Blood', 'Urine': 'Urine', 'Fused': 'Fused'}
display_column_labels = {'Response': 'Response', 'Early PFS': 'Early PFS', 'Early OS': 'Early OS'}
soft_cmap = LinearSegmentedColormap.from_list('deep_effect_size', ['#F6F8F7', '#DFECE8', '#BDD9D3', '#86BDB6', '#4A9898', '#1F6F78'])
fig, ax = plt.subplots(figsize=(7.0, 4.8))
im = ax.imshow(plot_matrix.values, cmap=soft_cmap, vmin=0, vmax=DISPLAY_MAX_PERCENT, aspect='auto', interpolation='nearest')
ax.set_xticks(np.arange(len(column_order)))
ax.set_xticklabels([display_column_labels[column] for column in column_order], fontsize=12)
ax.set_yticks(np.arange(len(row_order)))
ax.set_yticklabels([display_row_labels[row] for row in row_order], fontsize=12)
for row_idx, modality in enumerate(row_order):
    for col_idx, endpoint in enumerate(column_order):
        value_percent = plot_matrix.loc[modality, endpoint]
        r2_value = r2_matrix.loc[modality, endpoint]
        q_value = plot_q_matrix.loc[modality, endpoint]
        dispersion_q = plot_dispersion_q.loc[modality, endpoint]
        if not np.isfinite(value_percent):
            cell_text = 'NA'
        else:
            cell_text = f'{value_percent:.1f}%'
        ax.text(col_idx, row_idx, cell_text, ha='center', va='center', fontsize=11.5, fontweight='normal', color='#202020', zorder=4)
        if not SHOW_ANY_SIGNIFICANCE_MARK:
            continue
        robust_association = np.isfinite(q_value) and q_value < PRIMARY_FDR_THRESHOLD and np.isfinite(r2_value) and (r2_value >= PRIMARY_R2_THRESHOLD) and (not np.isfinite(dispersion_q) or dispersion_q >= 0.05)
        if SHOW_EVIDENCE_OUTLINE and robust_association:
            pad = 0.03
            pad = 0.03
            ax.add_patch(Rectangle((col_idx - 0.5 + pad, row_idx - 0.5 + pad), 1.0 - 2 * pad, 1.0 - 2 * pad, fill=False, edgecolor='#333333', linewidth=0.9, joinstyle='miter', capstyle='butt', antialiased=True, zorder=6))
        if SHOW_DISPERSION_DAGGER and np.isfinite(dispersion_q) and (dispersion_q < 0.05):
            ax.text(col_idx + 0.35, row_idx - 0.31, '†', ha='center', va='center', fontsize=10, color='#333333', zorder=6)
ax.set_xticks(np.arange(-0.5, len(column_order), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(row_order), 1), minor=True)
ax.grid(False)
ax.tick_params(which='minor', bottom=False, left=False)
ax.tick_params(axis='both', which='major', length=0)
fused_row_idx = row_order.index('Fused')
for tick_label in ax.get_yticklabels():
    if tick_label.get_text() == 'Fused':
        tick_label.set_fontweight('semibold')
cbar = fig.colorbar(im, ax=ax, fraction=0.047, pad=0.035)
cbar.set_label('Adjusted incremental variance (%)', fontsize=11.5, labelpad=8)
cbar.set_ticks(np.arange(0, DISPLAY_MAX_PERCENT + 0.1, 1))
cbar.ax.tick_params(labelsize=8.5, length=3)
cbar.outline.set_linewidth(0.6)
for spine in ax.spines.values():
    spine.set_visible(False)
if SHOW_ANY_SIGNIFICANCE_MARK:
    footnote = 'Cells show clinically adjusted incremental R². Thin outlines indicate FDR < 0.05, R² ≥ 3%, and no evidence of unequal dispersion; † indicates PERMDISP FDR < 0.05. Exact FDR values are reported separately.'
else:
    footnote = 'Cells show clinically adjusted incremental R². Exact PERMANOVA and PERMDISP FDR values are reported separately.'
fig.subplots_adjust(left=0.24, right=0.9, top=0.88, bottom=0.11)
fig3b_png = os.path.join(out_dir, get_value('science.figures.plot_074', 'plot_074.png'))
fig3b_pdf = os.path.join(out_dir, get_value('science.figures.plot_075', 'plot_075.pdf'))
plt.savefig(fig3b_png, dpi=600, bbox_inches='tight', facecolor='white')
plt.savefig(fig3b_pdf, bbox_inches='tight', facecolor='white')
plt.show()





In [ ]:
import os
from collections import OrderedDict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'Arial', 'font.size': 11.0, 'axes.titlesize': 14.0, 'axes.labelsize': 11.5, 'xtick.labelsize': 10.5, 'ytick.labelsize': 10.5, 'legend.fontsize': 10.0, 'figure.titlesize': 16.0, 'text.color': '#1F1F1F', 'axes.labelcolor': '#1F1F1F', 'axes.titlecolor': '#1F1F1F', 'xtick.color': '#1F1F1F', 'ytick.color': '#1F1F1F', 'axes.edgecolor': '#2B2B2B', 'axes.grid': False, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none'})
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
from lifelines import CoxPHFitter
import statsmodels.api as sm
from matplotlib.patches import Rectangle
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
project_dir = str(project_path(get_value('science.paths.directory_008', 'directory_008')))
input_dir = str(project_path(get_value('science.paths.directory_001', 'directory_001')))
output_root = str(project_path(get_value('science.paths.directory_055', 'directory_055')))
clinical_path = str(project_path(get_value('science.tables.clinical_006', 'clinical_006.xlsx')))
LABEL_FILENAME = globals().get('LABEL_FILENAME', get_value('science.tables.outcomes_001', 'outcomes_001.csv'))
PREFERRED_RESPONSE_COLUMN = globals().get('PREFERRED_RESPONSE_COLUMN', 'Response')
label_path = os.path.join(input_dir, LABEL_FILENAME)
files = {'Metagenomics': {'path': os.path.join(input_dir, get_value('science.tables.features_001', 'features_001.csv.gz')), 'method': 'pca', 'already_scaled': True}, 'Species': {'path': os.path.join(input_dir, get_value('science.tables.result_050', 'result_050.csv.gz')), 'method': 'braycurtis', 'already_scaled': False}, 'Blood': {'path': os.path.join(input_dir, get_value('science.tables.features_003', 'features_003.csv.gz')), 'method': 'pca', 'already_scaled': True}, 'Urine': {'path': os.path.join(input_dir, get_value('science.tables.features_004', 'features_004.csv.gz')), 'method': 'pca', 'already_scaled': True}, 'Radiomics': {'path': os.path.join(input_dir, get_value('science.tables.features_002', 'features_002.csv.gz')), 'method': 'pca', 'already_scaled': True}}
out_dir = os.path.join(output_root, get_value('science.paths.directory_056', 'directory_056'), get_value('science.paths.directory_057', 'directory_057'))
os.makedirs(out_dir, exist_ok=True)
required_paths = [clinical_path, label_path] + [info['path'] for info in files.values()]
missing_paths = [p for p in required_paths if not os.path.exists(p)]
if missing_paths:
    print('\nMissing files for the Figure 3C cell:')
    for p in missing_paths:
        print(' -', p)
    raise FileNotFoundError('Please correct the paths shown above.')
N_COMPONENTS = 3
SHOW_CELL_VALUES = True
SPEARMAN_VMIN = -0.6
SPEARMAN_VMAX = 0.6
COX_MIN_ABS_LIMIT = 0.8
COX_MAX_ABS_LIMIT = 1.5
ADJUSTMENT_COVARIATES = ['TBS_z', 'PVTT', 'ALBI_grade2', 'NLR_z']
FDR_PRIMARY = 0.05
FDR_SUGGESTIVE = 0.1
CLINICAL_EFFECT_THRESHOLD = 0.2
OUTCOME_EFFECT_THRESHOLD = 0.25
PLOT_SURVIVAL_MODEL = 'baseline'
FIG3C_STYLE = {'figsize': (18.0, 8.35), 'font_family': 'Arial', 'panel_title': 14.0, 'domain_header': 12.5, 'outcome_header': 12.5, 'modality_group': 12.0, 'row_label': 11.2, 'column_label': 11.4, 'clinical_value': 11.0, 'outcome_value': 11.2, 'clinical_symbol': 14.0, 'clinical_symbol_x_offset': 0.22, 'clinical_symbol_y_offset': -0.31, 'outcome_dot_filled': 10.0, 'outcome_dot_open': 10.0, 'outcome_dot_linewidth': 0.65, 'outcome_dot_x_offset': 0.24, 'outcome_dot_y_offset': -0.32, 'colorbar_label': 11.6, 'colorbar_tick': 10.8, 'footnote': 9.0, 'cell_grid_linewidth': 0.7, 'group_separator_linewidth': 1.15}
plt.rcParams.update({'font.family': FIG3C_STYLE['font_family'], 'pdf.fonttype': 42, 'ps.fonttype': 42})
clinical_groups = OrderedDict({'Host background': ['Age', 'Hepatitis B', 'ECOG'], 'Tumor burden': ['AFP', 'PVIKA', 'TBS', 'PVTT', 'Metastasis'], 'Liver reserve / injury': ['Child-Pugh', 'ALBI_grade', 'ALT', 'PT', 'LDH'], 'Systemic inflammation / nutrition': ['PNI', 'NLR', 'SII', 'FIB-4']})
clinical_order = []
for _, columns in clinical_groups.items():
    clinical_order.extend(columns)
non_survival_vars = clinical_order.copy()
display_name = {'Age': 'Age', 'BMI': 'BMI', 'Hepatitis B': 'HBV', 'ECOG': 'ECOG', 'Cirrhosis': 'Cirrhosis', 'AFP': 'AFP', 'PVIKA': 'PIVKA-II', 'TBS': 'TBS', 'PVTT': 'PVTT', 'Metastasis': 'Metastasis', 'Child-Pugh': 'Child-Pugh', 'ALBI_grade': 'ALBI grade', 'ALT': 'ALT', 'AST': 'AST', 'PT': 'PT', 'LDH': 'LDH', 'PNI': 'PNI', 'HGB': 'HGB', 'NLR': 'NLR', 'SII': 'SII', 'LMR': 'LMR', 'FIB-4': 'FIB-4', 'Response': 'Response', 'PFS_Cox': 'PFS', 'OS_Cox': 'OS'}

def normalize_colname(x):
    x = str(x).strip().lower()
    x = x.replace(' ', '')
    x = x.replace('_', '')
    x = x.replace('-', '')
    x = x.replace(get_value('science.translation.open_parenthesis', '('), '(').replace(get_value('science.translation.close_parenthesis', ')'), ')')
    return x

def find_col(df, candidates, required=False):
    norm_map = {normalize_colname(c): c for c in df.columns}
    for cand in candidates:
        key = normalize_colname(cand)
        if key in norm_map:
            return norm_map[key]
    if required:
        raise ValueError(f'Cannot find any of columns: {candidates}')
    return None

def to_binary(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    yes_set = ['1', 'yes', 'y', 'true', 'positive', 'pos', 'present', 'with', get_value('science.translation.analysis_001', 'present'), get_value('science.translation.analysis_002', 'yes'), 'responder', 'response', 'r', 'pr', 'cr']
    no_set = ['0', 'no', 'n', 'false', 'negative', 'neg', 'absent', 'without', get_value('science.translation.analysis_005', 'absent'), get_value('science.translation.analysis_006', 'no'), 'non-responder', 'nonresponder', 'non_responder', 'nr', 'sd', 'pd']
    if s in yes_set:
        return 1.0
    if s in no_set:
        return 0.0
    try:
        return float(s)
    except Exception:
        return np.nan

def child_pugh_to_numeric(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().upper()
    mapping = {'A': 1, 'B': 2, 'C': 3, '1': 1, '2': 2, '3': 3}
    if s in mapping:
        return float(mapping[s])
    try:
        return float(s)
    except Exception:
        return np.nan

def to_numeric_series(df, candidates, required=False):
    col = find_col(df, candidates, required=required)
    if col is None:
        return None
    return pd.to_numeric(df[col], errors='coerce')

def safe_div(a, b):
    out = a / b
    out = out.replace([np.inf, -np.inf], np.nan)
    return out

def bh_fdr_vector(pvals):
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full(len(pvals), np.nan)
    valid = ~np.isnan(pvals)
    if valid.sum() == 0:
        return qvals
    pv = pvals[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    adj = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    qv = np.empty(len(ranked))
    qv[order] = adj
    qvals[valid] = qv
    return qvals

def fdr_matrix_from_p_matrix(p_mat):
    flat = p_mat.values.flatten().astype(float)
    qflat = bh_fdr_vector(flat)
    qmat = pd.DataFrame(qflat.reshape(p_mat.shape), index=p_mat.index, columns=p_mat.columns)
    return qmat

def star_from_p(p):
    if pd.isna(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return ''

def format_value_with_star(v, p):
    if pd.isna(v):
        return ''
    return f'{v:.2f}{star_from_p(p)}'

def read_matrix(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    df.columns = [str(c).strip() for c in df.columns]
    sample_candidates = ['sample', 'sample_id', 'Sample ID', 'Sample', 'ID']
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    sample_col = None
    for candidate in sample_candidates:
        key = str(candidate).strip().lower()
        if key in lower_map:
            sample_col = lower_map[key]
            break
    if sample_col is None:
        sample_col = df.columns[0]
    df = df.rename(columns={sample_col: 'sample'})
    df['sample'] = df['sample'].astype(str).str.lower().str.strip()
    df = df.drop_duplicates(subset='sample', keep='first').set_index('sample')
    df = df.apply(pd.to_numeric, errors='coerce')
    return df

def run_pcoa(distance_matrix, n_components=3):
    D = np.asarray(distance_matrix, dtype=float)
    D = np.nan_to_num(D, nan=0.0, posinf=0.0, neginf=0.0)
    n = D.shape[0]
    D2 = D ** 2
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ D2 @ J
    eigvals, eigvecs = np.linalg.eigh(B)
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]
    pos = eigvals > 0
    eigvals_pos = eigvals[pos]
    eigvecs_pos = eigvecs[:, pos]
    if len(eigvals_pos) < n_components:
        raise ValueError(f'Only {len(eigvals_pos)} positive eigenvalues, cannot extract {n_components} PCoA components.')
    coords = eigvecs_pos[:, :n_components] * np.sqrt(eigvals_pos[:n_components])
    var_exp = eigvals_pos[:n_components] / eigvals_pos.sum() * 100
    return (coords, var_exp)

def orient_scores_deterministically(scores, loadings=None):
    pass
    scores = np.asarray(scores, dtype=float).copy()
    for component in range(scores.shape[1]):
        flip = False
        if loadings is not None:
            component_loadings = loadings[component]
            feature_idx = np.argmax(np.abs(component_loadings))
            flip = component_loadings[feature_idx] < 0
        else:
            sample_idx = np.argmax(np.abs(scores[:, component]))
            flip = scores[sample_idx, component] < 0
        if flip:
            scores[:, component] *= -1
    return scores

def get_modal_axes(modal_name, info):
    X = read_matrix(info['path'])
    X = X.dropna(axis=1, how='all')
    if info['method'] in {'pcoa', 'braycurtis'}:
        X = X.fillna(0)
        X = X.loc[:, X.sum(axis=0) > 0]
        row_sums = X.sum(axis=1)
        if (row_sums <= 0).any():
            bad_samples = X.index[row_sums <= 0].tolist()
            raise ValueError(f'{modal_name} contains samples with zero retained reads: {bad_samples[:10]}')
        X_rel = X.div(row_sums, axis=0)
        dist = pairwise_distances(X_rel, metric='braycurtis')
        dist = np.nan_to_num(dist, nan=0.0, posinf=0.0, neginf=0.0)
        coords, var_exp = run_pcoa(dist, n_components=N_COMPONENTS)
        coords = orient_scores_deterministically(coords, loadings=None)
        cols = [f'{modal_name}_PCoA{i}' for i in range(1, N_COMPONENTS + 1)]
    elif info['method'] == 'pca':
        X = X.fillna(X.median())
        X = X.loc[:, X.var(axis=0) > 0]
        if info['already_scaled']:
            X_proc = X.copy()
        else:
            X_proc = pd.DataFrame(StandardScaler().fit_transform(X), index=X.index, columns=X.columns)
        pca = PCA(n_components=N_COMPONENTS, random_state=42)
        coords = pca.fit_transform(X_proc)
        coords = orient_scores_deterministically(coords, loadings=pca.components_)
        var_exp = pca.explained_variance_ratio_[:N_COMPONENTS] * 100
        cols = [f'{modal_name}_PC{i}' for i in range(1, N_COMPONENTS + 1)]
    else:
        raise ValueError(f"Unknown method for {modal_name}: {info['method']}")
    axes_df = pd.DataFrame(coords, index=X.index, columns=cols)
    print(f'{modal_name}: n={axes_df.shape[0]}, features={X.shape[1]}')
    print('Explained variance:', [f'{v:.1f}%' for v in var_exp])
    return axes_df
axes_list = []
for modal_name, info in files.items():
    axes_list.append(get_modal_axes(modal_name, info))
modal_axes = pd.concat(axes_list, axis=1, join='inner')
print(f'\nCommon samples across all modality matrices: n = {modal_axes.shape[0]}')
clin_raw = pd.read_excel(clinical_path, engine='openpyxl')
clin_raw.columns = [str(c).strip() for c in clin_raw.columns]
sample_col = find_col(clin_raw, ['sample_id', 'sample', 'ID', 'PatientID'], required=True)
clin_raw[sample_col] = clin_raw[sample_col].astype(str).str.lower().str.strip()
clin_raw = clin_raw.drop_duplicates(subset=sample_col, keep='first').set_index(sample_col)
cohort_col = find_col(clin_raw, ['Cohort'], required=False)
if cohort_col is not None:
    clin_raw = clin_raw[clin_raw[cohort_col].astype(str).str.lower().str.strip() == 'internal'].copy()
outcome_raw = pd.read_csv(label_path, encoding='utf-8-sig')
outcome_raw.columns = [str(c).strip() for c in outcome_raw.columns]
outcome_sample_col = find_col(outcome_raw, ['Sample ID', 'sample_id', 'sample', 'Sample', 'ID'], required=True)
response_candidates = list(dict.fromkeys([PREFERRED_RESPONSE_COLUMN, 'Response', 'Response', 'Group']))
outcome_response_col = find_col(outcome_raw, response_candidates, required=True)

print(f'[Outcome audit] Response column used: {outcome_response_col}')
pass
outcome_raw[outcome_sample_col] = outcome_raw[outcome_sample_col].astype(str).str.lower().str.strip()
outcome_raw = outcome_raw.drop_duplicates(subset=outcome_sample_col, keep='first').set_index(outcome_sample_col)
required_outcome_cols = ['PFS_time', 'PFS_status', 'OS_time', 'OS_status']
for col in required_outcome_cols:
    if col not in outcome_raw.columns:
        raise ValueError(f'Missing outcome column in new label file: {col}')
outcome_labels = pd.DataFrame({'Response': outcome_raw[outcome_response_col].apply(to_binary), 'PFS_time': pd.to_numeric(outcome_raw['PFS_time'], errors='coerce'), 'PFS_status': pd.to_numeric(outcome_raw['PFS_status'], errors='coerce'), 'OS_time': pd.to_numeric(outcome_raw['OS_time'], errors='coerce'), 'OS_status': pd.to_numeric(outcome_raw['OS_status'], errors='coerce')}, index=outcome_raw.index)
print(f"[Outcome audit] n={len(outcome_labels)}, Response={outcome_labels['Response'].value_counts(dropna=False).sort_index().to_dict()}, PFS events={int(outcome_labels['PFS_status'].fillna(0).sum())}, OS events={int(outcome_labels['OS_status'].fillna(0).sum())}")
common_clinical_samples = clin_raw.index.intersection(outcome_labels.index)
clin_raw = clin_raw.loc[common_clinical_samples].copy()
outcome_labels = outcome_labels.loc[common_clinical_samples].copy()
print('\nClinical baseline and new outcome labels matched:', len(common_clinical_samples))
clin = pd.DataFrame(index=clin_raw.index)
clin['Age'] = to_numeric_series(clin_raw, ['Age'], required=True)
clin['BMI'] = to_numeric_series(clin_raw, ['BMI'], required=False)
clin['Hepatitis B'] = clin_raw[find_col(clin_raw, ['Hepatitis B', 'HBV'], required=True)].apply(to_binary)
clin['ECOG'] = to_numeric_series(clin_raw, ['ECOG'], required=True)
clin['Cirrhosis'] = clin_raw[find_col(clin_raw, ['Cirrhosis'], required=True)].apply(to_binary)
clin['AFP'] = to_numeric_series(clin_raw, ['AFP'], required=True)
clin['PVIKA'] = to_numeric_series(clin_raw, ['PVIKA', 'PIVKA', 'PIVKA-II', 'PIVKA2', 'PIVKA II'], required=True)
clin['TBS'] = to_numeric_series(clin_raw, ['TBS'], required=False)
clin['PVTT'] = clin_raw[find_col(clin_raw, ['PVTT'], required=True)].apply(to_binary)
clin['Metastasis'] = clin_raw[find_col(clin_raw, ['Metastasis', 'Extrahepatic metastasis'], required=True)].apply(to_binary)
clin['Child-Pugh'] = clin_raw[find_col(clin_raw, ['Child-Pugh', 'ChildPugh'], required=True)].apply(child_pugh_to_numeric)
clin['ALBI_grade'] = to_numeric_series(clin_raw, ['ALBI_grade', 'ALBI grade', 'ALBI Grade'], required=True)
invalid_albi_levels = sorted(set(clin['ALBI_grade'].dropna().unique()) - {1, 2})
if invalid_albi_levels:
    raise ValueError(f'ALBI_grade contains values other than 1 or 2: {invalid_albi_levels}')
clin['ALT'] = to_numeric_series(clin_raw, ['ALT'], required=True)
clin['AST'] = to_numeric_series(clin_raw, ['AST'], required=True)
clin['PT'] = to_numeric_series(clin_raw, ['PT'], required=True)
clin['LDH'] = to_numeric_series(clin_raw, ['LDH'], required=True)
clin['PNI'] = to_numeric_series(clin_raw, ['PNI'], required=False)
clin['HGB'] = to_numeric_series(clin_raw, ['HGB', 'Hemoglobin'], required=True)
clin['NLR'] = to_numeric_series(clin_raw, ['NLR'], required=False)
clin['SII'] = to_numeric_series(clin_raw, ['SII'], required=False)
clin['LMR'] = to_numeric_series(clin_raw, ['LMR'], required=False)
clin['FIB-4'] = to_numeric_series(clin_raw, ['FIB-4', 'FIB4'], required=False)
clin['Response'] = outcome_labels['Response'].reindex(clin_raw.index)
if 'BMI' in clin.columns and clin['BMI'].isna().all():
    ht = to_numeric_series(clin_raw, ['Ht', 'Height'], required=False)
    wt = to_numeric_series(clin_raw, ['Wt', 'Weight'], required=False)
    if ht is not None and wt is not None:
        ht_m = ht.copy()
        if ht_m.dropna().median() > 3:
            ht_m = ht_m / 100
        clin['BMI'] = safe_div(wt, ht_m ** 2)
if 'TBS' in clin.columns and clin['TBS'].isna().all():
    size = to_numeric_series(clin_raw, ['Size', 'Tumor size'], required=False)
    number = to_numeric_series(clin_raw, ['Number', 'Tumor number'], required=False)
    if size is not None and number is not None:
        clin['TBS'] = np.sqrt(size ** 2 + number ** 2)
if 'PNI' in clin.columns and clin['PNI'].isna().all():
    alb = to_numeric_series(clin_raw, ['ALB', 'Albumin'], required=False)
    lym = to_numeric_series(clin_raw, ['LYM', 'Lymphocyte'], required=False)
    if alb is not None and lym is not None:
        alb_g_l = alb.copy()
        if alb_g_l.dropna().median() < 10:
            alb_g_l = alb_g_l * 10
        clin['PNI'] = alb_g_l + 5 * lym
if 'NLR' in clin.columns and clin['NLR'].isna().all():
    neu = to_numeric_series(clin_raw, ['NEU', 'Neutrophil'], required=False)
    lym = to_numeric_series(clin_raw, ['LYM', 'Lymphocyte'], required=False)
    if neu is not None and lym is not None:
        clin['NLR'] = safe_div(neu, lym)
if 'SII' in clin.columns and clin['SII'].isna().all():
    plt_col = to_numeric_series(clin_raw, ['PLT', 'Platelet'], required=False)
    neu = to_numeric_series(clin_raw, ['NEU', 'Neutrophil'], required=False)
    lym = to_numeric_series(clin_raw, ['LYM', 'Lymphocyte'], required=False)
    if plt_col is not None and neu is not None and (lym is not None):
        clin['SII'] = safe_div(plt_col * neu, lym)
if 'LMR' in clin.columns and clin['LMR'].isna().all():
    lym = to_numeric_series(clin_raw, ['LYM', 'Lymphocyte'], required=False)
    mono = to_numeric_series(clin_raw, ['MONO', 'Monocyte'], required=False)
    if lym is not None and mono is not None:
        clin['LMR'] = safe_div(lym, mono)
if 'FIB-4' in clin.columns and clin['FIB-4'].isna().all():
    plt_col = to_numeric_series(clin_raw, ['PLT', 'Platelet'], required=False)
    if plt_col is not None:
        denominator = plt_col * np.sqrt(clin['ALT'].where(clin['ALT'] > 0, np.nan))
        clin['FIB-4'] = safe_div(clin['Age'] * clin['AST'], denominator)
surv_df = outcome_labels[['PFS_time', 'PFS_status', 'OS_time', 'OS_status']].reindex(clin_raw.index).copy()
missing_cols = [c for c in non_survival_vars if c not in clin.columns or clin[c].isna().all()]
if len(missing_cols) > 0:
    raise ValueError('The following selected clinical variables are missing or all NA:\n' + '\n'.join(missing_cols))
clin_use = clin[non_survival_vars + ['Response']].copy()
binary_like_cols = ['Hepatitis B', 'Cirrhosis', 'PVTT', 'Metastasis', 'Response']
for col in clin_use.columns:
    if col in binary_like_cols:
        mode_val = clin_use[col].mode(dropna=True)
        fill_val = mode_val.iloc[0] if len(mode_val) else 0
        clin_use[col] = clin_use[col].fillna(fill_val)
    else:
        clin_use[col] = pd.to_numeric(clin_use[col], errors='coerce')
        clin_use[col] = clin_use[col].fillna(clin_use[col].median())
print('\nSelected clinical variables:')
print(clin_use.columns.tolist())
print('\nMissing values after imputation:')
print(clin_use.isna().sum())
common_samples = modal_axes.index.intersection(clin_use.index).intersection(surv_df.index)
modal_axes = modal_axes.loc[common_samples].copy()
clin_use = clin_use.loc[common_samples].copy()
surv_df = surv_df.loc[common_samples].copy()
print(f'\nSamples used for Figure 3C: n = {len(common_samples)}')
if len(common_samples) < 10:
    raise ValueError('Too few common samples. Please check sample IDs.')

def zscore_values(series):
    values = pd.to_numeric(series, errors='coerce').astype(float)
    standard_deviation = values.std(ddof=0)
    if not np.isfinite(standard_deviation) or standard_deviation <= 0:
        return pd.Series(0.0, index=values.index)
    return (values - values.mean()) / standard_deviation

def significance_symbol(q_value, effect_value, effect_threshold):
    pass
    if not np.isfinite(q_value) or not np.isfinite(effect_value):
        return ''
    if abs(effect_value) < effect_threshold:
        return ''
    if q_value < FDR_PRIMARY:
        return '*'
    if q_value < FDR_SUGGESTIVE:
        return '·'
    return ''

def prepare_adjustment_table(clinical_df):
    adjustment = pd.DataFrame(index=clinical_df.index)
    adjustment['TBS'] = pd.to_numeric(clinical_df['TBS'], errors='coerce')
    adjustment['PVTT'] = pd.to_numeric(clinical_df['PVTT'], errors='coerce')
    adjustment['ALBI_grade'] = pd.to_numeric(clinical_df['ALBI_grade'], errors='coerce')
    adjustment['NLR'] = pd.to_numeric(clinical_df['NLR'], errors='coerce')
    adjustment['Response'] = pd.to_numeric(clinical_df['Response'], errors='coerce')
    adjustment['PVTT'] = adjustment['PVTT'].fillna(adjustment['PVTT'].mode(dropna=True).iloc[0] if len(adjustment['PVTT'].mode(dropna=True)) else 0)
    adjustment['Response'] = adjustment['Response'].fillna(adjustment['Response'].mode(dropna=True).iloc[0] if len(adjustment['Response'].mode(dropna=True)) else 0)
    adjustment['ALBI_grade'] = adjustment['ALBI_grade'].fillna(adjustment['ALBI_grade'].mode(dropna=True).iloc[0] if len(adjustment['ALBI_grade'].mode(dropna=True)) else 1)
    invalid_levels = sorted(set(adjustment['ALBI_grade'].dropna().unique()) - {1, 2})
    if invalid_levels:
        raise ValueError(f'ALBI_grade contains values other than 1 or 2: {invalid_levels}')
    for column in ['TBS', 'NLR']:
        adjustment[column] = adjustment[column].fillna(adjustment[column].median())
    adjustment['TBS_z'] = zscore_values(np.log1p(adjustment['TBS'].clip(lower=0)))
    adjustment['NLR_z'] = zscore_values(np.log1p(adjustment['NLR'].clip(lower=0)))
    adjustment['ALBI_grade2'] = (adjustment['ALBI_grade'] == 2).astype(float)
    return adjustment[ADJUSTMENT_COVARIATES + ['Response']].astype(float)
adjustment_df = prepare_adjustment_table(clin.loc[common_samples])
spearman_r = pd.DataFrame(index=modal_axes.columns, columns=non_survival_vars, dtype=float)
spearman_p = pd.DataFrame(index=modal_axes.columns, columns=non_survival_vars, dtype=float)
for axis in modal_axes.columns:
    for variable in non_survival_vars:
        r_value, p_value = spearmanr(modal_axes[axis], clin_use[variable], nan_policy='omit')
        spearman_r.loc[axis, variable] = r_value
        spearman_p.loc[axis, variable] = p_value
spearman_q = pd.DataFrame(np.nan, index=spearman_p.index, columns=spearman_p.columns, dtype=float)
for domain_name, variables in clinical_groups.items():
    domain_p = spearman_p[variables].to_numpy(dtype=float).ravel()
    domain_q = bh_fdr_vector(domain_p).reshape(spearman_p.shape[0], len(variables))
    spearman_q.loc[:, variables] = domain_q
response_beta = pd.Series(np.nan, index=modal_axes.columns, dtype=float)
response_p = pd.Series(np.nan, index=modal_axes.columns, dtype=float)
response_or = pd.Series(np.nan, index=modal_axes.columns, dtype=float)
for axis in modal_axes.columns:
    axis_z = zscore_values(modal_axes[axis])
    model_df = pd.concat([adjustment_df[['Response'] + ADJUSTMENT_COVARIATES], axis_z.rename('axis_z')], axis=1).dropna()
    try:
        design = sm.add_constant(model_df[['axis_z'] + ADJUSTMENT_COVARIATES], has_constant='add')
        model = sm.GLM(model_df['Response'], design, family=sm.families.Binomial()).fit()
        beta = float(model.params['axis_z'])
        p_value = float(model.pvalues['axis_z'])
        response_beta.loc[axis] = beta
        response_or.loc[axis] = np.exp(beta)
        response_p.loc[axis] = p_value
    except Exception as error:
        print(f'[Warning] Adjusted logistic failed for {axis}: {error}')
response_q = pd.Series(bh_fdr_vector(response_p.values), index=response_p.index)
survival_endpoints = OrderedDict({'PFS': ('PFS_time', 'PFS_status'), 'OS': ('OS_time', 'OS_status')})
cox_log_hr = pd.DataFrame(np.nan, index=modal_axes.columns, columns=survival_endpoints.keys(), dtype=float)
cox_p = cox_log_hr.copy()
cox_hr = cox_log_hr.copy()
cox_response_adjusted_log_hr = cox_log_hr.copy()
cox_response_adjusted_p = cox_log_hr.copy()
for axis in modal_axes.columns:
    axis_z = zscore_values(modal_axes[axis])
    for endpoint, (time_col, event_col) in survival_endpoints.items():
        base_df = pd.concat([surv_df[[time_col, event_col]], adjustment_df[ADJUSTMENT_COVARIATES], axis_z.rename('axis_z')], axis=1).dropna()
        base_df = base_df.rename(columns={time_col: 'time', event_col: 'event'})
        try:
            cox_model = CoxPHFitter()
            cox_model.fit(base_df, duration_col='time', event_col='event', robust=True)
            beta = float(cox_model.params_['axis_z'])
            p_value = float(cox_model.summary.loc['axis_z', 'p'])
            cox_log_hr.loc[axis, endpoint] = beta
            cox_hr.loc[axis, endpoint] = np.exp(beta)
            cox_p.loc[axis, endpoint] = p_value
        except Exception as error:
            print(f'[Warning] Baseline-adjusted {endpoint} Cox failed for {axis}: {error}')
        sensitivity_df = pd.concat([surv_df[[time_col, event_col]], adjustment_df[ADJUSTMENT_COVARIATES + ['Response']], axis_z.rename('axis_z')], axis=1).dropna()
        sensitivity_df = sensitivity_df.rename(columns={time_col: 'time', event_col: 'event'})
        try:
            sensitivity_model = CoxPHFitter()
            sensitivity_model.fit(sensitivity_df, duration_col='time', event_col='event', robust=True)
            cox_response_adjusted_log_hr.loc[axis, endpoint] = float(sensitivity_model.params_['axis_z'])
            cox_response_adjusted_p.loc[axis, endpoint] = float(sensitivity_model.summary.loc['axis_z', 'p'])
        except Exception as error:
            print(f'[Warning] Response-adjusted {endpoint} sensitivity Cox failed for {axis}: {error}')
cox_q = pd.DataFrame(np.nan, index=cox_p.index, columns=cox_p.columns, dtype=float)
for endpoint in cox_p.columns:
    cox_q[endpoint] = bh_fdr_vector(cox_p[endpoint].values)
cox_response_adjusted_q = pd.DataFrame(np.nan, index=cox_response_adjusted_p.index, columns=cox_response_adjusted_p.columns, dtype=float)
for endpoint in cox_response_adjusted_p.columns:
    cox_response_adjusted_q[endpoint] = bh_fdr_vector(cox_response_adjusted_p[endpoint].values)
row_order = []
for modality in ['Metagenomics', 'Species', 'Blood', 'Urine', 'Radiomics']:
    row_order.extend([row for row in modal_axes.columns if row.startswith(modality)])
spearman_r = spearman_r.loc[row_order, non_survival_vars]
spearman_p = spearman_p.loc[row_order, non_survival_vars]
spearman_q = spearman_q.loc[row_order, non_survival_vars]
response_beta = response_beta.loc[row_order]
response_or = response_or.loc[row_order]
response_p = response_p.loc[row_order]
response_q = response_q.loc[row_order]
cox_log_hr = cox_log_hr.loc[row_order, list(survival_endpoints.keys())]
cox_hr = cox_hr.loc[row_order, list(survival_endpoints.keys())]
cox_p = cox_p.loc[row_order, list(survival_endpoints.keys())]
cox_q = cox_q.loc[row_order, list(survival_endpoints.keys())]
cox_response_adjusted_log_hr = cox_response_adjusted_log_hr.loc[row_order, list(survival_endpoints.keys())]
cox_response_adjusted_p = cox_response_adjusted_p.loc[row_order, list(survival_endpoints.keys())]
cox_response_adjusted_q = cox_response_adjusted_q.loc[row_order, list(survival_endpoints.keys())]
outcome_effect = pd.DataFrame({'Response': response_beta, 'PFS': cox_log_hr['PFS'], 'OS': cox_log_hr['OS']}, index=row_order)
outcome_p = pd.DataFrame({'Response': response_p, 'PFS': cox_p['PFS'], 'OS': cox_p['OS']}, index=row_order)
outcome_q = pd.DataFrame({'Response': response_q, 'PFS': cox_q['PFS'], 'OS': cox_q['OS']}, index=row_order)
n_rows = len(row_order)
n_clinical = len(non_survival_vars)
n_outcomes = outcome_effect.shape[1]
fig = plt.figure(figsize=FIG3C_STYLE['figsize'])
grid = fig.add_gridspec(nrows=1, ncols=4, width_ratios=[n_clinical, 0.45, 3.4, 0.45], wspace=0.16)
ax_corr = fig.add_subplot(grid[0, 0])
cax_corr = fig.add_subplot(grid[0, 1])
ax_outcome = fig.add_subplot(grid[0, 2])
cax_outcome = fig.add_subplot(grid[0, 3])
clinical_cmap = LinearSegmentedColormap.from_list('deep_clinical_diverging', ['#1F5A8A', '#4B93B5', '#A8CCDA', '#F7F7F5', '#E7B09A', '#C65E4A', '#8F2F2C'])
im_corr = ax_corr.imshow(spearman_r.values.astype(float), cmap=clinical_cmap, vmin=SPEARMAN_VMIN, vmax=SPEARMAN_VMAX, aspect='auto')
colorbar_corr = fig.colorbar(im_corr, cax=cax_corr)
colorbar_corr.set_label('Spearman r', fontsize=FIG3C_STYLE['colorbar_label'])
colorbar_corr.ax.tick_params(labelsize=FIG3C_STYLE['colorbar_tick'])
ax_corr.set_xticks(np.arange(n_clinical))
ax_corr.set_xticklabels([display_name.get(column, column) for column in non_survival_vars], rotation=45, ha='right', fontsize=FIG3C_STYLE['column_label'])
ax_corr.set_yticks(np.arange(n_rows))
ax_corr.set_yticklabels(row_order, fontsize=FIG3C_STYLE['row_label'])
if SHOW_CELL_VALUES:
    for row_index, axis in enumerate(row_order):
        for column_index, variable in enumerate(non_survival_vars):
            value = float(spearman_r.loc[axis, variable])
            q_value = float(spearman_q.loc[axis, variable])
            if not np.isfinite(value):
                continue
            symbol = significance_symbol(q_value, value, CLINICAL_EFFECT_THRESHOLD)
            ax_corr.text(column_index, row_index, f'{value:.2f}', ha='center', va='center', fontsize=FIG3C_STYLE['clinical_value'], color='#222222' if abs(value) >= CLINICAL_EFFECT_THRESHOLD else '#4A4A4A')
            if symbol:
                ax_corr.text(column_index + FIG3C_STYLE['clinical_symbol_x_offset'], row_index + FIG3C_STYLE['clinical_symbol_y_offset'], symbol, ha='center', va='center', fontsize=FIG3C_STYLE['clinical_symbol'], color='#333333', fontweight='bold', zorder=6)
ax_corr.set_xticks(np.arange(-0.5, n_clinical, 1), minor=True)
ax_corr.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
ax_corr.grid(False)
ax_corr.tick_params(which='minor', bottom=False, left=False)
start_column = 0
band_colors = ['#E7E7E7', '#DCEAF2', '#F1DEDA', '#DDEDDD']
for domain_index, (domain_name, variables) in enumerate(clinical_groups.items()):
    length = len(variables)
    center = start_column + (length - 1) / 2
    if start_column > 0:
        ax_corr.axvline(start_column - 0.5, color='black', linewidth=FIG3C_STYLE['group_separator_linewidth'])
    rectangle = Rectangle((start_column - 0.5, -1.35), length, 0.2, facecolor=band_colors[domain_index % len(band_colors)], edgecolor='none', clip_on=False)
    ax_corr.add_patch(rectangle)
    ax_corr.text(center, -1.62, domain_name, ha='center', va='center', fontsize=FIG3C_STYLE['domain_header'], fontweight='bold')
    start_column += length
outcome_abs_max = np.nanmax(np.abs(outcome_effect.values.astype(float)))
if not np.isfinite(outcome_abs_max):
    outcome_abs_max = COX_MIN_ABS_LIMIT
outcome_limit = np.ceil(max(COX_MIN_ABS_LIMIT, min(COX_MAX_ABS_LIMIT, outcome_abs_max)) * 10) / 10
soft_outcome_cmap = LinearSegmentedColormap.from_list('deep_outcome_diverging', ['#2F6D91', '#88B3C5', '#F7F7F5', '#D99D82', '#9E463E'])
outcome_norm = TwoSlopeNorm(vmin=-outcome_limit, vcenter=0, vmax=outcome_limit)
im_outcome = ax_outcome.imshow(outcome_effect.values.astype(float), cmap=soft_outcome_cmap, norm=outcome_norm, aspect='auto')
colorbar_outcome = fig.colorbar(im_outcome, cax=cax_outcome)
colorbar_outcome.set_label('Adjusted log-odds / log(HR) per 1-SD axis', fontsize=FIG3C_STYLE['colorbar_label'])
colorbar_outcome.ax.tick_params(labelsize=FIG3C_STYLE['colorbar_tick'])
ax_outcome.set_xticks(np.arange(n_outcomes))
ax_outcome.set_xticklabels(['Response', 'PFS', 'OS'], fontsize=FIG3C_STYLE['column_label'])
ax_outcome.set_yticks(np.arange(n_rows))
ax_outcome.set_yticklabels([])
if SHOW_CELL_VALUES:
    for row_index, axis in enumerate(row_order):
        for column_index, endpoint in enumerate(outcome_effect.columns):
            value = float(outcome_effect.loc[axis, endpoint])
            q_value = float(outcome_q.loc[axis, endpoint])
            if not np.isfinite(value):
                continue
            ax_outcome.text(column_index, row_index, f'{value:.2f}', ha='center', va='center', fontsize=FIG3C_STYLE['outcome_value'], color='#222222' if abs(value) >= OUTCOME_EFFECT_THRESHOLD else '#505050')
            if np.isfinite(q_value) and q_value < FDR_PRIMARY and (abs(value) >= OUTCOME_EFFECT_THRESHOLD):
                ax_outcome.scatter(column_index + FIG3C_STYLE['outcome_dot_x_offset'], row_index + FIG3C_STYLE['outcome_dot_y_offset'], s=FIG3C_STYLE['outcome_dot_filled'], facecolor='#333333', edgecolor='none', zorder=6)
            elif np.isfinite(q_value) and q_value < FDR_SUGGESTIVE and (abs(value) >= OUTCOME_EFFECT_THRESHOLD):
                ax_outcome.scatter(column_index + FIG3C_STYLE['outcome_dot_x_offset'], row_index + FIG3C_STYLE['outcome_dot_y_offset'], s=FIG3C_STYLE['outcome_dot_open'], facecolor='white', edgecolor='#333333', linewidth=FIG3C_STYLE['outcome_dot_linewidth'], zorder=6)
ax_outcome.set_xticks(np.arange(-0.5, n_outcomes, 1), minor=True)
ax_outcome.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
ax_outcome.grid(False)
ax_outcome.tick_params(which='minor', bottom=False, left=False)
ax_outcome.text(1.0, -1.62, 'Adjusted outcomes', ha='center', va='center', fontsize=FIG3C_STYLE['outcome_header'], fontweight='bold')
for axis_object in [ax_corr, ax_outcome]:
    for boundary in [2.5, 5.5, 8.5, 11.5]:
        axis_object.axhline(boundary, color='black', linewidth=FIG3C_STYLE['group_separator_linewidth'])
modality_centers = {'Metagenomics': 1, 'Species': 4, 'Blood': 7, 'Urine': 10, 'Radiomics': 13}
for modality, center in modality_centers.items():
    ax_corr.text(-3.15, center, modality, ha='right', va='center', fontsize=FIG3C_STYLE['modality_group'], fontweight='bold', rotation=90)
ax_corr.set_ylim(n_rows - 0.5, -1.85)
ax_outcome.set_ylim(n_rows - 0.5, -1.85)
ax_corr.set_xlim(-0.5, n_clinical - 0.5)
ax_outcome.set_xlim(-0.5, n_outcomes - 0.5)
plt.tight_layout(rect=[0, 0.015, 1, 0.985])
figure_png = os.path.join(out_dir, get_value('science.figures.plot_076', 'plot_076.png'))
figure_pdf = os.path.join(out_dir, get_value('science.figures.plot_077', 'plot_077.pdf'))
plt.savefig(figure_png, dpi=600, bbox_inches='tight')
plt.savefig(figure_pdf, bbox_inches='tight')
plt.show()



for table, filename in [(spearman_r, get_value('science.tables.clinical_014', 'clinical_014.csv')), (spearman_p, get_value('science.tables.clinical_015', 'clinical_015.csv')), (spearman_q, get_value('science.tables.clinical_016', 'clinical_016.csv')), (outcome_effect, get_value('science.tables.result_053', 'result_053.csv')), (outcome_p, get_value('science.tables.result_054', 'result_054.csv')), (outcome_q, get_value('science.tables.result_055', 'result_055.csv')), (cox_response_adjusted_log_hr, get_value('science.tables.result_056', 'result_056.csv')), (cox_response_adjusted_p, get_value('science.tables.result_057', 'result_057.csv')), (cox_response_adjusted_q, get_value('science.tables.result_058', 'result_058.csv'))]:
    table.to_csv(os.path.join(out_dir, filename), encoding='utf-8-sig')
modal_axes.reset_index().rename(columns={'index': 'sample'}).to_csv(os.path.join(out_dir, get_value('science.tables.result_059', 'result_059.csv')), index=False, encoding='utf-8-sig')
adjustment_df.reset_index().rename(columns={'index': 'sample'}).to_csv(os.path.join(out_dir, get_value('science.tables.clinical_017', 'clinical_017.csv')), index=False, encoding='utf-8-sig')



In [ ]:
import os
import warnings
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'Arial', 'font.size': 11.0, 'axes.titlesize': 14.0, 'axes.labelsize': 11.5, 'xtick.labelsize': 10.5, 'ytick.labelsize': 10.5, 'legend.fontsize': 10.0, 'figure.titlesize': 16.0, 'text.color': '#1F1F1F', 'axes.labelcolor': '#1F1F1F', 'axes.titlecolor': '#1F1F1F', 'xtick.color': '#1F1F1F', 'ytick.color': '#1F1F1F', 'axes.edgecolor': '#2B2B2B', 'axes.grid': False, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none'})
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import rankdata, spearmanr
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import pairwise_distances, roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
warnings.filterwarnings('ignore', category=RuntimeWarning)
FIG3D1_STYLE = {'figure_size': (15.8, 6.5), 'main_title': 16.0, 'panel_title': 14.0, 'axis_label': 11.5, 'axis_tick': 10.5, 'footnote': 9.0, 'bar_pair_label': 10.5, 'bar_value': 10.0, 'bar_symbol': 10.0, 'bar_value_offset_pt': 5.0, 'bar_symbol_offset_pt': 28.0, 'bar_symbol_y_offset_pt': -1.5, 'benchmark_row_label': 10.5, 'benchmark_column_label': 10.5, 'benchmark_value': 10.0, 'benchmark_delta': 9.5, 'benchmark_colorbar_label': 11.0, 'benchmark_colorbar_tick': 10.0, 'benchmark_grid_width': 1.0, 'left_panel_width': 1.15, 'right_panel_width': 1.0, 'panel_spacing': 0.34}
FIG3D1_COLORS = {'bar_positive': '#557A8D', 'bar_negative': '#758C98', 'zero_line': '#666666', 'text': '#1F1F1F', 'muted_text': '#333333', 'grid': '#FFFFFF', 'single_outline': '#969696', 'fusion_outline': '#4F4F4F', 'low': '#D7E3E8', 'mid': '#F6F5F1', 'high': '#C99B8D'}
plt.rcParams.update({'font.family': 'Arial', 'axes.unicode_minus': False, 'pdf.fonttype': 42, 'ps.fonttype': 42})
benchmark_cmap = LinearSegmentedColormap.from_list('calm_blue_neutral_terracotta', [FIG3D1_COLORS['low'], FIG3D1_COLORS['mid'], FIG3D1_COLORS['high']])
project_dir = str(project_path(get_value('science.paths.directory_008', 'directory_008')))
input_dir = os.path.join(project_dir, get_value('science.paths.directory_009', 'directory_009'))
output_root = os.path.join(project_dir, get_value('science.paths.directory_053', 'directory_053'))
LABEL_FILENAME = globals().get('LABEL_FILENAME', get_value('science.tables.outcomes_001', 'outcomes_001.csv'))
PREFERRED_RESPONSE_COLUMN = globals().get('PREFERRED_RESPONSE_COLUMN', 'Response')
label_path = os.path.join(input_dir, LABEL_FILENAME)
MODALITY_FILES = {'Metagenomics': os.path.join(input_dir, get_value('science.tables.features_001', 'features_001.csv.gz')), 'Species': os.path.join(input_dir, get_value('science.tables.result_060', 'result_060.csv')), 'Blood': os.path.join(input_dir, get_value('science.tables.features_003', 'features_003.csv.gz')), 'Urine': os.path.join(input_dir, get_value('science.tables.features_004', 'features_004.csv.gz')), 'Radiomics': os.path.join(input_dir, get_value('science.tables.features_002', 'features_002.csv.gz'))}
MODALITY_ORDER = ['Metagenomics', 'Species', 'Blood', 'Urine', 'Radiomics']
SINGLE_METHODS = MODALITY_ORDER.copy()
FUSION_METHODS = ['Pooled PCA fusion', 'Block-PCA early fusion', 'Equal-weight late fusion']
BENCHMARK_METHODS = SINGLE_METHODS + FUSION_METHODS
out_dir = os.path.join(output_root, get_value('science.paths.directory_058', 'directory_058'))
os.makedirs(out_dir, exist_ok=True)
N_SPLITS = 5
N_REPEATS = 5
RANDOM_STATE = 42
LATENT_COMPONENTS_PER_MODALITY = 5
POOLED_PCA_COMPONENTS = 15
POOLED_MAX_FEATURES_PER_MODALITY = 500
SPECIES_MIN_PREVALENCE = 0.1
LOGISTIC_C = 1.0
COX_PENALIZER_SEQUENCE = [0.1, 0.5, 1.0, 2.0]
SIMILARITY_PCA_VARIANCE_TARGET = 0.7
SIMILARITY_PCA_MAX_COMPONENTS = None
SIMILARITY_PCA_MIN_COMPONENTS = 2
N_PERMUTATIONS = 4999
required_paths = [label_path] + list(MODALITY_FILES.values())
missing_paths = [path for path in required_paths if not os.path.exists(path)]
if missing_paths:
    print('\nMissing required files:')
    for path in missing_paths:
        print(' -', path)
    raise FileNotFoundError('Please correct the file paths in Section 1.')

def normalize_sample_id(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    if value.endswith('.0') and value[:-2].isdigit():
        value = value[:-2]
    return value

def detect_column(columns, candidates, required=True):
    normalized = {str(column).strip().lower(): column for column in columns}
    for candidate in candidates:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]
    if required:
        raise KeyError(f'Could not find any of these columns: {candidates}')
    return None

def read_matrix(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    df.columns = [str(column).strip() for column in df.columns]
    sample_col = detect_column(df.columns, ['sample', 'sample_id', 'sample id', 'id'], required=False)
    if sample_col is None:
        sample_col = df.columns[0]
    df = df.rename(columns={sample_col: 'sample'})
    df['sample'] = df['sample'].map(normalize_sample_id)
    df = df.dropna(subset=['sample']).drop_duplicates('sample').set_index('sample')
    return df.apply(pd.to_numeric, errors='coerce')

def read_labels(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    df.columns = [str(column).strip() for column in df.columns]
    sample_col = detect_column(df.columns, ['sample', 'sample_id', 'sample id', 'id'])
    response_candidates = list(dict.fromkeys([PREFERRED_RESPONSE_COLUMN, 'Response', 'Response', 'response_label', 'Group']))
    response_col = detect_column(df.columns, response_candidates)
    pfs_time_col = detect_column(df.columns, ['pfs_time', 'pfs time'])
    pfs_status_col = detect_column(df.columns, ['pfs_status', 'pfs_event', 'pfs status'])
    os_time_col = detect_column(df.columns, ['os_time', 'os time'])
    os_status_col = detect_column(df.columns, ['os_status', 'os_event', 'os status'])
    
    print(f'[Outcome audit] Response column used: {response_col}')
    pass
    labels = pd.DataFrame({'sample': df[sample_col].map(normalize_sample_id), 'Response': pd.to_numeric(df[response_col], errors='coerce'), 'PFS_time': pd.to_numeric(df[pfs_time_col], errors='coerce'), 'PFS_status': pd.to_numeric(df[pfs_status_col], errors='coerce'), 'OS_time': pd.to_numeric(df[os_time_col], errors='coerce'), 'OS_status': pd.to_numeric(df[os_status_col], errors='coerce')})
    labels = labels.dropna(subset=['sample']).drop_duplicates('sample').set_index('sample')
    required = ['Response', 'PFS_time', 'PFS_status', 'OS_time', 'OS_status']
    labels = labels.dropna(subset=required)
    labels['Response'] = labels['Response'].astype(int)
    labels['PFS_status'] = labels['PFS_status'].astype(int)
    labels['OS_status'] = labels['OS_status'].astype(int)
    print(f"[Outcome audit] n={len(labels)}, Response={labels['Response'].value_counts().sort_index().to_dict()}, PFS events={int(labels['PFS_status'].sum())}, OS events={int(labels['OS_status'].sum())}")
    return labels

def bh_fdr(p_values):
    p_values = np.asarray(p_values, dtype=float)
    q_values = np.full(p_values.shape, np.nan)
    valid = np.isfinite(p_values)
    if valid.sum() == 0:
        return q_values
    p = p_values[valid]
    order = np.argsort(p)
    ranked = p[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    restored = np.empty_like(adjusted)
    restored[order] = adjusted
    q_values[valid] = restored
    return q_values

def display_q_symbol(q_value):
    if not np.isfinite(q_value):
        return ''
    if q_value < 0.05:
        return '*'
    if q_value < 0.1:
        return '·'
    return ''
modality_matrices = {name: read_matrix(path) for name, path in MODALITY_FILES.items()}
labels = read_labels(label_path)
common_samples = set(labels.index)
for matrix in modality_matrices.values():
    common_samples &= set(matrix.index)
common_samples = [sample for sample in labels.index if sample in common_samples]
if len(common_samples) < 30:
    raise ValueError(f'Only {len(common_samples)} common samples were found.')
labels = labels.loc[common_samples].copy()
modality_matrices = {name: matrix.loc[common_samples].copy() for name, matrix in modality_matrices.items()}
print(f'Common samples across labels and all five modalities: n={len(common_samples)}')
for name in MODALITY_ORDER:
    pass
cv_strata = labels['Response'].astype(str) + '_' + labels['OS_status'].astype(str)
stratum_counts = cv_strata.value_counts()
if stratum_counts.min() < N_SPLITS:
    print('Composite strata contain a rare group; falling back to Response-only stratification.')
    cv_strata = labels['Response'].astype(str)
cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
CV_SPLITS = [(train_idx, test_idx) for train_idx, test_idx in cv.split(np.zeros(len(labels)), cv_strata)]
print(f'Repeated outer CV folds: {len(CV_SPLITS)}')

def _remove_invalid_training_features(train_df, test_df):
    train_df = train_df.replace([np.inf, -np.inf], np.nan)
    test_df = test_df.replace([np.inf, -np.inf], np.nan)
    valid_columns = train_df.columns[train_df.notna().any(axis=0)]
    train_df = train_df.loc[:, valid_columns]
    test_df = test_df.reindex(columns=valid_columns)
    variance = train_df.var(axis=0, skipna=True, ddof=0)
    valid_columns = variance.index[np.isfinite(variance) & (variance > 1e-12)]
    train_df = train_df.loc[:, valid_columns]
    test_df = test_df.reindex(columns=valid_columns)
    if train_df.shape[1] == 0:
        raise ValueError('No valid training features remain after filtering.')
    return (train_df, test_df)

def prepare_species_raw(train_df, test_df):
    train_df = train_df.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
    test_df = test_df.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
    nonzero_columns = train_df.columns[train_df.sum(axis=0) > 0]
    train_df = train_df.loc[:, nonzero_columns]
    test_df = test_df.reindex(columns=nonzero_columns, fill_value=0)
    prevalence = (train_df > 0).mean(axis=0)
    selected = prevalence.index[prevalence >= SPECIES_MIN_PREVALENCE]
    train_df = train_df.loc[:, selected]
    test_df = test_df.reindex(columns=selected, fill_value=0)
    if train_df.shape[1] == 0:
        raise ValueError('No Species features passed the training-fold prevalence filter.')
    train_sum = train_df.sum(axis=1).replace(0, np.nan)
    test_sum = test_df.sum(axis=1).replace(0, np.nan)
    train_rel = train_df.div(train_sum, axis=0).fillna(0)
    test_rel = test_df.div(test_sum, axis=0).fillna(0)
    return (np.sqrt(train_rel), np.sqrt(test_rel))

def prepare_standard_raw(train_df, test_df):
    train_df, test_df = _remove_invalid_training_features(train_df, test_df)
    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()
    train_array = imputer.fit_transform(train_df)
    test_array = imputer.transform(test_df)
    train_array = scaler.fit_transform(train_array)
    test_array = scaler.transform(test_array)
    columns = train_df.columns
    train_out = pd.DataFrame(train_array, index=train_df.index, columns=columns)
    test_out = pd.DataFrame(test_array, index=test_df.index, columns=columns)
    return (train_out, test_out)

def prepare_raw_modality_block(modality, train_indices, test_indices, max_features=None):
    matrix = modality_matrices[modality]
    train_df = matrix.iloc[train_indices].copy()
    test_df = matrix.iloc[test_indices].copy()
    if modality == 'Species':
        train_processed, test_processed = prepare_species_raw(train_df, test_df)
        scaler = StandardScaler()
        train_array = scaler.fit_transform(train_processed)
        test_array = scaler.transform(test_processed)
        train_processed = pd.DataFrame(train_array, index=train_processed.index, columns=train_processed.columns)
        test_processed = pd.DataFrame(test_array, index=test_processed.index, columns=test_processed.columns)
    else:
        train_processed, test_processed = prepare_standard_raw(train_df, test_df)
    if max_features is not None and train_processed.shape[1] > max_features:
        variance = train_processed.var(axis=0, ddof=0).sort_values(ascending=False)
        selected = variance.index[:max_features]
        train_processed = train_processed.loc[:, selected]
        test_processed = test_processed.reindex(columns=selected, fill_value=0)
    return (train_processed.to_numpy(dtype=float), test_processed.to_numpy(dtype=float))

def prepare_latent_modality_block(modality, train_indices, test_indices, n_components=None):
    if n_components is None:
        n_components = LATENT_COMPONENTS_PER_MODALITY
    train_raw, test_raw = prepare_raw_modality_block(modality, train_indices, test_indices)
    max_components = min(n_components, train_raw.shape[0] - 1, train_raw.shape[1])
    if max_components < 1:
        raise ValueError(f'Too few dimensions for {modality} PCA.')
    pca = PCA(n_components=max_components, svd_solver='randomized', random_state=RANDOM_STATE)
    train_latent = pca.fit_transform(train_raw)
    test_latent = pca.transform(test_raw)
    scaler = StandardScaler()
    train_latent = scaler.fit_transform(train_latent)
    test_latent = scaler.transform(test_latent)
    return (train_latent, test_latent)

def build_block_pca_fusion(train_indices, test_indices, modalities=MODALITY_ORDER):
    train_blocks, test_blocks = ([], [])
    for modality in modalities:
        train_block, test_block = prepare_latent_modality_block(modality, train_indices, test_indices)
        train_blocks.append(train_block)
        test_blocks.append(test_block)
    train_design = np.column_stack(train_blocks)
    test_design = np.column_stack(test_blocks)
    scaler = StandardScaler()
    return (scaler.fit_transform(train_design), scaler.transform(test_design))

def build_pooled_pca_fusion(train_indices, test_indices, modalities=MODALITY_ORDER):
    train_blocks, test_blocks = ([], [])
    for modality in modalities:
        train_block, test_block = prepare_raw_modality_block(modality, train_indices, test_indices, max_features=POOLED_MAX_FEATURES_PER_MODALITY)
        train_blocks.append(train_block)
        test_blocks.append(test_block)
    train_pooled = np.column_stack(train_blocks)
    test_pooled = np.column_stack(test_blocks)
    n_components = min(POOLED_PCA_COMPONENTS, train_pooled.shape[0] - 1, train_pooled.shape[1])
    if n_components < 1:
        raise ValueError('Too few dimensions for pooled PCA fusion.')
    pca = PCA(n_components=n_components, svd_solver='randomized', random_state=RANDOM_STATE)
    train_design = pca.fit_transform(train_pooled)
    test_design = pca.transform(test_pooled)
    scaler = StandardScaler()
    return (scaler.fit_transform(train_design), scaler.transform(test_design))

def fit_response_model(train_design, train_labels, test_design):
    model = LogisticRegression(penalty='l2', C=LOGISTIC_C, solver='liblinear', max_iter=5000, random_state=RANDOM_STATE)
    model.fit(train_design, train_labels['Response'].astype(int))
    train_probability = model.predict_proba(train_design)[:, 1]
    test_probability = model.predict_proba(test_design)[:, 1]
    return (train_probability, test_probability)

def fit_cox_model(train_design, train_labels, test_design, endpoint):
    feature_names = [f'x_{index}' for index in range(train_design.shape[1])]
    train_survival = pd.DataFrame(train_design, columns=feature_names, index=train_labels.index)
    test_survival = pd.DataFrame(test_design, columns=feature_names)
    train_survival['time'] = train_labels[f'{endpoint}_time'].to_numpy()
    train_survival['event'] = train_labels[f'{endpoint}_status'].to_numpy()
    last_error = None
    for penalizer in COX_PENALIZER_SEQUENCE:
        try:
            model = CoxPHFitter(penalizer=penalizer)
            model.fit(train_survival, duration_col='time', event_col='event', show_progress=False)
            train_risk = model.predict_log_partial_hazard(train_survival[feature_names]).to_numpy()
            test_risk = model.predict_log_partial_hazard(test_survival).to_numpy()
            return (train_risk, test_risk, penalizer)
        except Exception as error:
            last_error = error
    raise RuntimeError(f'Cox model failed for {endpoint}: {last_error}')

def safe_auc(y_true, probability):
    y_true = np.asarray(y_true, dtype=int)
    probability = np.asarray(probability, dtype=float)
    if np.unique(y_true).size < 2:
        return np.nan
    return roc_auc_score(y_true, probability)

def safe_c_index(time, status, risk):
    try:
        return concordance_index(np.asarray(time), -np.asarray(risk), np.asarray(status))
    except Exception:
        return np.nan

def evaluate_design(train_design, test_design, train_labels, test_labels):
    _, response_probability = fit_response_model(train_design, train_labels, test_design)
    metrics = {'Response_AUC': safe_auc(test_labels['Response'], response_probability)}
    for endpoint in ['PFS', 'OS']:
        _, test_risk, _ = fit_cox_model(train_design, train_labels, test_design, endpoint)
        metrics[f'{endpoint}_Cindex'] = safe_c_index(test_labels[f'{endpoint}_time'], test_labels[f'{endpoint}_status'], test_risk)
    return metrics

def evaluate_equal_weight_late_fusion(latent_cache, train_labels, test_labels):
    response_predictions = []
    survival_predictions = {'PFS': [], 'OS': []}
    for modality in MODALITY_ORDER:
        train_design, test_design = latent_cache[modality]
        _, response_probability = fit_response_model(train_design, train_labels, test_design)
        response_predictions.append(response_probability)
        for endpoint in ['PFS', 'OS']:
            train_risk, test_risk, _ = fit_cox_model(train_design, train_labels, test_design, endpoint)
            train_mean = np.mean(train_risk)
            train_sd = np.std(train_risk, ddof=0)
            if not np.isfinite(train_sd) or train_sd < 1e-08:
                train_sd = 1.0
            survival_predictions[endpoint].append((test_risk - train_mean) / train_sd)
    mean_probability = np.mean(np.column_stack(response_predictions), axis=1)
    metrics = {'Response_AUC': safe_auc(test_labels['Response'], mean_probability)}
    for endpoint in ['PFS', 'OS']:
        mean_risk = np.mean(np.column_stack(survival_predictions[endpoint]), axis=1)
        metrics[f'{endpoint}_Cindex'] = safe_c_index(test_labels[f'{endpoint}_time'], test_labels[f'{endpoint}_status'], mean_risk)
    return metrics
benchmark_records = []
for fold_number, (train_indices, test_indices) in enumerate(CV_SPLITS, start=1):
    train_labels = labels.iloc[train_indices].copy()
    test_labels = labels.iloc[test_indices].copy()
    
    latent_cache = {modality: prepare_latent_modality_block(modality, train_indices, test_indices) for modality in MODALITY_ORDER}
    for modality in SINGLE_METHODS:
        train_design, test_design = latent_cache[modality]
        metrics = evaluate_design(train_design, test_design, train_labels, test_labels)
        for metric, value in metrics.items():
            benchmark_records.append({'Fold': fold_number, 'Method': modality, 'Metric': metric, 'Value': value})
    train_design, test_design = build_pooled_pca_fusion(train_indices, test_indices)
    metrics = evaluate_design(train_design, test_design, train_labels, test_labels)
    for metric, value in metrics.items():
        benchmark_records.append({'Fold': fold_number, 'Method': 'Pooled PCA fusion', 'Metric': metric, 'Value': value})
    train_design = np.column_stack([latent_cache[modality][0] for modality in MODALITY_ORDER])
    test_design = np.column_stack([latent_cache[modality][1] for modality in MODALITY_ORDER])
    fusion_scaler = StandardScaler()
    train_design = fusion_scaler.fit_transform(train_design)
    test_design = fusion_scaler.transform(test_design)
    metrics = evaluate_design(train_design, test_design, train_labels, test_labels)
    for metric, value in metrics.items():
        benchmark_records.append({'Fold': fold_number, 'Method': 'Block-PCA early fusion', 'Metric': metric, 'Value': value})
    metrics = evaluate_equal_weight_late_fusion(latent_cache, train_labels, test_labels)
    for metric, value in metrics.items():
        benchmark_records.append({'Fold': fold_number, 'Method': 'Equal-weight late fusion', 'Metric': metric, 'Value': value})
benchmark_fold_results = pd.DataFrame(benchmark_records)
benchmark_summary = benchmark_fold_results.groupby(['Method', 'Metric'], as_index=False).agg(Mean=('Value', 'mean'), Median=('Value', 'median'), Lower_95=('Value', lambda values: np.nanquantile(values, 0.025)), Upper_95=('Value', lambda values: np.nanquantile(values, 0.975)), SD=('Value', 'std'), Fold_N=('Value', lambda values: np.isfinite(values).sum()))
benchmark_fold_results.to_csv(os.path.join(out_dir, get_value('science.tables.metrics_026', 'metrics_026.csv')), index=False, encoding='utf-8-sig')
benchmark_summary.to_csv(os.path.join(out_dir, get_value('science.tables.metrics_027', 'metrics_027.csv')), index=False, encoding='utf-8-sig')

def normalize_distance(distance_matrix):
    distance = np.asarray(distance_matrix, dtype=float)
    distance = np.nan_to_num(distance, nan=0.0, posinf=0.0, neginf=0.0)
    upper = distance[np.triu_indices_from(distance, k=1)]
    positive = upper[upper > 0]
    scale = np.median(positive) if positive.size else 1.0
    if not np.isfinite(scale) or scale <= 0:
        scale = 1.0
    return distance / scale

def build_similarity_distance(modality):
    matrix = modality_matrices[modality]
    if modality == 'Species':
        X = matrix.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
        X = X.loc[:, X.sum(axis=0) > 0]
        prevalence = (X > 0).mean(axis=0)
        X = X.loc[:, prevalence >= SPECIES_MIN_PREVALENCE]
        row_sum = X.sum(axis=1).replace(0, np.nan)
        X_relative = X.div(row_sum, axis=0).fillna(0)
        distance = pairwise_distances(X_relative, metric='braycurtis')
        metadata = {'Features': X_relative.shape[1], 'Components': np.nan, 'Variance': np.nan}
        return (normalize_distance(distance), metadata)
    X = matrix.replace([np.inf, -np.inf], np.nan)
    X = X.loc[:, X.notna().any(axis=0)]
    variance = X.var(axis=0, skipna=True, ddof=0)
    X = X.loc[:, np.isfinite(variance) & (variance > 1e-12)]
    X = X.fillna(X.median())
    X_scaled = StandardScaler().fit_transform(X)
    max_components = min(X_scaled.shape[0] - 1, X_scaled.shape[1])
    if SIMILARITY_PCA_MAX_COMPONENTS is not None:
        max_components = min(max_components, int(SIMILARITY_PCA_MAX_COMPONENTS))
    pca = PCA(n_components=max_components, svd_solver='full')
    scores = pca.fit_transform(X_scaled)
    cumulative = np.cumsum(pca.explained_variance_ratio_)
    n_keep = int(np.searchsorted(cumulative, SIMILARITY_PCA_VARIANCE_TARGET) + 1)
    n_keep = max(min(SIMILARITY_PCA_MIN_COMPONENTS, max_components), n_keep)
    n_keep = min(n_keep, max_components)
    distance = pairwise_distances(scores[:, :n_keep], metric='euclidean')
    metadata = {'Features': X.shape[1], 'Components': n_keep, 'Variance': cumulative[n_keep - 1]}
    return (normalize_distance(distance), metadata)

def upper_triangle_vector(distance_matrix):
    return distance_matrix[np.triu_indices_from(distance_matrix, k=1)]

def mantel_style_spearman(distance_a, distance_b, n_permutations, random_state):
    rng = np.random.default_rng(random_state)
    triangle = np.triu_indices_from(distance_a, k=1)
    vector_a = distance_a[triangle]
    vector_b = distance_b[triangle]
    rank_a = rankdata(vector_a)
    rank_b = rankdata(vector_b)
    rank_matrix_b = np.zeros_like(distance_b, dtype=float)
    rank_matrix_b[triangle] = rank_b
    rank_matrix_b[triangle[1], triangle[0]] = rank_b
    rank_a = (rank_a - rank_a.mean()) / rank_a.std(ddof=0)
    observed_rank_b = (rank_b - rank_b.mean()) / rank_b.std(ddof=0)
    observed = float(np.mean(rank_a * observed_rank_b))
    permuted = np.empty(n_permutations, dtype=float)
    for index in range(n_permutations):
        order = rng.permutation(distance_b.shape[0])
        permuted_rank_b = rank_matrix_b[np.ix_(order, order)][triangle]
        permuted_rank_b = (permuted_rank_b - permuted_rank_b.mean()) / permuted_rank_b.std(ddof=0)
        permuted[index] = np.mean(rank_a * permuted_rank_b)
    p_value = (1 + np.sum(np.abs(permuted) >= abs(observed))) / (n_permutations + 1)
    return (observed, p_value)
similarity_distances = {}
similarity_metadata = []
for modality in MODALITY_ORDER:
    distance, metadata = build_similarity_distance(modality)
    similarity_distances[modality] = distance
    similarity_metadata.append({'Modality': modality, **metadata})
pair_records = []
for pair_index, (modality_a, modality_b) in enumerate(combinations(MODALITY_ORDER, 2), start=1):
    rho, p_value = mantel_style_spearman(similarity_distances[modality_a], similarity_distances[modality_b], n_permutations=N_PERMUTATIONS, random_state=RANDOM_STATE + pair_index)
    pair_records.append({'Modality_A': modality_a, 'Modality_B': modality_b, 'Pair': f'{modality_a} vs {modality_b}', 'Distance_similarity_rho': rho, 'P_value': p_value})
pair_results = pd.DataFrame(pair_records)
pair_results['FDR'] = bh_fdr(pair_results['P_value'].to_numpy())
pair_results.to_csv(os.path.join(out_dir, get_value('science.tables.result_061', 'result_061.csv')), index=False, encoding='utf-8-sig')
pd.DataFrame(similarity_metadata).to_csv(os.path.join(out_dir, get_value('science.tables.result_062', 'result_062.csv')), index=False, encoding='utf-8-sig')
metric_order = ['Response_AUC', 'PFS_Cindex', 'OS_Cindex']
metric_display = ['Response AUC', 'PFS C-index', 'OS C-index']
benchmark_matrix = benchmark_summary.pivot(index='Method', columns='Metric', values='Mean').reindex(index=BENCHMARK_METHODS, columns=metric_order)
single_mean_matrix = benchmark_matrix.loc[SINGLE_METHODS]
best_single_mean = single_mean_matrix.max(axis=0)
best_single_name = single_mean_matrix.idxmax(axis=0)
benchmark_delta_matrix = benchmark_matrix.subtract(best_single_mean, axis=1)
best_single_table = pd.DataFrame({'Metric': metric_order, 'Best_single_modality': [best_single_name[metric] for metric in metric_order], 'Best_single_mean': [best_single_mean[metric] for metric in metric_order]})
best_single_table.to_csv(os.path.join(out_dir, get_value('science.tables.result_063', 'result_063.csv')), index=False, encoding='utf-8-sig')
fig = plt.figure(figsize=FIG3D1_STYLE['figure_size'])
grid = fig.add_gridspec(nrows=1, ncols=2, width_ratios=[FIG3D1_STYLE['left_panel_width'], FIG3D1_STYLE['right_panel_width']], wspace=FIG3D1_STYLE['panel_spacing'])
ax_bar = fig.add_subplot(grid[0, 0])
ax_benchmark = fig.add_subplot(grid[0, 1])
plot_pairs = pair_results.sort_values('Distance_similarity_rho', ascending=True).reset_index(drop=True)
bar_values = plot_pairs['Distance_similarity_rho'].to_numpy()
bar_y = np.arange(len(plot_pairs))
bar_colors = [FIG3D1_COLORS['bar_positive'] if value >= 0 else FIG3D1_COLORS['bar_negative'] for value in bar_values]
ax_bar.barh(bar_y, bar_values, color=bar_colors, edgecolor='none', height=0.78)
ax_bar.axvline(0, color=FIG3D1_COLORS['zero_line'], linewidth=0.8)
ax_bar.set_yticks(bar_y)
ax_bar.set_yticklabels(plot_pairs['Pair'], fontsize=FIG3D1_STYLE['bar_pair_label'])
ax_bar.tick_params(axis='x', labelsize=FIG3D1_STYLE['axis_tick'])
ax_bar.set_xlabel('Distance-matrix Spearman ρ', fontsize=FIG3D1_STYLE['axis_label'])
ax_bar.set_title('Overall cross-modal similarity', fontsize=FIG3D1_STYLE['panel_title'], pad=12)
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)
x_span = max(0.05, np.nanmax(bar_values) - np.nanmin(bar_values))
ax_bar.set_xlim(np.nanmin(bar_values) - 0.08 * x_span, np.nanmax(bar_values) + 0.25 * x_span)
for row_index, row in plot_pairs.iterrows():
    value = row['Distance_similarity_rho']
    symbol = display_q_symbol(row['FDR'])
    direction = 1 if value >= 0 else -1
    ha = 'left' if value >= 0 else 'right'
    ax_bar.annotate(f'{value:.2f}', xy=(value, row_index), xytext=(direction * FIG3D1_STYLE['bar_value_offset_pt'], 0), textcoords='offset points', ha=ha, va='center', fontsize=FIG3D1_STYLE['bar_value'], color=FIG3D1_COLORS['text'])
    if symbol:
        ax_bar.annotate(symbol, xy=(value, row_index), xytext=(direction * FIG3D1_STYLE['bar_symbol_offset_pt'], FIG3D1_STYLE['bar_symbol_y_offset_pt']), textcoords='offset points', ha=ha, va='center', fontsize=FIG3D1_STYLE['bar_symbol'], color=FIG3D1_COLORS['muted_text'])
finite_values = benchmark_matrix.to_numpy(dtype=float)
vmin = max(0.45, np.nanmin(finite_values) - 0.02)
vmax = min(1.0, np.nanmax(finite_values) + 0.02)
benchmark_image = ax_benchmark.imshow(benchmark_matrix.values, cmap=benchmark_cmap, vmin=vmin, vmax=vmax, aspect='auto')
ax_benchmark.set_xticks(np.arange(len(metric_order)))
ax_benchmark.set_xticklabels(metric_display, fontsize=FIG3D1_STYLE['benchmark_column_label'])
ax_benchmark.set_yticks(np.arange(len(BENCHMARK_METHODS)))
ax_benchmark.set_yticklabels(BENCHMARK_METHODS, fontsize=FIG3D1_STYLE['benchmark_row_label'])
ax_benchmark.set_title('Simple integration benchmark', fontsize=FIG3D1_STYLE['panel_title'], pad=12)
for row_index, method in enumerate(BENCHMARK_METHODS):
    for column_index, metric in enumerate(metric_order):
        value = benchmark_matrix.loc[method, metric]
        delta = benchmark_delta_matrix.loc[method, metric]
        if not np.isfinite(value):
            continue
        ax_benchmark.text(column_index, row_index - 0.1, f'{value:.3f}', ha='center', va='center', fontsize=FIG3D1_STYLE['benchmark_value'], color=FIG3D1_COLORS['text'])
        if method in FUSION_METHODS:
            delta_text = f'Δ {delta:+.3f}'
            delta_color = FIG3D1_COLORS['text'] if delta >= 0 else FIG3D1_COLORS['muted_text']
            ax_benchmark.text(column_index, row_index + 0.22, delta_text, ha='center', va='center', fontsize=FIG3D1_STYLE['benchmark_delta'], color=delta_color)
ax_benchmark.set_xticks(np.arange(-0.5, len(metric_order), 1), minor=True)
ax_benchmark.set_yticks(np.arange(-0.5, len(BENCHMARK_METHODS), 1), minor=True)
ax_benchmark.grid(False)
ax_benchmark.tick_params(which='minor', bottom=False, left=False)
for row_index, method in enumerate(BENCHMARK_METHODS):
    edge_color = FIG3D1_COLORS['fusion_outline'] if method in FUSION_METHODS else FIG3D1_COLORS['single_outline']
    line_width = 0.9 if method in FUSION_METHODS else 0.35
    rectangle = plt.Rectangle((-0.5, row_index - 0.5), len(metric_order), 1.0, fill=False, edgecolor=edge_color, linewidth=line_width, zorder=5)
    ax_benchmark.add_patch(rectangle)
benchmark_colorbar = fig.colorbar(benchmark_image, ax=ax_benchmark, fraction=0.05, pad=0.04)
benchmark_colorbar.set_label('Mean repeated-CV performance', fontsize=FIG3D1_STYLE['benchmark_colorbar_label'])
benchmark_colorbar.ax.tick_params(labelsize=FIG3D1_STYLE['benchmark_colorbar_tick'])
best_single_note = '; '.join((f'{display}: {best_single_name[metric]}' for display, metric in zip(metric_display, metric_order)))
fig.suptitle('Cross-modal similarity and simple integration benchmark', fontsize=FIG3D1_STYLE['main_title'], y=1.01)
plt.tight_layout(rect=[0, 0.02, 1, 0.97])
figure_png = os.path.join(out_dir, get_value('science.figures.plot_078', 'plot_078.png'))
figure_pdf = os.path.join(out_dir, get_value('science.figures.plot_079', 'plot_079.pdf'))
plt.savefig(figure_png, dpi=600, bbox_inches='tight')
plt.savefig(figure_pdf, bbox_inches='tight')
plt.show()





print('  modality_matrices, labels, CV_SPLITS, prepare_latent_modality_block,')



In [ ]:
import os
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'Arial', 'font.size': 11.0, 'axes.titlesize': 14.0, 'axes.labelsize': 11.5, 'xtick.labelsize': 10.5, 'ytick.labelsize': 10.5, 'legend.fontsize': 10.0, 'figure.titlesize': 16.0, 'text.color': '#1F1F1F', 'axes.labelcolor': '#1F1F1F', 'axes.titlecolor': '#1F1F1F', 'xtick.color': '#1F1F1F', 'ytick.color': '#1F1F1F', 'axes.edgecolor': '#2B2B2B', 'axes.grid': False, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none'})
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from sklearn.preprocessing import StandardScaler
PART1_REQUIRED_OBJECTS = ['MODALITY_ORDER', 'CV_SPLITS', 'N_SPLITS', 'labels', 'prepare_latent_modality_block', 'evaluate_design', 'output_root']
missing_objects = [name for name in PART1_REQUIRED_OBJECTS if name not in globals()]
if missing_objects:
    raise RuntimeError('Please run Figure 3D Part 1 first. Missing objects: ' + ', '.join(missing_objects))
FIG3D2_STYLE = {'figure_size': (15.8, 5.8), 'main_title': 16.0, 'panel_title': 14.0, 'footnote': 9.0, 'row_label': 10.5, 'column_label': 10.5, 'axis_title': 11.5, 'delta_value': 10.0, 'positive_fraction': 9.5, 'stability_symbol': 10.0, 'stability_x_offset': 0.33, 'stability_y_offset': -0.32, 'colorbar_label': 11.0, 'colorbar_tick': 10.0, 'cell_border_width': 1.0, 'panel_spacing': 0.3}
FIG3D2_COLORS = {'negative': '#91ACB8', 'neutral': '#F5F3EF', 'positive': '#C79C8D', 'diagonal': '#ECECEA', 'text': '#1F1F1F', 'muted_text': '#6B6B6B', 'grid': '#FFFFFF', 'symbol': '#4B4B4B'}
plt.rcParams.update({'font.family': 'Arial', 'axes.unicode_minus': False, 'pdf.fonttype': 42, 'ps.fonttype': 42})
increment_cmap = LinearSegmentedColormap.from_list('soft_directional_increment', [FIG3D2_COLORS['negative'], FIG3D2_COLORS['neutral'], FIG3D2_COLORS['positive']])
part2_out_dir = os.path.join(output_root, get_value('science.paths.directory_059', 'directory_059'))
os.makedirs(part2_out_dir, exist_ok=True)
METRIC_ORDER = ['Response_AUC', 'PFS_Cindex', 'OS_Cindex']
METRIC_DISPLAY = {'Response_AUC': 'Response AUC', 'PFS_Cindex': 'PFS C-index', 'OS_Cindex': 'OS C-index'}
FIXED_DELTA_COLOR_LIMIT = None
STABILITY_POSITIVE_FOLD_THRESHOLD = 0.8

def build_pair_design(latent_cache, modality_a, modality_b):
    pass
    train_design = np.column_stack([latent_cache[modality_a][0], latent_cache[modality_b][0]])
    test_design = np.column_stack([latent_cache[modality_a][1], latent_cache[modality_b][1]])
    scaler = StandardScaler()
    train_design = scaler.fit_transform(train_design)
    test_design = scaler.transform(test_design)
    return (train_design, test_design)
incremental_records = []
for fold_number, (train_indices, test_indices) in enumerate(CV_SPLITS, start=1):
    repeat_number = (fold_number - 1) // N_SPLITS + 1
    split_number = (fold_number - 1) % N_SPLITS + 1
    train_labels = labels.iloc[train_indices].copy()
    test_labels = labels.iloc[test_indices].copy()
    
    latent_cache = {modality: prepare_latent_modality_block(modality, train_indices, test_indices) for modality in MODALITY_ORDER}
    single_metrics = {}
    for modality in MODALITY_ORDER:
        train_design, test_design = latent_cache[modality]
        single_metrics[modality] = evaluate_design(train_design, test_design, train_labels, test_labels)
    for modality_a, modality_b in combinations(MODALITY_ORDER, 2):
        train_design, test_design = build_pair_design(latent_cache, modality_a, modality_b)
        pair_metrics = evaluate_design(train_design, test_design, train_labels, test_labels)
        directions = [(modality_a, modality_b), (modality_b, modality_a)]
        for base_modality, added_modality in directions:
            for metric in METRIC_ORDER:
                base_value = single_metrics[base_modality][metric]
                pair_value = pair_metrics[metric]
                incremental_records.append({'Fold': fold_number, 'Repeat': repeat_number, 'Split': split_number, 'Base_modality': base_modality, 'Added_modality': added_modality, 'Metric': metric, 'Base_value': base_value, 'Pair_value': pair_value, 'Delta': pair_value - base_value})
incremental_fold_results = pd.DataFrame(incremental_records)
incremental_summary = incremental_fold_results.groupby(['Base_modality', 'Added_modality', 'Metric'], as_index=False).agg(Mean_delta=('Delta', 'mean'), Median_delta=('Delta', 'median'), Lower_95=('Delta', lambda values: np.nanquantile(values, 0.025)), Upper_95=('Delta', lambda values: np.nanquantile(values, 0.975)), Positive_fold_fraction=('Delta', lambda values: np.mean(np.asarray(values) > 0)), Negative_fold_fraction=('Delta', lambda values: np.mean(np.asarray(values) < 0)), SD_delta=('Delta', 'std'), Fold_N=('Delta', lambda values: np.isfinite(values).sum()))
repeat_level_results = incremental_fold_results.groupby(['Base_modality', 'Added_modality', 'Metric', 'Repeat'], as_index=False).agg(Repeat_mean_delta=('Delta', 'mean'))
repeat_summary = repeat_level_results.groupby(['Base_modality', 'Added_modality', 'Metric'], as_index=False).agg(Mean_repeat_delta=('Repeat_mean_delta', 'mean'), Positive_repeat_fraction=('Repeat_mean_delta', lambda values: np.mean(np.asarray(values) > 0)), Repeat_SD=('Repeat_mean_delta', 'std'), Repeat_N=('Repeat_mean_delta', 'size'))
incremental_summary = incremental_summary.merge(repeat_summary, on=['Base_modality', 'Added_modality', 'Metric'], how='left')
asymmetry_records = []
for modality_a, modality_b in combinations(MODALITY_ORDER, 2):
    for metric in METRIC_ORDER:
        a_added_to_b = incremental_summary.loc[(incremental_summary['Base_modality'] == modality_b) & (incremental_summary['Added_modality'] == modality_a) & (incremental_summary['Metric'] == metric), 'Mean_delta']
        b_added_to_a = incremental_summary.loc[(incremental_summary['Base_modality'] == modality_a) & (incremental_summary['Added_modality'] == modality_b) & (incremental_summary['Metric'] == metric), 'Mean_delta']
        if len(a_added_to_b) == 1 and len(b_added_to_a) == 1:
            asymmetry_records.append({'Modality_A': modality_a, 'Modality_B': modality_b, 'Metric': metric, 'Delta_A_added_to_B': float(a_added_to_b.iloc[0]), 'Delta_B_added_to_A': float(b_added_to_a.iloc[0]), 'Directional_asymmetry_A_minus_B': float(a_added_to_b.iloc[0]) - float(b_added_to_a.iloc[0])})
asymmetry_table = pd.DataFrame(asymmetry_records)
incremental_fold_results.to_csv(os.path.join(part2_out_dir, get_value('science.tables.metrics_028', 'metrics_028.csv')), index=False, encoding='utf-8-sig')
repeat_level_results.to_csv(os.path.join(part2_out_dir, get_value('science.tables.metrics_029', 'metrics_029.csv')), index=False, encoding='utf-8-sig')
incremental_summary.to_csv(os.path.join(part2_out_dir, get_value('science.tables.metrics_030', 'metrics_030.csv')), index=False, encoding='utf-8-sig')
asymmetry_table.to_csv(os.path.join(part2_out_dir, get_value('science.tables.metrics_031', 'metrics_031.csv')), index=False, encoding='utf-8-sig')

def make_directional_matrix(metric, value_column):
    matrix = pd.DataFrame(np.nan, index=MODALITY_ORDER, columns=MODALITY_ORDER, dtype=float)
    subset = incremental_summary.loc[incremental_summary['Metric'] == metric]
    for _, row in subset.iterrows():
        matrix.loc[row['Added_modality'], row['Base_modality']] = row[value_column]
    return matrix
mean_delta_matrices = {metric: make_directional_matrix(metric, 'Mean_delta') for metric in METRIC_ORDER}
positive_fraction_matrices = {metric: make_directional_matrix(metric, 'Positive_fold_fraction') for metric in METRIC_ORDER}
lower_interval_matrices = {metric: make_directional_matrix(metric, 'Lower_95') for metric in METRIC_ORDER}
all_delta_values = np.concatenate([matrix.to_numpy().ravel() for matrix in mean_delta_matrices.values()])
all_delta_values = all_delta_values[np.isfinite(all_delta_values)]
if FIXED_DELTA_COLOR_LIMIT is None:
    raw_limit = np.nanmax(np.abs(all_delta_values))
    delta_color_limit = max(0.03, np.ceil(raw_limit * 100) / 100)
else:
    delta_color_limit = float(FIXED_DELTA_COLOR_LIMIT)
increment_norm = TwoSlopeNorm(vmin=-delta_color_limit, vcenter=0, vmax=delta_color_limit)
print(f'Common directional-increment color limit: +/-{delta_color_limit:.3f}')
fig, axes = plt.subplots(1, 3, figsize=FIG3D2_STYLE['figure_size'], gridspec_kw={'wspace': FIG3D2_STYLE['panel_spacing']})
for panel_index, metric in enumerate(METRIC_ORDER):
    ax = axes[panel_index]
    delta_matrix = mean_delta_matrices[metric]
    positive_matrix = positive_fraction_matrices[metric]
    lower_matrix = lower_interval_matrices[metric]
    display_matrix = delta_matrix.to_numpy().copy()
    np.fill_diagonal(display_matrix, 0.0)
    image = ax.imshow(display_matrix, cmap=increment_cmap, norm=increment_norm, aspect='equal')
    for diagonal_index in range(len(MODALITY_ORDER)):
        ax.add_patch(plt.Rectangle((diagonal_index - 0.5, diagonal_index - 0.5), 1.0, 1.0, facecolor=FIG3D2_COLORS['diagonal'], edgecolor=FIG3D2_COLORS['grid'], linewidth=FIG3D2_STYLE['cell_border_width'], zorder=3))
        ax.text(diagonal_index, diagonal_index, '—', ha='center', va='center', fontsize=FIG3D2_STYLE['delta_value'], color=FIG3D2_COLORS['muted_text'], zorder=5)
    for row_index, added_modality in enumerate(MODALITY_ORDER):
        for column_index, base_modality in enumerate(MODALITY_ORDER):
            if row_index == column_index:
                continue
            delta = delta_matrix.loc[added_modality, base_modality]
            positive_fraction = positive_matrix.loc[added_modality, base_modality]
            lower = lower_matrix.loc[added_modality, base_modality]
            if not np.isfinite(delta):
                continue
            ax.text(column_index, row_index - 0.09, f'{delta:+.3f}', ha='center', va='center', fontsize=FIG3D2_STYLE['delta_value'], color=FIG3D2_COLORS['text'], zorder=5)
            ax.text(column_index, row_index + 0.22, f'({positive_fraction:.0%}+)', ha='center', va='center', fontsize=FIG3D2_STYLE['positive_fraction'], color=FIG3D2_COLORS['muted_text'], zorder=5)
            stable_gain = np.isfinite(lower) and lower > 0 and (positive_fraction >= STABILITY_POSITIVE_FOLD_THRESHOLD)
            if stable_gain:
                ax.text(column_index + FIG3D2_STYLE['stability_x_offset'], row_index + FIG3D2_STYLE['stability_y_offset'], '†', ha='center', va='center', fontsize=FIG3D2_STYLE['stability_symbol'], color=FIG3D2_COLORS['symbol'], zorder=6)
    ax.set_xticks(np.arange(len(MODALITY_ORDER)))
    ax.set_yticks(np.arange(len(MODALITY_ORDER)))
    ax.set_xticklabels(MODALITY_ORDER, rotation=38, ha='right', fontsize=FIG3D2_STYLE['column_label'])
    ax.set_yticklabels(MODALITY_ORDER, fontsize=FIG3D2_STYLE['row_label'])
    ax.set_xlabel('Base modality', fontsize=FIG3D2_STYLE['axis_title'])
    if panel_index == 0:
        ax.set_ylabel('Added modality', fontsize=FIG3D2_STYLE['axis_title'])
    else:
        ax.set_ylabel('')
    ax.set_title(METRIC_DISPLAY[metric], fontsize=FIG3D2_STYLE['panel_title'], pad=12)
    ax.set_xticks(np.arange(-0.5, len(MODALITY_ORDER), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(MODALITY_ORDER), 1), minor=True)
    ax.grid(False)
    ax.tick_params(which='minor', bottom=False, left=False)
colorbar = fig.colorbar(image, ax=axes, fraction=0.022, pad=0.025)
colorbar.set_label('Mean incremental performance', fontsize=FIG3D2_STYLE['colorbar_label'])
colorbar.ax.tick_params(labelsize=FIG3D2_STYLE['colorbar_tick'])
fig.suptitle('Complete directional incremental value across modality pairs', fontsize=FIG3D2_STYLE['main_title'], y=0.995)
plt.tight_layout(rect=[0.02, 0.02, 0.96, 0.94])
figure_png = os.path.join(part2_out_dir, get_value('science.figures.plot_080', 'plot_080.png'))
figure_pdf = os.path.join(part2_out_dir, get_value('science.figures.plot_081', 'plot_081.pdf'))
plt.savefig(figure_png, dpi=600, bbox_inches='tight')
plt.savefig(figure_pdf, bbox_inches='tight')
plt.show()
print('\nKO-Species directional results:')
ko_species_rows = incremental_summary.loc[(incremental_summary['Base_modality'] == 'Metagenomics') & (incremental_summary['Added_modality'] == 'Species') | (incremental_summary['Base_modality'] == 'Species') & (incremental_summary['Added_modality'] == 'Metagenomics')].copy()
print(ko_species_rows[['Base_modality', 'Added_modality', 'Metric', 'Mean_delta', 'Lower_95', 'Upper_95', 'Positive_fold_fraction', 'Positive_repeat_fraction']].to_string(index=False))






In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'Arial', 'font.size': 11.0, 'axes.titlesize': 14.0, 'axes.labelsize': 11.5, 'xtick.labelsize': 10.5, 'ytick.labelsize': 10.5, 'legend.fontsize': 10.0, 'figure.titlesize': 16.0, 'text.color': '#1F1F1F', 'axes.labelcolor': '#1F1F1F', 'axes.titlecolor': '#1F1F1F', 'xtick.color': '#1F1F1F', 'ytick.color': '#1F1F1F', 'axes.edgecolor': '#2B2B2B', 'axes.grid': False, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none'})
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from sklearn.preprocessing import StandardScaler
PART1_REQUIRED_OBJECTS = ['MODALITY_ORDER', 'CV_SPLITS', 'N_SPLITS', 'labels', 'prepare_latent_modality_block', 'build_pooled_pca_fusion', 'evaluate_design', 'fit_response_model', 'fit_cox_model', 'safe_auc', 'safe_c_index', 'output_root']
missing_objects = [name for name in PART1_REQUIRED_OBJECTS if name not in globals()]
if missing_objects:
    raise RuntimeError('Please run Figure 3D Part 1 first. Missing objects: ' + ', '.join(missing_objects))
FIG3D3_STYLE = {'figure_size': (13.8, 5.8), 'main_title': 16.0, 'panel_title': 14.0, 'footnote': 9.0, 'row_label': 10.5, 'column_label': 10.5, 'y_axis_title': 11.5, 'delta_value': 10.0, 'positive_fraction': 9.5, 'stability_symbol': 10.0, 'symbol_x_offset': 0.34, 'symbol_y_offset': -0.33, 'colorbar_label': 11.0, 'colorbar_tick': 10.0, 'cell_border_width': 1.0, 'panel_spacing': 0.32}
FIG3D3_COLORS = {'negative': '#8FAAB6', 'neutral': '#F5F3EF', 'positive': '#C5A094', 'text': '#1F1F1F', 'muted_text': '#6B6B6B', 'grid': '#FFFFFF', 'positive_symbol': '#4B4B4B', 'negative_symbol': '#536B76'}
plt.rcParams.update({'font.family': 'Arial', 'axes.unicode_minus': False, 'pdf.fonttype': 42, 'ps.fonttype': 42})
lomo_cmap = LinearSegmentedColormap.from_list('soft_lomo', [FIG3D3_COLORS['negative'], FIG3D3_COLORS['neutral'], FIG3D3_COLORS['positive']])
RUN_BLOCK_PCA_LOMO = True
RUN_LATE_FUSION_LOMO = True
RUN_POOLED_PCA_LOMO = False
STABILITY_FRACTION_THRESHOLD = 0.8
FIXED_LOMO_COLOR_LIMIT = None
part3_out_dir = os.path.join(output_root, get_value('science.paths.directory_060', 'directory_060'))
os.makedirs(part3_out_dir, exist_ok=True)
METRIC_ORDER = ['Response_AUC', 'PFS_Cindex', 'OS_Cindex']
METRIC_DISPLAY = ['Response AUC', 'PFS C-index', 'OS C-index']

def build_cached_block_fusion(latent_cache, modalities):
    pass
    train_design = np.column_stack([latent_cache[name][0] for name in modalities])
    test_design = np.column_stack([latent_cache[name][1] for name in modalities])
    scaler = StandardScaler()
    return (scaler.fit_transform(train_design), scaler.transform(test_design))

def evaluate_late_fusion_subset(latent_cache, modalities, train_labels, test_labels):
    pass
    response_predictions = []
    survival_predictions = {'PFS': [], 'OS': []}
    for modality in modalities:
        train_design, test_design = latent_cache[modality]
        _, test_probability = fit_response_model(train_design, train_labels, test_design)
        response_predictions.append(test_probability)
        for endpoint in ['PFS', 'OS']:
            train_risk, test_risk, _ = fit_cox_model(train_design, train_labels, test_design, endpoint)
            train_mean = np.mean(train_risk)
            train_sd = np.std(train_risk, ddof=0)
            if not np.isfinite(train_sd) or train_sd < 1e-08:
                train_sd = 1.0
            survival_predictions[endpoint].append((test_risk - train_mean) / train_sd)
    mean_probability = np.mean(np.column_stack(response_predictions), axis=1)
    metrics = {'Response_AUC': safe_auc(test_labels['Response'], mean_probability)}
    for endpoint in ['PFS', 'OS']:
        mean_risk = np.mean(np.column_stack(survival_predictions[endpoint]), axis=1)
        metrics[f'{endpoint}_Cindex'] = safe_c_index(test_labels[f'{endpoint}_time'], test_labels[f'{endpoint}_status'], mean_risk)
    return metrics

def evaluate_strategy(strategy, modalities, latent_cache, train_indices, test_indices, train_labels, test_labels):
    pass
    if strategy == 'Block-PCA early fusion':
        train_design, test_design = build_cached_block_fusion(latent_cache, modalities)
        return evaluate_design(train_design, test_design, train_labels, test_labels)
    if strategy == 'Equal-weight late fusion':
        return evaluate_late_fusion_subset(latent_cache, modalities, train_labels, test_labels)
    if strategy == 'Pooled PCA fusion':
        train_design, test_design = build_pooled_pca_fusion(train_indices, test_indices, modalities=modalities)
        return evaluate_design(train_design, test_design, train_labels, test_labels)
    raise ValueError(f'Unknown LOMO strategy: {strategy}')
LOMO_STRATEGIES = []
if RUN_BLOCK_PCA_LOMO:
    LOMO_STRATEGIES.append('Block-PCA early fusion')
if RUN_LATE_FUSION_LOMO:
    LOMO_STRATEGIES.append('Equal-weight late fusion')
if RUN_POOLED_PCA_LOMO:
    LOMO_STRATEGIES.append('Pooled PCA fusion')
if not LOMO_STRATEGIES:
    raise ValueError('At least one LOMO strategy must be enabled.')
lomo_records = []
for fold_number, (train_indices, test_indices) in enumerate(CV_SPLITS, start=1):
    repeat_number = (fold_number - 1) // N_SPLITS + 1
    split_number = (fold_number - 1) % N_SPLITS + 1
    train_labels = labels.iloc[train_indices].copy()
    test_labels = labels.iloc[test_indices].copy()
    
    latent_cache = {modality: prepare_latent_modality_block(modality, train_indices, test_indices) for modality in MODALITY_ORDER}
    for strategy in LOMO_STRATEGIES:
        full_metrics = evaluate_strategy(strategy, MODALITY_ORDER, latent_cache, train_indices, test_indices, train_labels, test_labels)
        for removed_modality in MODALITY_ORDER:
            reduced_modalities = [modality for modality in MODALITY_ORDER if modality != removed_modality]
            reduced_metrics = evaluate_strategy(strategy, reduced_modalities, latent_cache, train_indices, test_indices, train_labels, test_labels)
            for metric in METRIC_ORDER:
                full_value = full_metrics[metric]
                reduced_value = reduced_metrics[metric]
                lomo_records.append({'Fold': fold_number, 'Repeat': repeat_number, 'Split': split_number, 'Strategy': strategy, 'Removed_modality': removed_modality, 'Metric': metric, 'Full_value': full_value, 'Without_value': reduced_value, 'Delta_loss': full_value - reduced_value})
lomo_fold_results = pd.DataFrame(lomo_records)
lomo_summary = lomo_fold_results.groupby(['Strategy', 'Removed_modality', 'Metric'], as_index=False).agg(Mean_loss=('Delta_loss', 'mean'), Median_loss=('Delta_loss', 'median'), Lower_95=('Delta_loss', lambda values: np.nanquantile(values, 0.025)), Upper_95=('Delta_loss', lambda values: np.nanquantile(values, 0.975)), Positive_fold_fraction=('Delta_loss', lambda values: np.mean(np.asarray(values) > 0)), Negative_fold_fraction=('Delta_loss', lambda values: np.mean(np.asarray(values) < 0)), SD_loss=('Delta_loss', 'std'), Fold_N=('Delta_loss', lambda values: np.isfinite(values).sum()), Mean_full_performance=('Full_value', 'mean'), Mean_without_performance=('Without_value', 'mean'))
lomo_repeat_results = lomo_fold_results.groupby(['Strategy', 'Removed_modality', 'Metric', 'Repeat'], as_index=False).agg(Repeat_mean_loss=('Delta_loss', 'mean'))
lomo_repeat_summary = lomo_repeat_results.groupby(['Strategy', 'Removed_modality', 'Metric'], as_index=False).agg(Mean_repeat_loss=('Repeat_mean_loss', 'mean'), Positive_repeat_fraction=('Repeat_mean_loss', lambda values: np.mean(np.asarray(values) > 0)), Negative_repeat_fraction=('Repeat_mean_loss', lambda values: np.mean(np.asarray(values) < 0)), Repeat_SD=('Repeat_mean_loss', 'std'), Repeat_N=('Repeat_mean_loss', 'size'))
lomo_summary = lomo_summary.merge(lomo_repeat_summary, on=['Strategy', 'Removed_modality', 'Metric'], how='left')
robustness_records = []
for modality in MODALITY_ORDER:
    subset = lomo_summary.loc[lomo_summary['Removed_modality'] == modality].copy()
    stable_positive = (subset['Lower_95'] > 0) & (subset['Positive_fold_fraction'] >= STABILITY_FRACTION_THRESHOLD)
    stable_negative = (subset['Upper_95'] < 0) & (subset['Negative_fold_fraction'] >= STABILITY_FRACTION_THRESHOLD)
    robustness_records.append({'Modality': modality, 'Strategy_endpoint_tests': len(subset), 'Positive_mean_loss_count': int((subset['Mean_loss'] > 0).sum()), 'Nonpositive_mean_loss_count': int((subset['Mean_loss'] <= 0).sum()), 'Stable_positive_contribution_count': int(stable_positive.sum()), 'Stable_removal_benefit_count': int(stable_negative.sum()), 'Mean_loss_across_strategy_endpoints': subset['Mean_loss'].mean(), 'Median_loss_across_strategy_endpoints': subset['Mean_loss'].median()})
lomo_robustness_summary = pd.DataFrame(robustness_records)
lomo_fold_results.to_csv(os.path.join(part3_out_dir, get_value('science.tables.metrics_032', 'metrics_032.csv')), index=False, encoding='utf-8-sig')
lomo_repeat_results.to_csv(os.path.join(part3_out_dir, get_value('science.tables.metrics_033', 'metrics_033.csv')), index=False, encoding='utf-8-sig')
lomo_summary.to_csv(os.path.join(part3_out_dir, get_value('science.tables.metrics_034', 'metrics_034.csv')), index=False, encoding='utf-8-sig')
lomo_robustness_summary.to_csv(os.path.join(part3_out_dir, get_value('science.tables.metrics_035', 'metrics_035.csv')), index=False, encoding='utf-8-sig')

def make_lomo_matrix(strategy, value_column):
    matrix = pd.DataFrame(np.nan, index=MODALITY_ORDER, columns=METRIC_ORDER, dtype=float)
    subset = lomo_summary.loc[lomo_summary['Strategy'] == strategy]
    for _, row in subset.iterrows():
        matrix.loc[row['Removed_modality'], row['Metric']] = row[value_column]
    return matrix
mean_loss_matrices = {strategy: make_lomo_matrix(strategy, 'Mean_loss') for strategy in LOMO_STRATEGIES}
positive_fraction_matrices = {strategy: make_lomo_matrix(strategy, 'Positive_fold_fraction') for strategy in LOMO_STRATEGIES}
negative_fraction_matrices = {strategy: make_lomo_matrix(strategy, 'Negative_fold_fraction') for strategy in LOMO_STRATEGIES}
lower_interval_matrices = {strategy: make_lomo_matrix(strategy, 'Lower_95') for strategy in LOMO_STRATEGIES}
upper_interval_matrices = {strategy: make_lomo_matrix(strategy, 'Upper_95') for strategy in LOMO_STRATEGIES}
all_loss_values = np.concatenate([matrix.to_numpy().ravel() for matrix in mean_loss_matrices.values()])
all_loss_values = all_loss_values[np.isfinite(all_loss_values)]
if FIXED_LOMO_COLOR_LIMIT is None:
    raw_limit = np.nanmax(np.abs(all_loss_values))
    lomo_color_limit = max(0.02, np.ceil(raw_limit * 100) / 100)
else:
    lomo_color_limit = float(FIXED_LOMO_COLOR_LIMIT)
lomo_norm = TwoSlopeNorm(vmin=-lomo_color_limit, vcenter=0, vmax=lomo_color_limit)
print(f'Common LOMO color limit: +/-{lomo_color_limit:.3f}')
panel_count = len(LOMO_STRATEGIES)
figure_width = FIG3D3_STYLE['figure_size'][0]
if panel_count == 3:
    figure_width = max(17.0, figure_width)
fig, axes = plt.subplots(1, panel_count, figsize=(figure_width, FIG3D3_STYLE['figure_size'][1]), squeeze=False, gridspec_kw={'wspace': FIG3D3_STYLE['panel_spacing']})
axes = axes.ravel()
for panel_index, strategy in enumerate(LOMO_STRATEGIES):
    ax = axes[panel_index]
    loss_matrix = mean_loss_matrices[strategy]
    positive_matrix = positive_fraction_matrices[strategy]
    negative_matrix = negative_fraction_matrices[strategy]
    lower_matrix = lower_interval_matrices[strategy]
    upper_matrix = upper_interval_matrices[strategy]
    image = ax.imshow(loss_matrix.to_numpy(), cmap=lomo_cmap, norm=lomo_norm, aspect='auto')
    for row_index, modality in enumerate(MODALITY_ORDER):
        for column_index, metric in enumerate(METRIC_ORDER):
            mean_loss = loss_matrix.loc[modality, metric]
            positive_fraction = positive_matrix.loc[modality, metric]
            negative_fraction = negative_matrix.loc[modality, metric]
            lower = lower_matrix.loc[modality, metric]
            upper = upper_matrix.loc[modality, metric]
            if not np.isfinite(mean_loss):
                continue
            ax.text(column_index, row_index - 0.09, f'{mean_loss:+.3f}', ha='center', va='center', fontsize=FIG3D3_STYLE['delta_value'], color=FIG3D3_COLORS['text'], zorder=5)
            ax.text(column_index, row_index + 0.22, f'({positive_fraction:.0%}+)', ha='center', va='center', fontsize=FIG3D3_STYLE['positive_fraction'], color=FIG3D3_COLORS['muted_text'], zorder=5)
            stable_contribution = np.isfinite(lower) and lower > 0 and (positive_fraction >= STABILITY_FRACTION_THRESHOLD)
            stable_removal_benefit = np.isfinite(upper) and upper < 0 and (negative_fraction >= STABILITY_FRACTION_THRESHOLD)
            symbol = '†' if stable_contribution else '‡' if stable_removal_benefit else ''
            symbol_color = FIG3D3_COLORS['positive_symbol'] if stable_contribution else FIG3D3_COLORS['negative_symbol']
            if symbol:
                ax.text(column_index + FIG3D3_STYLE['symbol_x_offset'], row_index + FIG3D3_STYLE['symbol_y_offset'], symbol, ha='center', va='center', fontsize=FIG3D3_STYLE['stability_symbol'], color=symbol_color, zorder=6)
    ax.set_xticks(np.arange(len(METRIC_ORDER)))
    ax.set_yticks(np.arange(len(MODALITY_ORDER)))
    ax.set_xticklabels(METRIC_DISPLAY, fontsize=FIG3D3_STYLE['column_label'])
    ax.set_yticklabels(MODALITY_ORDER, fontsize=FIG3D3_STYLE['row_label'])
    if panel_index == 0:
        ax.set_ylabel('Removed modality', fontsize=FIG3D3_STYLE['y_axis_title'])
    else:
        ax.set_ylabel('')
    ax.set_title(strategy, fontsize=FIG3D3_STYLE['panel_title'], pad=12)
    ax.set_xticks(np.arange(-0.5, len(METRIC_ORDER), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(MODALITY_ORDER), 1), minor=True)
    ax.grid(False)
    ax.tick_params(which='minor', bottom=False, left=False)
colorbar = fig.colorbar(image, ax=axes, fraction=0.025, pad=0.025)
colorbar.set_label('Mean performance loss after removal', fontsize=FIG3D3_STYLE['colorbar_label'])
colorbar.ax.tick_params(labelsize=FIG3D3_STYLE['colorbar_tick'])
fig.suptitle('Leave-one-modality-out contribution across simple integration strategies', fontsize=FIG3D3_STYLE['main_title'], y=0.995)
plt.tight_layout(rect=[0.02, 0.02, 0.96, 0.94])
figure_png = os.path.join(part3_out_dir, get_value('science.figures.plot_082', 'plot_082.png'))
figure_pdf = os.path.join(part3_out_dir, get_value('science.figures.plot_083', 'plot_083.pdf'))
plt.savefig(figure_png, dpi=600, bbox_inches='tight')
plt.savefig(figure_pdf, bbox_inches='tight')
plt.show()
species_lomo = lomo_summary.loc[lomo_summary['Removed_modality'] == 'Species', ['Strategy', 'Metric', 'Mean_loss', 'Lower_95', 'Upper_95', 'Positive_fold_fraction', 'Positive_repeat_fraction', 'Mean_full_performance', 'Mean_without_performance']].copy()
print('\nSpecies leave-one-modality-out results:')
print(species_lomo.to_string(index=False))
species_robustness = lomo_robustness_summary.loc[lomo_robustness_summary['Modality'] == 'Species']
print('\nSpecies strategy-robustness summary:')
print(species_robustness.to_string(index=False))
if 'incremental_summary' in globals():
    print('\nPart 2 is available. KO-Species directional additions:')
    directional_species = incremental_summary.loc[(incremental_summary['Base_modality'] == 'Metagenomics') & (incremental_summary['Added_modality'] == 'Species') | (incremental_summary['Base_modality'] == 'Species') & (incremental_summary['Added_modality'] == 'Metagenomics'), ['Base_modality', 'Added_modality', 'Metric', 'Mean_delta', 'Lower_95', 'Upper_95', 'Positive_fold_fraction', 'Positive_repeat_fraction']].copy()
    print(directional_species.to_string(index=False))






In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, FancyArrowPatch
plt.rcParams.update({'font.family': 'Arial', 'font.size': 12.0, 'axes.titlesize': 14.0, 'axes.labelsize': 12.5, 'xtick.labelsize': 11.2, 'ytick.labelsize': 11.2, 'legend.fontsize': 11.0, 'figure.titlesize': 14.0, 'text.color': '#1F1F1F', 'axes.labelcolor': '#1F1F1F', 'axes.titlecolor': '#1F1F1F', 'xtick.color': '#1F1F1F', 'ytick.color': '#1F1F1F', 'axes.edgecolor': '#2B2B2B', 'axes.grid': False, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none', 'axes.unicode_minus': False})
MAIN_D_REQUIRED_OBJECTS = ['incremental_summary', 'repeat_level_results', 'MODALITY_ORDER', 'output_root']
main_d_missing = [name for name in MAIN_D_REQUIRED_OBJECTS if name not in globals()]
if main_d_missing:
    raise RuntimeError('Run Figure 3D Parts 1 and 2 first. Missing objects: ' + ', '.join(main_d_missing))
main_d_out_dir = os.path.join(output_root, get_value('science.paths.directory_061', 'directory_061'))
os.makedirs(main_d_out_dir, exist_ok=True)
MAIN_MODALITY_ORDER = ['Metagenomics', 'Species', 'Radiomics', 'Blood', 'Urine']
MAIN_METRICS = ['Response_AUC', 'PFS_Cindex', 'OS_Cindex']
MAIN_METRIC_SHORT = {'Response_AUC': 'Response', 'PFS_Cindex': 'PFS', 'OS_Cindex': 'OS'}
MAIN_METRIC_FULL = {'Response_AUC': 'Response AUC', 'PFS_Cindex': 'PFS C-index', 'OS_Cindex': 'OS C-index'}
MAIN_METRIC_COLORS = {'Response_AUC': '#356D9A', 'PFS_Cindex': '#B97824', 'OS_Cindex': '#73536D'}
modality_repeat_contribution = repeat_level_results.groupby(['Repeat', 'Added_modality', 'Metric'], as_index=False).agg(Repeat_mean_increment=('Repeat_mean_delta', 'mean'))
modality_contribution_summary = modality_repeat_contribution.groupby(['Added_modality', 'Metric'], as_index=False).agg(Mean_increment=('Repeat_mean_increment', 'mean'), Median_increment=('Repeat_mean_increment', 'median'), Repeat_q025=('Repeat_mean_increment', lambda x: np.nanquantile(np.asarray(x, dtype=float), 0.025)), Repeat_q975=('Repeat_mean_increment', lambda x: np.nanquantile(np.asarray(x, dtype=float), 0.975)), Positive_repeat_fraction=('Repeat_mean_increment', lambda x: np.mean(np.asarray(x, dtype=float) > 0)), Repeat_N=('Repeat_mean_increment', 'size'))
modality_repeat_contribution.to_csv(os.path.join(main_d_out_dir, get_value('science.tables.metrics_036', 'metrics_036.csv')), index=False, encoding='utf-8-sig')
modality_contribution_summary.to_csv(os.path.join(main_d_out_dir, get_value('science.tables.metrics_037', 'metrics_037.csv')), index=False, encoding='utf-8-sig')
NETWORK_MIN_ABS_DELTA = 0.02
NETWORK_SHOW_EDGE_VALUES = True
NETWORK_POSITIVE_FRACTION_FOR_FULL_ALPHA = 0.8
NETWORK_STYLE = {'node_radius': 0.18, 'node_label': 10.8, 'node_edge_width': 1.15, 'arrow_min_width': 1.0, 'arrow_max_width': 4.8, 'arrow_head_size': 13.0, 'arrow_curvature': 0.18, 'edge_value': 9.0, 'edge_label_box_alpha': 0.94}
NETWORK_COLORS = {'node_fill': '#EFEAE3', 'node_edge': '#3F4548', 'positive': '#A94F3D', 'negative': '#3E7189', 'text': '#181818'}
node_angles = np.deg2rad([90, 18, -54, -126, 162])
NETWORK_POSITIONS = {modality: (float(np.cos(angle)), float(np.sin(angle))) for modality, angle in zip(MAIN_MODALITY_ORDER, node_angles)}

def _short_modality_name(name):
    return 'KO' if name == 'Metagenomics' else name

def _edge_alpha(delta, positive_fraction):
    directional_fraction = positive_fraction if delta >= 0 else 1.0 - positive_fraction
    directional_fraction = np.clip(directional_fraction, 0.5, 1.0)
    full_alpha_fraction = max(NETWORK_POSITIVE_FRACTION_FOR_FULL_ALPHA, 0.51)
    scaled = (directional_fraction - 0.5) / (full_alpha_fraction - 0.5)
    return float(np.clip(0.68 + 0.3 * scaled, 0.68, 0.98))

def _draw_directional_network(ax, metric, global_max_abs_delta):
    subset = incremental_summary.loc[incremental_summary['Metric'] == metric].copy()
    subset = subset.loc[np.abs(subset['Mean_delta']) >= NETWORK_MIN_ABS_DELTA]
    for modality in MAIN_MODALITY_ORDER:
        x, y = NETWORK_POSITIONS[modality]
        circle = Circle((x, y), NETWORK_STYLE['node_radius'], facecolor=NETWORK_COLORS['node_fill'], edgecolor=NETWORK_COLORS['node_edge'], linewidth=NETWORK_STYLE['node_edge_width'], zorder=6)
        ax.add_patch(circle)
        ax.text(x, y, _short_modality_name(modality), ha='center', va='center', fontsize=NETWORK_STYLE['node_label'], color=NETWORK_COLORS['text'], zorder=7)
    modality_index = {name: index for index, name in enumerate(MAIN_MODALITY_ORDER)}
    for _, row in subset.iterrows():
        source = row['Added_modality']
        target = row['Base_modality']
        if source not in NETWORK_POSITIONS or target not in NETWORK_POSITIONS:
            continue
        delta = float(row['Mean_delta'])
        positive_fraction = float(row['Positive_fold_fraction'])
        source_xy = NETWORK_POSITIONS[source]
        target_xy = NETWORK_POSITIONS[target]
        source_index = modality_index[source]
        target_index = modality_index[target]
        curvature = NETWORK_STYLE['arrow_curvature']
        curvature = curvature if source_index < target_index else -curvature
        relative_size = abs(delta) / max(global_max_abs_delta, 1e-12)
        line_width = NETWORK_STYLE['arrow_min_width'] + relative_size * (NETWORK_STYLE['arrow_max_width'] - NETWORK_STYLE['arrow_min_width'])
        edge_color = NETWORK_COLORS['positive'] if delta >= 0 else NETWORK_COLORS['negative']
        edge_alpha = _edge_alpha(delta, positive_fraction)
        arrow = FancyArrowPatch(source_xy, target_xy, arrowstyle='-|>', mutation_scale=NETWORK_STYLE['arrow_head_size'], connectionstyle=f'arc3,rad={curvature}', linewidth=line_width, color=edge_color, alpha=edge_alpha, shrinkA=20, shrinkB=20, zorder=3)
        ax.add_patch(arrow)
        if NETWORK_SHOW_EDGE_VALUES:
            source_array = np.asarray(source_xy, dtype=float)
            target_array = np.asarray(target_xy, dtype=float)
            midpoint = 0.5 * (source_array + target_array)
            direction = target_array - source_array
            normal = np.array([-direction[1], direction[0]], dtype=float)
            normal /= max(np.linalg.norm(normal), 1e-12)
            label_position = midpoint + normal * curvature * 0.62
            ax.text(label_position[0], label_position[1], f'{delta:+.3f}', ha='center', va='center', fontsize=NETWORK_STYLE['edge_value'], color=NETWORK_COLORS['text'], zorder=8, bbox={'boxstyle': 'round,pad=0.10', 'facecolor': 'white', 'edgecolor': 'none', 'alpha': NETWORK_STYLE['edge_label_box_alpha']})
    ax.set_title(MAIN_METRIC_SHORT[metric], fontsize=14.0, pad=6, fontweight='bold')
    ax.set_xlim(-1.32, 1.32)
    ax.set_ylim(-1.18, 1.28)
    ax.set_aspect('equal', adjustable='box')
    ax.set_anchor('C')
    ax.axis('off')
network_all_deltas = incremental_summary.loc[incremental_summary['Metric'].isin(MAIN_METRICS), 'Mean_delta'].to_numpy(dtype=float)
network_global_max = np.nanmax(np.abs(network_all_deltas))
fig = plt.figure(figsize=(18.2, 4.65))
gs = fig.add_gridspec(nrows=1, ncols=4, width_ratios=[1.18, 1.0, 1.0, 1.0], wspace=0.08)
ax_bar = fig.add_subplot(gs[0, 0])
bar_y = np.arange(len(MAIN_MODALITY_ORDER), dtype=float)
bar_height = 0.21
bar_offsets = {'Response_AUC': -bar_height, 'PFS_Cindex': 0.0, 'OS_Cindex': bar_height}
for metric in MAIN_METRICS:
    sub = modality_contribution_summary.loc[modality_contribution_summary['Metric'] == metric].set_index('Added_modality').reindex(MAIN_MODALITY_ORDER)
    values = sub['Mean_increment'].to_numpy(dtype=float)
    ax_bar.barh(bar_y + bar_offsets[metric], values, height=bar_height * 0.88, color=MAIN_METRIC_COLORS[metric], edgecolor='none', alpha=0.98, label=MAIN_METRIC_SHORT[metric], zorder=3)
ax_bar.axvline(0, color='#555555', linewidth=1.0, zorder=2)
ax_bar.set_yticks(bar_y)
ax_bar.set_yticklabels([_short_modality_name(m) for m in MAIN_MODALITY_ORDER], fontsize=12.0)
ax_bar.invert_yaxis()
ax_bar.set_xlabel('Mean Δ performance', fontsize=12.5)
ax_bar.grid(False)
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)
ax_bar.spines['left'].set_visible(False)
ax_bar.tick_params(axis='y', length=0)
ax_bar.legend(loc='lower center', bbox_to_anchor=(0.5, 1.005), ncol=3, frameon=False, fontsize=10.8, handlelength=1.2, columnspacing=1.0)
network_axes = [fig.add_subplot(gs[0, column]) for column in [1, 2, 3]]
for ax_network, metric in zip(network_axes, MAIN_METRICS):
    _draw_directional_network(ax_network, metric, network_global_max)
edge_legend_handles = [Line2D([0], [0], color=NETWORK_COLORS['positive'], lw=3.0, label='Positive Δ'), Line2D([0], [0], color=NETWORK_COLORS['negative'], lw=3.0, label='Negative Δ')]
fig.legend(handles=edge_legend_handles, loc='upper center', bbox_to_anchor=(0.74, 0.985), ncol=2, frameon=False, fontsize=10.8, handlelength=2.0)
ax_bar.set_box_aspect(0.79)
ax_bar.set_anchor('C')
plt.tight_layout(rect=[0.005, 0.02, 0.995, 0.91])
main_d_png = os.path.join(main_d_out_dir, get_value('science.figures.plot_084', 'plot_084.png'))
main_d_pdf = os.path.join(main_d_out_dir, get_value('science.figures.plot_085', 'plot_085.pdf'))
plt.savefig(main_d_png, dpi=600, bbox_inches='tight')
plt.savefig(main_d_pdf, bbox_inches='tight')
plt.show()



print('\nModality-level contribution summary:')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
FOREST_REQUIRED_OBJECTS = ['lomo_repeat_results', 'LOMO_STRATEGIES', 'MODALITY_ORDER', 'output_root']
forest_missing = [name for name in FOREST_REQUIRED_OBJECTS if name not in globals()]
if forest_missing:
    raise RuntimeError('Run Figure 3D Parts 1 and 3 first. Missing objects: ' + ', '.join(forest_missing))
FIG3D_FOREST_STYLE = {'figure_size': (15.8, 4.15), 'main_title': 14.0, 'panel_title': 13.5, 'footnote': 9.0, 'row_label': 11.5, 'axis_label': 12.0, 'axis_tick': 10.8, 'marker_size': 6.5, 'interval_width': 1.5, 'cap_size': 3.0, 'strategy_offset': 0.115, 'value_label': 10.0, 'value_offset_fraction': 0.014, 'x_padding_fraction': 0.24, 'panel_spacing': 0.24}
FIG3D_FOREST_COLORS = {'Block-PCA early fusion': '#7A4E40', 'Equal-weight late fusion': '#2F6782', 'Pooled PCA fusion': '#5F5F5F', 'zero': '#666666', 'text': '#181818', 'muted_text': '#2F2F2F', 'grid': '#DDDDDD'}
FOREST_MARKERS = {'Block-PCA early fusion': 'o', 'Equal-weight late fusion': '^', 'Pooled PCA fusion': 's'}
FOREST_METRICS = ['Response_AUC', 'PFS_Cindex', 'OS_Cindex']
FOREST_METRIC_LABELS = {'Response_AUC': 'Response AUC', 'PFS_Cindex': 'PFS C-index', 'OS_Cindex': 'OS C-index'}
forest_out_dir = os.path.join(output_root, get_value('science.paths.directory_062', 'directory_062'))
os.makedirs(forest_out_dir, exist_ok=True)
lomo_forest_summary = lomo_repeat_results.groupby(['Strategy', 'Removed_modality', 'Metric'], as_index=False).agg(Mean_repeat_loss=('Repeat_mean_loss', 'mean'), Min_repeat_loss=('Repeat_mean_loss', 'min'), Max_repeat_loss=('Repeat_mean_loss', 'max'), Positive_repeat_fraction=('Repeat_mean_loss', lambda x: np.mean(np.asarray(x) > 0)), Repeat_N=('Repeat_mean_loss', 'size'))
lomo_forest_summary.to_csv(os.path.join(forest_out_dir, get_value('science.tables.metrics_038', 'metrics_038.csv')), index=False, encoding='utf-8-sig')
forest_values = lomo_forest_summary[['Min_repeat_loss', 'Max_repeat_loss']].to_numpy(dtype=float)
forest_limit = np.nanmax(np.abs(forest_values))
forest_limit = max(0.025, np.ceil(forest_limit * 100) / 100)
forest_plot_limit = forest_limit * (1.0 + FIG3D_FOREST_STYLE['x_padding_fraction'])
fig, axes = plt.subplots(1, 3, figsize=FIG3D_FOREST_STYLE['figure_size'], sharey=True, gridspec_kw={'wspace': FIG3D_FOREST_STYLE['panel_spacing']})
base_y = np.arange(len(MODALITY_ORDER), dtype=float)
strategy_count = len(LOMO_STRATEGIES)
strategy_offsets = np.linspace(-FIG3D_FOREST_STYLE['strategy_offset'], FIG3D_FOREST_STYLE['strategy_offset'], strategy_count)
for panel_index, metric in enumerate(FOREST_METRICS):
    ax = axes[panel_index]
    ax.axvline(0, color=FIG3D_FOREST_COLORS['zero'], lw=1.0, ls='--', zorder=1)
    for strategy_index, strategy in enumerate(LOMO_STRATEGIES):
        subset = lomo_forest_summary.loc[(lomo_forest_summary['Strategy'] == strategy) & (lomo_forest_summary['Metric'] == metric)].set_index('Removed_modality').reindex(MODALITY_ORDER)
        mean_values = subset['Mean_repeat_loss'].to_numpy(dtype=float)
        lower_values = subset['Min_repeat_loss'].to_numpy(dtype=float)
        upper_values = subset['Max_repeat_loss'].to_numpy(dtype=float)
        y_values = base_y + strategy_offsets[strategy_index]
        xerr = np.vstack([mean_values - lower_values, upper_values - mean_values])
        ax.errorbar(mean_values, y_values, xerr=xerr, fmt=FOREST_MARKERS.get(strategy, 'o'), markersize=FIG3D_FOREST_STYLE['marker_size'], color=FIG3D_FOREST_COLORS.get(strategy, '#777777'), ecolor=FIG3D_FOREST_COLORS.get(strategy, '#777777'), elinewidth=FIG3D_FOREST_STYLE['interval_width'], capsize=FIG3D_FOREST_STYLE['cap_size'], markerfacecolor=FIG3D_FOREST_COLORS.get(strategy, '#777777'), markeredgecolor=FIG3D_FOREST_COLORS.get(strategy, '#777777'), markeredgewidth=1.0, alpha=0.98, zorder=4)
        label_offset = FIG3D_FOREST_STYLE['value_offset_fraction'] * (2 * forest_limit)
        for x_value, lower_value, upper_value, y_value in zip(mean_values, lower_values, upper_values, y_values):
            if not (np.isfinite(x_value) and np.isfinite(lower_value) and np.isfinite(upper_value)):
                continue
            if x_value >= 0:
                horizontal_alignment = 'left'
                text_x = upper_value + label_offset
            else:
                horizontal_alignment = 'right'
                text_x = lower_value - label_offset
            ax.text(text_x, y_value, f'{x_value:+.3f}', ha=horizontal_alignment, va='center', fontsize=FIG3D_FOREST_STYLE['value_label'], color=FIG3D_FOREST_COLORS['text'], clip_on=False, zorder=6)
    ax.set_xlim(-forest_plot_limit, forest_plot_limit)
    ax.set_ylim(-0.55, len(MODALITY_ORDER) - 0.45)
    ax.invert_yaxis()
    ax.set_yticks(base_y)
    ax.set_yticklabels([_short_modality_name(m) for m in MODALITY_ORDER], fontsize=FIG3D_FOREST_STYLE['row_label'])
    ax.set_xlabel('Performance loss after removal', fontsize=FIG3D_FOREST_STYLE['axis_label'])
    ax.set_title(FOREST_METRIC_LABELS[metric], fontsize=FIG3D_FOREST_STYLE['panel_title'], pad=10)
    ax.tick_params(axis='x', labelsize=FIG3D_FOREST_STYLE['axis_tick'])
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
legend_handles = [Line2D([0], [0], marker=FOREST_MARKERS.get(strategy, 'o'), color='none', markerfacecolor=FIG3D_FOREST_COLORS.get(strategy, '#777777'), markeredgecolor=FIG3D_FOREST_COLORS.get(strategy, '#777777'), markeredgewidth=1.2, markersize=6.5, label=strategy) for strategy in LOMO_STRATEGIES]
fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.985), ncol=len(LOMO_STRATEGIES), frameon=False, fontsize=10.8)
plt.tight_layout(rect=[0.02, 0.02, 0.99, 0.88])
forest_png = os.path.join(forest_out_dir, get_value('science.figures.plot_086', 'plot_086.png'))
forest_pdf = os.path.join(forest_out_dir, get_value('science.figures.plot_087', 'plot_087.pdf'))
plt.savefig(forest_png, dpi=600, bbox_inches='tight')
plt.savefig(forest_pdf, bbox_inches='tight')
plt.show()



import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BENCHMARK_REQUIRED_OBJECTS = ['benchmark_fold_results', 'BENCHMARK_METHODS', 'N_SPLITS', 'output_root']
benchmark_missing = [name for name in BENCHMARK_REQUIRED_OBJECTS if name not in globals()]
if benchmark_missing:
    raise RuntimeError('Run Figure 3D Part 1 first. Missing objects: ' + ', '.join(benchmark_missing))
FIG3D_BENCHMARK_DIST_STYLE = {'figure_size': (17.2, 6.0), 'main_title': 16.0, 'panel_title': 14.0, 'footnote': 9.0, 'method_label': 10.0, 'axis_label': 11.5, 'axis_tick': 10.5, 'box_width': 0.58, 'box_line_width': 0.9, 'point_size': 24.0, 'point_alpha': 0.82, 'jitter_width': 0.075, 'panel_spacing': 0.28}
FIG3D_BENCHMARK_DIST_COLORS = {'single_fill': '#B8CBD4', 'fusion_fill': '#D4B1A4', 'edge': '#5A5A5A', 'point': '#3F4A4E', 'separator': '#8A8A8A', 'text': '#1F1F1F', 'muted_text': '#333333', 'grid': '#E2E2E2'}
BENCHMARK_DIST_METRICS = ['Response_AUC', 'PFS_Cindex', 'OS_Cindex']
BENCHMARK_DIST_LABELS = {'Response_AUC': 'Response AUC', 'PFS_Cindex': 'PFS C-index', 'OS_Cindex': 'OS C-index'}
benchmark_dist_out_dir = os.path.join(output_root, get_value('science.paths.directory_063', 'directory_063'))
os.makedirs(benchmark_dist_out_dir, exist_ok=True)
benchmark_repeat_results = benchmark_fold_results.copy()
benchmark_repeat_results['Repeat'] = (benchmark_repeat_results['Fold'].astype(int) - 1) // int(N_SPLITS) + 1
benchmark_repeat_results = benchmark_repeat_results.groupby(['Repeat', 'Method', 'Metric'], as_index=False).agg(Repeat_mean_performance=('Value', 'mean'))
benchmark_repeat_results.to_csv(os.path.join(benchmark_dist_out_dir, get_value('science.tables.metrics_039', 'metrics_039.csv')), index=False, encoding='utf-8-sig')
rng = np.random.default_rng(20260801)
fig, axes = plt.subplots(1, 3, figsize=FIG3D_BENCHMARK_DIST_STYLE['figure_size'], gridspec_kw={'wspace': FIG3D_BENCHMARK_DIST_STYLE['panel_spacing']})
for panel_index, metric in enumerate(BENCHMARK_DIST_METRICS):
    ax = axes[panel_index]
    metric_data = benchmark_repeat_results.loc[benchmark_repeat_results['Metric'] == metric].copy()
    method_values = []
    for method in BENCHMARK_METHODS:
        values = metric_data.loc[metric_data['Method'] == method, 'Repeat_mean_performance'].dropna().to_numpy(dtype=float)
        method_values.append(values)
    box = ax.boxplot(method_values, positions=np.arange(len(BENCHMARK_METHODS)), widths=FIG3D_BENCHMARK_DIST_STYLE['box_width'], patch_artist=True, showfliers=False, medianprops={'color': FIG3D_BENCHMARK_DIST_COLORS['edge'], 'lw': 1.2}, whiskerprops={'color': FIG3D_BENCHMARK_DIST_COLORS['edge'], 'lw': FIG3D_BENCHMARK_DIST_STYLE['box_line_width']}, capprops={'color': FIG3D_BENCHMARK_DIST_COLORS['edge'], 'lw': FIG3D_BENCHMARK_DIST_STYLE['box_line_width']}, boxprops={'color': FIG3D_BENCHMARK_DIST_COLORS['edge'], 'lw': FIG3D_BENCHMARK_DIST_STYLE['box_line_width']})
    for method_index, patch in enumerate(box['boxes']):
        is_fusion = method_index >= 5
        patch.set_facecolor(FIG3D_BENCHMARK_DIST_COLORS['fusion_fill'] if is_fusion else FIG3D_BENCHMARK_DIST_COLORS['single_fill'])
        patch.set_alpha(0.72)
    for method_index, values in enumerate(method_values):
        jitter = rng.uniform(-FIG3D_BENCHMARK_DIST_STYLE['jitter_width'], FIG3D_BENCHMARK_DIST_STYLE['jitter_width'], size=len(values))
        ax.scatter(np.full(len(values), method_index) + jitter, values, s=FIG3D_BENCHMARK_DIST_STYLE['point_size'], color=FIG3D_BENCHMARK_DIST_COLORS['point'], alpha=FIG3D_BENCHMARK_DIST_STYLE['point_alpha'], zorder=4)
    ax.axvline(4.5, color=FIG3D_BENCHMARK_DIST_COLORS['separator'], lw=0.9, ls='--')
    ax.set_xticks(np.arange(len(BENCHMARK_METHODS)))
    ax.set_xticklabels(BENCHMARK_METHODS, rotation=38, ha='right', fontsize=FIG3D_BENCHMARK_DIST_STYLE['method_label'])
    ax.set_ylabel('Repeat-mean CV performance', fontsize=FIG3D_BENCHMARK_DIST_STYLE['axis_label'])
    ax.set_title(BENCHMARK_DIST_LABELS[metric], fontsize=FIG3D_BENCHMARK_DIST_STYLE['panel_title'], pad=10)
    ax.tick_params(axis='y', labelsize=FIG3D_BENCHMARK_DIST_STYLE['axis_tick'])
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle('Repeat-level performance distributions for simple integration benchmarks', fontsize=FIG3D_BENCHMARK_DIST_STYLE['main_title'], y=0.995)
plt.tight_layout(rect=[0.02, 0.02, 0.99, 0.94])
benchmark_dist_png = os.path.join(benchmark_dist_out_dir, get_value('science.figures.plot_088', 'plot_088.png'))
benchmark_dist_pdf = os.path.join(benchmark_dist_out_dir, get_value('science.figures.plot_089', 'plot_089.pdf'))
plt.savefig(benchmark_dist_png, dpi=600, bbox_inches='tight')
plt.savefig(benchmark_dist_pdf, bbox_inches='tight')
plt.show()



import os
from itertools import combinations
from math import factorial
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
SHAPLEY_REQUIRED_OBJECTS = ['MODALITY_ORDER', 'CV_SPLITS', 'N_SPLITS', 'labels', 'prepare_latent_modality_block', 'evaluate_design', 'output_root']
shapley_missing = [name for name in SHAPLEY_REQUIRED_OBJECTS if name not in globals()]
if shapley_missing:
    raise RuntimeError('Run Figure 3D Part 1 first. Missing objects: ' + ', '.join(shapley_missing))
FIG3D_SHAPLEY_STYLE = {'figure_size': (15.0, 5.7), 'main_title': 16.0, 'panel_title': 14.0, 'footnote': 9.0, 'row_label': 10.5, 'axis_label': 11.5, 'axis_tick': 10.5, 'marker_size': 6.2, 'interval_width': 1.3, 'cap_size': 2.7, 'value_label': 9.5, 'value_offset_fraction': 0.02, 'panel_spacing': 0.28}
FIG3D_SHAPLEY_COLORS = {'Metagenomics': '#7A9CAF', 'Species': '#A7B8BE', 'Blood': '#B88C7E', 'Urine': '#C2A58F', 'Radiomics': '#8D8176', 'zero': '#8A8A8A', 'text': '#1F1F1F', 'muted_text': '#333333', 'grid': '#E0E0E0'}
SHAPLEY_METRICS = ['Response_AUC', 'PFS_Cindex', 'OS_Cindex']
SHAPLEY_METRIC_LABELS = {'Response_AUC': 'Response AUC', 'PFS_Cindex': 'PFS C-index', 'OS_Cindex': 'OS C-index'}
SHAPLEY_MAX_FOLDS = None
SHAPLEY_EMPTY_COALITION_VALUE = 0.5
SHAPLEY_STRATEGY_NAME = 'Block-PCA early fusion'
shapley_out_dir = os.path.join(output_root, get_value('science.paths.directory_064', 'directory_064'))
os.makedirs(shapley_out_dir, exist_ok=True)
SHAPLEY_MODALITIES = tuple(MODALITY_ORDER)
SHAPLEY_MODALITY_COUNT = len(SHAPLEY_MODALITIES)

def _all_coalitions(modalities):
    coalitions = []
    for size in range(len(modalities) + 1):
        coalitions.extend(combinations(modalities, size))
    return [tuple(coalition) for coalition in coalitions]

def _build_shapley_block_design(latent_cache, coalition):
    train_design = np.column_stack([latent_cache[name][0] for name in coalition])
    test_design = np.column_stack([latent_cache[name][1] for name in coalition])
    scaler = StandardScaler()
    return (scaler.fit_transform(train_design), scaler.transform(test_design))

def _shapley_weight(coalition_size, modality_count):
    numerator = factorial(coalition_size) * factorial(modality_count - coalition_size - 1)
    return numerator / factorial(modality_count)
ALL_SHAPLEY_COALITIONS = _all_coalitions(SHAPLEY_MODALITIES)
print(f'Exact Shapley coalitions per fold: {len(ALL_SHAPLEY_COALITIONS)}')
selected_cv_splits = CV_SPLITS
if SHAPLEY_MAX_FOLDS is not None:
    selected_cv_splits = CV_SPLITS[:int(SHAPLEY_MAX_FOLDS)]
shapley_records = []
shapley_efficiency_records = []
coalition_value_records = []
for fold_number, (train_indices, test_indices) in enumerate(selected_cv_splits, start=1):
    repeat_number = (fold_number - 1) // int(N_SPLITS) + 1
    split_number = (fold_number - 1) % int(N_SPLITS) + 1
    train_labels = labels.iloc[train_indices].copy()
    test_labels = labels.iloc[test_indices].copy()
    
    latent_cache = {modality: prepare_latent_modality_block(modality, train_indices, test_indices) for modality in SHAPLEY_MODALITIES}
    coalition_values = {tuple(): {'Response_AUC': SHAPLEY_EMPTY_COALITION_VALUE, 'PFS_Cindex': SHAPLEY_EMPTY_COALITION_VALUE, 'OS_Cindex': SHAPLEY_EMPTY_COALITION_VALUE}}
    for coalition in ALL_SHAPLEY_COALITIONS:
        if len(coalition) == 0:
            continue
        train_design, test_design = _build_shapley_block_design(latent_cache, coalition)
        metrics = evaluate_design(train_design, test_design, train_labels, test_labels)
        coalition_values[tuple(sorted(coalition))] = metrics
        for metric in SHAPLEY_METRICS:
            coalition_value_records.append({'Fold': fold_number, 'Repeat': repeat_number, 'Split': split_number, 'Coalition': ' + '.join(coalition), 'Coalition_size': len(coalition), 'Metric': metric, 'Performance': metrics[metric]})
    for modality in SHAPLEY_MODALITIES:
        other_modalities = [name for name in SHAPLEY_MODALITIES if name != modality]
        for metric in SHAPLEY_METRICS:
            shapley_value = 0.0
            for coalition_size in range(len(other_modalities) + 1):
                for coalition in combinations(other_modalities, coalition_size):
                    coalition_key = tuple(sorted(coalition))
                    expanded_key = tuple(sorted(coalition + (modality,)))
                    marginal_gain = coalition_values[expanded_key][metric] - coalition_values[coalition_key][metric]
                    shapley_value += _shapley_weight(coalition_size, SHAPLEY_MODALITY_COUNT) * marginal_gain
            shapley_records.append({'Fold': fold_number, 'Repeat': repeat_number, 'Split': split_number, 'Strategy': SHAPLEY_STRATEGY_NAME, 'Modality': modality, 'Metric': metric, 'Shapley_value': shapley_value})
    full_key = tuple(sorted(SHAPLEY_MODALITIES))
    for metric in SHAPLEY_METRICS:
        fold_phi_sum = sum((row['Shapley_value'] for row in shapley_records if row['Fold'] == fold_number and row['Metric'] == metric))
        expected_sum = coalition_values[full_key][metric] - SHAPLEY_EMPTY_COALITION_VALUE
        shapley_efficiency_records.append({'Fold': fold_number, 'Repeat': repeat_number, 'Split': split_number, 'Metric': metric, 'Full_performance': coalition_values[full_key][metric], 'Empty_value': SHAPLEY_EMPTY_COALITION_VALUE, 'Sum_of_Shapley_values': fold_phi_sum, 'Expected_sum': expected_sum, 'Efficiency_error': fold_phi_sum - expected_sum})
shapley_fold_results = pd.DataFrame(shapley_records)
shapley_efficiency_check = pd.DataFrame(shapley_efficiency_records)
shapley_coalition_values = pd.DataFrame(coalition_value_records)
shapley_repeat_results = shapley_fold_results.groupby(['Strategy', 'Modality', 'Metric', 'Repeat'], as_index=False).agg(Repeat_mean_Shapley=('Shapley_value', 'mean'))
shapley_summary = shapley_repeat_results.groupby(['Strategy', 'Modality', 'Metric'], as_index=False).agg(Mean_Shapley=('Repeat_mean_Shapley', 'mean'), Min_repeat_Shapley=('Repeat_mean_Shapley', 'min'), Max_repeat_Shapley=('Repeat_mean_Shapley', 'max'), Median_Shapley=('Repeat_mean_Shapley', 'median'), Positive_repeat_fraction=('Repeat_mean_Shapley', lambda x: np.mean(np.asarray(x) > 0)), Repeat_SD=('Repeat_mean_Shapley', 'std'), Repeat_N=('Repeat_mean_Shapley', 'size'))
shapley_fold_results.to_csv(os.path.join(shapley_out_dir, get_value('science.tables.attribution_024', 'attribution_024.csv')), index=False, encoding='utf-8-sig')
shapley_repeat_results.to_csv(os.path.join(shapley_out_dir, get_value('science.tables.attribution_025', 'attribution_025.csv')), index=False, encoding='utf-8-sig')
shapley_summary.to_csv(os.path.join(shapley_out_dir, get_value('science.tables.attribution_026', 'attribution_026.csv')), index=False, encoding='utf-8-sig')
shapley_efficiency_check.to_csv(os.path.join(shapley_out_dir, get_value('science.tables.attribution_027', 'attribution_027.csv')), index=False, encoding='utf-8-sig')
shapley_coalition_values.to_csv(os.path.join(shapley_out_dir, get_value('science.tables.result_064', 'result_064.csv')), index=False, encoding='utf-8-sig')
max_efficiency_error = np.nanmax(np.abs(shapley_efficiency_check['Efficiency_error']))
print(f'Maximum absolute Shapley efficiency error: {max_efficiency_error:.3e}')
shapley_range_values = shapley_summary[['Min_repeat_Shapley', 'Max_repeat_Shapley']].to_numpy(dtype=float)
shapley_limit = np.nanmax(np.abs(shapley_range_values))
shapley_limit = max(0.02, np.ceil(shapley_limit * 100) / 100)
fig, axes = plt.subplots(1, 3, figsize=FIG3D_SHAPLEY_STYLE['figure_size'], sharey=True, gridspec_kw={'wspace': FIG3D_SHAPLEY_STYLE['panel_spacing']})
shapley_y = np.arange(len(SHAPLEY_MODALITIES), dtype=float)
for panel_index, metric in enumerate(SHAPLEY_METRICS):
    ax = axes[panel_index]
    subset = shapley_summary.loc[shapley_summary['Metric'] == metric].set_index('Modality').reindex(SHAPLEY_MODALITIES)
    mean_values = subset['Mean_Shapley'].to_numpy(dtype=float)
    lower_values = subset['Min_repeat_Shapley'].to_numpy(dtype=float)
    upper_values = subset['Max_repeat_Shapley'].to_numpy(dtype=float)
    xerr = np.vstack([mean_values - lower_values, upper_values - mean_values])
    ax.axvline(0, color=FIG3D_SHAPLEY_COLORS['zero'], lw=1.0, ls='--', zorder=1)
    for modality_index, modality in enumerate(SHAPLEY_MODALITIES):
        ax.errorbar(mean_values[modality_index], shapley_y[modality_index], xerr=xerr[:, modality_index].reshape(2, 1), fmt='o', markersize=FIG3D_SHAPLEY_STYLE['marker_size'], color=FIG3D_SHAPLEY_COLORS[modality], ecolor=FIG3D_SHAPLEY_COLORS[modality], elinewidth=FIG3D_SHAPLEY_STYLE['interval_width'], capsize=FIG3D_SHAPLEY_STYLE['cap_size'], markerfacecolor='white', markeredgewidth=1.3, zorder=4)
        label_offset = FIG3D_SHAPLEY_STYLE['value_offset_fraction'] * (2 * shapley_limit)
        x_value = mean_values[modality_index]
        horizontal_alignment = 'left' if x_value >= 0 else 'right'
        text_x = x_value + label_offset if x_value >= 0 else x_value - label_offset
        ax.text(text_x, shapley_y[modality_index], f'{x_value:+.3f}', ha=horizontal_alignment, va='center', fontsize=FIG3D_SHAPLEY_STYLE['value_label'], color=FIG3D_SHAPLEY_COLORS['text'])
    ax.set_xlim(-shapley_limit, shapley_limit)
    ax.set_ylim(-0.55, len(SHAPLEY_MODALITIES) - 0.45)
    ax.invert_yaxis()
    ax.set_yticks(shapley_y)
    ax.set_yticklabels(SHAPLEY_MODALITIES, fontsize=FIG3D_SHAPLEY_STYLE['row_label'])
    ax.set_xlabel('Mean marginal performance contribution', fontsize=FIG3D_SHAPLEY_STYLE['axis_label'])
    ax.set_title(SHAPLEY_METRIC_LABELS[metric], fontsize=FIG3D_SHAPLEY_STYLE['panel_title'], pad=10)
    ax.tick_params(axis='x', labelsize=FIG3D_SHAPLEY_STYLE['axis_tick'])
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle('Exact modality Shapley contributions under block-PCA early fusion', fontsize=FIG3D_SHAPLEY_STYLE['main_title'], y=0.995)
plt.tight_layout(rect=[0.02, 0.02, 0.99, 0.94])
shapley_png = os.path.join(shapley_out_dir, get_value('science.figures.plot_090', 'plot_090.png'))
shapley_pdf = os.path.join(shapley_out_dir, get_value('science.figures.plot_091', 'plot_091.pdf'))
plt.savefig(shapley_png, dpi=600, bbox_inches='tight')
plt.savefig(shapley_pdf, bbox_inches='tight')
plt.show()






In [ ]:
pass
from pathlib import Path
import json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'Arial', 'font.size': 11.0, 'axes.titlesize': 14.0, 'axes.labelsize': 11.5, 'xtick.labelsize': 10.5, 'ytick.labelsize': 10.5, 'legend.fontsize': 10.0, 'figure.titlesize': 16.0, 'text.color': '#1F1F1F', 'axes.labelcolor': '#1F1F1F', 'axes.titlecolor': '#1F1F1F', 'xtick.color': '#1F1F1F', 'ytick.color': '#1F1F1F', 'axes.edgecolor': '#2B2B2B', 'axes.grid': False, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none'})
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore', category=RuntimeWarning)
ROOT = Path(str(project_path('')))
AVATAR = ROOT / get_value('science.paths.directory_008', 'directory_008')
INPUT = AVATAR / get_value('science.paths.directory_009', 'directory_009')
METAB_BASE = ROOT / get_value('science.paths.directory_065', 'directory_065') / get_value('science.paths.directory_066', 'directory_066') / get_value('science.paths.directory_067', 'directory_067')
METAB_AUDIT = METAB_BASE / get_value('science.paths.directory_068', 'directory_068')
METAB_PREP = METAB_BASE / get_value('science.paths.directory_069', 'directory_069')
KO_PREP = ROOT / get_value('science.paths.directory_045', 'directory_045') / get_value('science.paths.directory_070', 'directory_070') / get_value('science.paths.directory_071', 'directory_071')
LEGACY = ROOT / get_value('science.paths.directory_045', 'directory_045') / get_value('science.paths.directory_048', 'directory_048')
OUT = AVATAR / get_value('science.paths.directory_053', 'directory_053') / get_value('science.paths.directory_072', 'directory_072')
OUT.mkdir(parents=True, exist_ok=True)
EXPECTED_N = 109
SPECIES_PREVALENCE = 0.1
N_AXES = 3
TOP_N = 5
DPI = 450
RANDOM_STATE = 20260801
MODALITIES = ['KO', 'Species', 'Blood', 'Urine', 'Radiomics']
COLORS = {'KO': '#6D8EAD', 'Species': '#7FA58A', 'Blood': '#B88A72', 'Urine': '#A69BC3', 'Radiomics': '#C5A45D'}
plt.rcParams.update({'font.family': 'Arial', 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none', 'axes.titlesize': 11, 'axes.labelsize': 9, 'xtick.labelsize': 8, 'ytick.labelsize': 8})
PATH_AUDIT = []

def norm_id(x):
    if pd.isna(x):
        return ''
    return re.sub('\\.0$', '', str(x).strip().strip('"').strip("'")).upper()

def resolve(label, candidates, required=False, note=''):
    candidates = [Path(x) for x in candidates]
    selected = next((p for p in candidates if p.exists()), None)
    PATH_AUDIT.append({'Item': label, 'Required': required, 'Status': 'FOUND' if selected else 'MISSING_REQUIRED' if required else 'MISSING_OPTIONAL', 'Selected path': str(selected) if selected else '', 'Candidate paths': ' | '.join(map(str, candidates)), 'Note': note})
    return selected

def unavailable(label, note):
    PATH_AUDIT.append({'Item': label, 'Required': False, 'Status': 'UNAVAILABLE_BY_DESIGN', 'Selected path': '', 'Candidate paths': '', 'Note': note})

def read_csv(path, **kwargs):
    last = None
    for enc in ['utf-8-sig', 'utf-8', 'gb18030']:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False, **kwargs)
        except UnicodeDecodeError as e:
            last = e
    raise last

def sample_col(df):
    m = {str(c).strip().lower(): c for c in df.columns}
    for x in ['sample id', 'sample_id', 'sample', 'id', 'patient id', 'patient_id', 'unnamed: 0', 'index']:
        if x in m:
            return m[x]
    return df.columns[0]

def read_matrix(path):
    df = read_csv(path)
    sc = sample_col(df)
    df = df.rename(columns={sc: 'Sample ID'})
    df['Sample ID'] = df['Sample ID'].map(norm_id)
    df = df.loc[df['Sample ID'] != ''].copy()
    if df['Sample ID'].duplicated().any():
        raise ValueError(f'Duplicate sample identifiers: {path}')
    df = df.set_index('Sample ID').apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).dropna(axis=1, how='all')
    return df

def read_meta(path):
    df = read_csv(path)
    sc = sample_col(df)
    df = df.rename(columns={sc: 'Sample ID'})
    df['Sample ID'] = df['Sample ID'].map(norm_id)
    return df.loc[df['Sample ID'] != ''].drop_duplicates('Sample ID').set_index('Sample ID', drop=False)

def bh_fdr(p):
    p = np.asarray(p, float)
    q = np.full(p.shape, np.nan)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return q
    pv = p[ok]
    order = np.argsort(pv)
    ranked = pv[order]
    n = len(ranked)
    qr = np.minimum.accumulate((ranked * n / np.arange(1, n + 1))[::-1])[::-1]
    qr = np.clip(qr, 0, 1)
    out = np.empty(n)
    out[order] = qr
    q[ok] = out
    return q

def mad_flag(v, cutoff=4.5):
    s = pd.Series(v, dtype=float)
    med = s.median()
    mad = np.nanmedian(np.abs(s - med))
    if not np.isfinite(mad) or mad <= 0:
        return pd.Series(False, index=s.index)
    return (0.67448975 * (s - med) / mad).abs() > cutoff

def clean_pca(X):
    X = X.replace([np.inf, -np.inf], np.nan).dropna(axis=1, how='all').copy()
    X = X.fillna(X.median())
    var = X.var(axis=0, ddof=0)
    X = X.loc[:, np.isfinite(var) & (var > 0)]
    if X.shape[1] == 0:
        raise ValueError('No features are available before PCA')
    return (StandardScaler().fit_transform(X), X.columns.tolist())

def run_pca(X):
    values, features = clean_pca(X)
    n = min(max(5, N_AXES), values.shape[0] - 1, values.shape[1])
    model = PCA(n_components=n, svd_solver='full', random_state=RANDOM_STATE)
    scores = model.fit_transform(values)
    comps = model.components_.copy()
    orient = []
    for a in range(N_AXES):
        idx = int(np.nanargmax(np.abs(comps[a])))
        sign = 1 if comps[a, idx] >= 0 else -1
        scores[:, a] *= sign
        comps[a] *= sign
        orient.append({'Axis': a + 1, 'Anchor feature': features[idx], 'Applied sign': sign})
    return {'method': 'PCA', 'scores': scores[:, :N_AXES], 'explained': model.explained_variance_ratio_[:N_AXES] * 100, 'features': features, 'effects': comps[:N_AXES].T, 'orientation': orient}

def run_species_pcoa(X):
    X = X.fillna(0).clip(lower=0)
    X = X.loc[:, X.sum(axis=0) > 0]
    sums = X.sum(axis=1)
    if (sums <= 0).any():
        raise ValueError('The species matrix contains samples with zero total abundance')
    rel = X.div(sums, axis=0)
    D = pairwise_distances(rel, metric='braycurtis')
    D = np.nan_to_num(D)
    n = len(X)
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ D ** 2 @ J
    eig, vec = np.linalg.eigh(B)
    order = np.argsort(eig)[::-1]
    eig = eig[order]
    vec = vec[:, order]
    pos = eig > 1e-12
    eig = eig[pos]
    vec = vec[:, pos]
    scores = vec[:, :N_AXES] * np.sqrt(eig[:N_AXES])
    explained = eig[:N_AXES] / eig.sum() * 100
    orient = []
    for a in range(N_AXES):
        r = np.array([spearmanr(rel[c], scores[:, a], nan_policy='omit').statistic for c in rel.columns])
        idx = int(np.nanargmax(np.abs(r)))
        sign = 1 if r[idx] >= 0 else -1
        scores[:, a] *= sign
        orient.append({'Axis': a + 1, 'Anchor feature': rel.columns[idx], 'Applied sign': sign})
    rows = []
    for a in range(N_AXES):
        tmp = []
        p = []
        for c in rel.columns:
            z = spearmanr(rel[c], scores[:, a], nan_policy='omit')
            tmp.append({'Modality': 'Species', 'Axis': a + 1, 'Feature': c, 'Effect': z.statistic, 'P value': z.pvalue})
            p.append(z.pvalue)
        q = bh_fdr(p)
        for r, qq in zip(tmp, q):
            r['FDR'] = qq
            rows.append(r)
    return {'method': 'Bray-Curtis PCoA', 'scores': scores, 'explained': explained, 'relative': rel, 'correlations': pd.DataFrame(rows), 'orientation': orient}

def savefig(fig, stem):
    for ext in ['png', 'pdf', 'svg']:
        kw = {'dpi': DPI} if ext == 'png' else {}
        fig.savefig(OUT / f'{stem}.{ext}', bbox_inches='tight', facecolor='white', **kw)
    

def panel(ax, label, x=-0.13, y=1.08):
    ax.text(x, y, label, transform=ax.transAxes, fontsize=14, fontweight='bold', va='top')

def missing_panel(ax, title, msg):
    ax.axis('off')
    ax.set_title(title, loc='left')
    ax.text(0.03, 0.55, msg, transform=ax.transAxes, va='center', wrap=True, fontsize=10.5)

def short(x, n=46):
    x = str(x).replace('log-sigma-4-0-mm-3D_', 'LoG_').replace('wavelet-', 'Wav-').replace('original_', 'Orig_')
    return x if len(x) <= n else x[:n - 1] + '…'

def batch_r2(scores, labels):
    scores = np.asarray(scores, float)
    labels = np.asarray(labels, str)
    grand = scores.mean(0)
    total = ((scores - grand) ** 2).sum()
    between = 0
    for g in np.unique(labels):
        m = labels == g
        between += m.sum() * ((scores[m].mean(0) - grand) ** 2).sum()
    return between / total if total > 0 else np.nan
P = {}
P['KO'] = resolve('KO model-ready', [INPUT / get_value('science.tables.features_001', 'features_001.csv.gz'), KO_PREP / get_value('science.tables.features_001', 'features_001.csv.gz')], True)
P['KO_QC'] = resolve('KO sample QC', [KO_PREP / get_value('science.tables.result_065', 'result_065.csv')])
P['KO_SUM'] = resolve('KO preprocessing summary', [KO_PREP / get_value('science.tables.metrics_040', 'metrics_040.csv')])
P['Species_raw'] = resolve('Species raw matrix', [INPUT / get_value('science.tables.result_066', 'result_066.csv.gz'), INPUT / get_value('science.tables.result_060', 'result_060.csv'), LEGACY / get_value('science.tables.result_066', 'result_066.csv.gz'), LEGACY / get_value('science.tables.result_060', 'result_060.csv')])
P['Species_filtered'] = resolve('Species p10 matrix', [INPUT / get_value('science.tables.result_050', 'result_050.csv.gz'), INPUT / get_value('science.tables.result_067', 'result_067.csv')])
P['Species'] = P['Species_filtered'] or P['Species_raw']
if P['Species'] is None:
    PATH_AUDIT.append({'Item': 'Species analysis matrix', 'Required': True, 'Status': 'MISSING_REQUIRED', 'Selected path': '', 'Candidate paths': 'raw or p10', 'Note': 'At least one species matrix is required'})
for m, num in [('Blood', '10'), ('Urine', '11')]:
    P[m] = resolve(f'{m} model-ready', [INPUT / f"{m}{get_value('science.tables.features_025', 'features_025.csv.gz')}", METAB_PREP / f"{m}{get_value('science.tables.features_025', 'features_025.csv.gz')}"], True)
    P[f'{m}_pre'] = resolve(f'{m} pre-ComBat', [METAB_PREP / f"{m}{get_value('science.tables.result_068', 'result_068.csv.gz')}"])
    P[f'{m}_post'] = resolve(f'{m} post-ComBat', [METAB_PREP / f"{m}{get_value('science.tables.result_069', 'result_069.csv.gz')}"])
    P[f'{m}_meta'] = resolve(f'{m} batch metadata', [METAB_AUDIT / f"{num}_{m}{get_value('science.tables.result_070', 'result_070.csv')}"])
    P[f'{m}_manifest'] = resolve(f'{m} feature manifest', [METAB_PREP / f"{m}{get_value('science.tables.result_071', 'result_071.csv')}"])
    P[f'{m}_QC'] = resolve(f'{m} sample QC', [METAB_PREP / f"{m}{get_value('science.tables.result_072', 'result_072.csv')}"])
P['METAB_SUM'] = resolve('Blood/Urine preprocessing summary', [METAB_PREP / get_value('science.tables.metrics_041', 'metrics_041.csv')])
P['Radiomics'] = resolve('Radiomics model-ready', [INPUT / get_value('science.tables.features_002', 'features_002.csv.gz'), LEGACY / get_value('science.tables.result_073', 'result_073.csv')], True)
unavailable('Radiomics ICC results', 'No traceable ICC results table is currently available.')
unavailable('Radiomics raw-feature count and ICC-filtering log', 'No traceable raw-feature count or ICC-filtering log is currently available; only the final model-ready feature count is reported.')
audit = pd.DataFrame(PATH_AUDIT)
audit.to_csv(OUT / get_value('science.tables.result_074', 'result_074.csv'), index=False, encoding='utf-8-sig')
miss = audit.loc[audit.Status.eq('MISSING_REQUIRED'), 'Item'].tolist()
if miss:
    raise FileNotFoundError('Required files are missing:\n- ' + '\n- '.join(miss) + f"\nSee {OUT / get_value('science.tables.result_074', 'result_074.csv')}")

X = {m: read_matrix(P[m]) for m in MODALITIES}
order = X['KO'].index.tolist()
common = [s for s in order if all((s in X[m].index for m in MODALITIES))]
align = []
for m in MODALITIES:
    align.append({'Modality': m, 'Original samples': X[m].shape[0], 'Original features': X[m].shape[1], 'Common samples': len(common), 'Missing cells': int(X[m].isna().sum().sum())})
    X[m] = X[m].loc[common].copy()
pd.DataFrame(align).to_csv(OUT / get_value('science.tables.result_075', 'result_075.csv'), index=False, encoding='utf-8-sig')
if len(common) != EXPECTED_N:
    warnings.warn(f'Common sample count={len(common)}, rather than the expected {EXPECTED_N}')

def val(df, modality, names):
    if df is None or df.empty:
        return np.nan
    row = df.iloc[0]
    mc = next((c for c in df.columns if str(c).strip().lower() == 'modality'), None)
    if mc:
        tmp = df.loc[df[mc].astype(str).str.lower() == modality.lower()]
        if tmp.empty:
            return np.nan
        row = tmp.iloc[0]
    lm = {str(c).strip().lower(): c for c in df.columns}
    for n in names:
        if n.lower() in lm:
            return pd.to_numeric(pd.Series([row[lm[n.lower()]]]), errors='coerce').iloc[0]
    return np.nan
ko_sum = read_csv(P['KO_SUM']) if P['KO_SUM'] else None
met_sum = read_csv(P['METAB_SUM']) if P['METAB_SUM'] else None
Sraw = read_matrix(P['Species_raw']) if P['Species_raw'] else X['Species'].copy()
Sraw = Sraw.loc[[s for s in common if s in Sraw.index]].fillna(0)
allzero = int((Sraw.sum(0) <= 0).sum())
Snz = Sraw.loc[:, Sraw.sum(0) > 0]
prev = (Snz > 0).mean(0)
kept = int((prev >= SPECIES_PREVALENCE).sum())
rows = [{'Modality': 'KO', 'Raw/input features': val(ko_sum, 'KO', ['Input KO features', 'Raw features']), 'After filter': val(ko_sum, 'KO', ['KO features passing detection filter', 'Features after detection filter']), 'Final model-ready': X['KO'].shape[1], 'Processing': 'Detection >=20%; log1p; zero-variance removal; internal Z-score', 'Batch correction': 'No', 'Note': 'KO summary used when available'}, {'Modality': 'Species', 'Raw/input features': Sraw.shape[1], 'After filter': kept, 'Final model-ready': X['Species'].shape[1], 'Processing': f'All-zero removal; prevalence >={SPECIES_PREVALENCE:.0%}; relative abundance for Bray-Curtis', 'Batch correction': 'No', 'Note': f'All-zero features={allzero}'}]
for m in ['Blood', 'Urine']:
    rows.append({'Modality': m, 'Raw/input features': val(met_sum, m, ['Raw features']), 'After filter': val(met_sum, m, ['Features after detection filter']), 'Final model-ready': X[m].shape[1], 'Processing': 'Detection >=50%; imputation; PQN; log2; POS/NEG ComBat; zero-variance removal; internal Z-score', 'Batch correction': 'Yes', 'Note': 'preprocessing summary table used when available'})
rows.append({'Modality': 'Radiomics', 'Raw/input features': np.nan, 'After filter': np.nan, 'Final model-ready': X['Radiomics'].shape[1], 'Processing': 'Current model-ready matrix only', 'Batch correction': 'No', 'Note': 'ICC/raw feature-count log unavailable; no values inferred'})
ret = pd.DataFrame(rows)
ret.to_csv(OUT / get_value('science.tables.metrics_042', 'metrics_042.csv'), index=False, encoding='utf-8-sig')
pd.DataFrame({'Feature': prev.index, 'Internal prevalence': prev.values, 'Pass p10': prev.values >= SPECIES_PREVALENCE}).sort_values('Internal prevalence', ascending=False).to_csv(OUT / get_value('science.tables.result_076', 'result_076.csv'), index=False, encoding='utf-8-sig')

def parse_rad(f):
    s = str(f)
    l = s.lower()
    trans = 'Original' if l.startswith('original_') else 'Wavelet' if l.startswith('wavelet-') else 'LoG' if l.startswith('log-sigma-') else 'Other'
    z = re.search('_(shape2d|shape|firstorder|glcm|gldm|glrlm|glszm|ngtdm)_', l)
    cls = z.group(1).upper() if z else 'OTHER'
    cls = 'First-order' if cls == 'FIRSTORDER' else cls
    return (trans, cls)
rad = pd.DataFrame([{'Feature': f, 'Transformation family': parse_rad(f)[0], 'Feature class': parse_rad(f)[1]} for f in X['Radiomics'].columns])
rad.to_csv(OUT / get_value('science.tables.result_077', 'result_077.csv'), index=False, encoding='utf-8-sig')
rad.groupby(['Feature class', 'Transformation family']).size().rename('Feature count').reset_index().to_csv(OUT / get_value('science.tables.metrics_043', 'metrics_043.csv'), index=False, encoding='utf-8-sig')
fig, axarr = plt.subplots(2, 3, figsize=(18, 10.8))
axarr = axarr.ravel()
letters = list('ABCDEF')
ax = axarr[0]
cols = ['Raw/input features', 'After filter', 'Final model-ready']
tab = ret.set_index('Modality').loc[MODALITIES, cols]
a = tab.to_numpy(float)
log = np.where(np.isfinite(a) & (a > 0), np.log10(a), np.nan)
cmap = plt.get_cmap('Blues').copy()
cmap.set_bad('#ECECEC')
ax.imshow(np.ma.masked_invalid(log), aspect='auto', cmap=cmap)
ax.set_xticks(range(3))
ax.set_xticklabels(['Raw/input', 'After filter', 'Model-ready'], rotation=20, ha='right')
ax.set_yticks(range(5))
ax.set_yticklabels(MODALITIES)
ax.set_title('Feature-retention audit', loc='left')
for i in range(5):
    for j in range(3):
        ax.text(j, i, 'NA' if not np.isfinite(a[i, j]) else f'{int(a[i, j]):,}', ha='center', va='center', fontsize=10)
panel(ax, 'A')
ax = axarr[1]
if P['KO_QC']:
    q = read_meta(P['KO_QC'])
    q = q.loc[q.Cohort.astype(str).str.lower().eq('internal')] if 'Cohort' in q else q
    if {'Raw total KO abundance', 'Raw detected KO N'}.issubset(q.columns):
        xx = np.log10(pd.to_numeric(q['Raw total KO abundance'], errors='coerce').clip(lower=1))
        yy = pd.to_numeric(q['Raw detected KO N'], errors='coerce')
        ff = mad_flag(xx) | mad_flag(yy)
        ax.scatter(xx[~ff], yy[~ff], s=25, color=COLORS['KO'], alpha=0.72, edgecolor='white', linewidth=0.3)
        ax.scatter(xx[ff], yy[ff], s=42, facecolor='none', edgecolor='#7A3030')
        ax.set_xlabel('log10(total KO abundance)')
        ax.set_ylabel('Detected KO count')
        ax.set_title(f'KO sample QC (flagged={ff.sum()})', loc='left')
    else:
        missing_panel(ax, 'KO sample QC', 'The microbial QC table is missing expected columns')
else:
    missing_panel(ax, 'KO sample QC', 'Microbial QC table not found; raw QC is not inferred from the model-ready matrix')
panel(ax, 'B')
ax = axarr[2]
tot = Sraw.sum(1)
det = (Sraw > 0).sum(1)
xx = np.log10(tot.clip(lower=1))
yy = det
ff = mad_flag(xx) | mad_flag(yy)
ax.scatter(xx[~ff], yy[~ff], s=25, color=COLORS['Species'], alpha=0.72, edgecolor='white', linewidth=0.3)
ax.scatter(xx[ff], yy[ff], s=42, facecolor='none', edgecolor='#7A3030')
ax.set_xlabel('log10(total species reads/abundance)')
ax.set_ylabel('Detected species count')
ax.set_title(f'Species sample QC (flagged={ff.sum()})', loc='left')
panel(ax, 'C')
for k, m, L in [(3, 'Blood', 'D'), (4, 'Urine', 'E')]:
    ax = axarr[k]
    if P[f'{m}_QC']:
        q = read_meta(P[f'{m}_QC'])
        q = q.loc[q.Cohort.astype(str).str.lower().eq('internal')] if 'Cohort' in q else q
        if {'Raw zero/NA rate', 'Raw log10 total signal'}.issubset(q.columns):
            xx = pd.to_numeric(q['Raw zero/NA rate'], errors='coerce')
            yy = pd.to_numeric(q['Raw log10 total signal'], errors='coerce')
            ff = mad_flag(xx) | mad_flag(yy)
            ax.scatter(xx[~ff], yy[~ff], s=25, color=COLORS[m], alpha=0.72, edgecolor='white', linewidth=0.3)
            ax.scatter(xx[ff], yy[ff], s=42, facecolor='none', edgecolor='#7A3030')
            ax.set_xlabel('Raw zero/NA rate')
            ax.set_ylabel('Raw log10 total signal')
            ax.set_title(f'{m} sample QC (flagged={ff.sum()})', loc='left')
        else:
            missing_panel(ax, f'{m} sample QC', 'The QC file is missing expected columns')
    else:
        missing_panel(ax, f'{m} sample QC', 'Sample QC file not found')
    panel(ax, L)
ax = axarr[5]
cc = rad['Feature class'].value_counts().sort_values()
ax.barh(range(len(cc)), cc.values, color=COLORS['Radiomics'], alpha=0.84)
ax.set_yticks(range(len(cc)))
ax.set_yticklabels(cc.index)
ax.set_xlabel('Current model-ready feature count')
ax.set_title('Radiomics feature composition', loc='left')
for i, v in enumerate(cc.values):
    ax.text(v, i, f'  {v}', va='center', fontsize=10)
for ax in axarr:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle('Supplementary Figure S3-1 | Modality-specific QC and feature retention', fontsize=16, y=1.01)
fig.tight_layout(h_pad=2.8, w_pad=2)
savefig(fig, 'Supplementary_Figure_S3_1_QC_and_feature_retention')
plt.close(fig)

def ion(x):
    s = str(x).strip().upper()
    return 'POS' if 'POS' in s or s in ['+', 'P'] else 'NEG' if 'NEG' in s or s in ['-', 'N'] else s

def mode_features(path, available):
    d = read_csv(path)
    lm = {str(c).strip().lower(): c for c in d.columns}
    fc = next((lm[x] for x in ['model column name', 'feature', 'feature id', 'cleaned candidate'] if x in lm), None)
    mc = next((lm[x] for x in ['ion mode', 'mode'] if x in lm), None)
    if fc is None or mc is None:
        raise ValueError(f'The manifest is missing the feature-name or ion-mode column: {path}')
    d[fc] = d[fc].astype(str)
    d[mc] = d[mc].map(ion)
    avail = set(available)
    return {m: [f for f in d.loc[d[mc].eq(m), fc] if f in avail] for m in ['POS', 'NEG']}
batch_results = {}
score_rows = []
effect_rows = []
for m in ['Blood', 'Urine']:
    needed = [P[f'{m}_pre'], P[f'{m}_post'], P[f'{m}_meta'], P[f'{m}_manifest']]
    if not all(needed):
        for mode in ['POS', 'NEG']:
            batch_results[m, mode] = None
        continue
    pre = read_matrix(P[f'{m}_pre'])
    post = read_matrix(P[f'{m}_post'])
    meta = read_meta(P[f'{m}_meta'])
    meta = meta.loc[meta.Cohort.astype(str).str.lower().eq('internal')] if 'Cohort' in meta else meta
    ids = [s for s in pre.index if s in post.index and s in meta.index]
    pre = pre.loc[ids]
    post = post.loc[ids]
    mf = mode_features(P[f'{m}_manifest'], set(pre.columns) & set(post.columns))
    for mode in ['POS', 'NEG']:
        bc = f'{mode} Batch ID'
        features = mf.get(mode, [])
        if bc not in meta or len(features) < 2:
            batch_results[m, mode] = None
            continue
        labels = meta.loc[ids, bc].astype(str)
        result = {}
        for stage, mat in [('Pre-ComBat', pre), ('Post-ComBat', post)]:
            v, used = clean_pca(mat.loc[:, features])
            n = min(10, v.shape[0] - 1, v.shape[1])
            pc = PCA(n_components=n, svd_solver='full', random_state=RANDOM_STATE)
            s = pc.fit_transform(v)
            r2 = batch_r2(s, labels)
            result[stage] = {'scores': s, 'exp': pc.explained_variance_ratio_ * 100, 'labels': labels.to_numpy(), 'ids': ids, 'r2': r2, 'features': len(used)}
            effect_rows.append({'Modality': m, 'Ion mode': mode, 'Stage': stage, 'Samples': len(ids), 'Features': len(used), 'Batch levels': labels.nunique(), 'Descriptive batch R2 on first PCs': r2})
            for sid, b, p1, p2 in zip(ids, labels, s[:, 0], s[:, 1]):
                score_rows.append({'Sample ID': sid, 'Modality': m, 'Ion mode': mode, 'Stage': stage, 'Batch ID': b, 'PC1': p1, 'PC2': p2, 'PC1 explained variance (%)': pc.explained_variance_ratio_[0] * 100, 'PC2 explained variance (%)': pc.explained_variance_ratio_[1] * 100})
        batch_results[m, mode] = result
pd.DataFrame(score_rows).to_csv(OUT / get_value('science.tables.result_078', 'result_078.csv'), index=False, encoding='utf-8-sig')
pd.DataFrame(effect_rows).to_csv(OUT / get_value('science.tables.metrics_044', 'metrics_044.csv'), index=False, encoding='utf-8-sig')
fig, axs = plt.subplots(4, 2, figsize=(12.5, 17))
spec = [('Blood', 'POS'), ('Blood', 'NEG'), ('Urine', 'POS'), ('Urine', 'NEG')]
letters = list('ABCDEFGH')
li = 0
for r, (m, mode) in enumerate(spec):
    res = batch_results.get((m, mode))
    for c, stage in enumerate(['Pre-ComBat', 'Post-ComBat']):
        ax = axs[r, c]
        if res is None:
            missing_panel(ax, f'{m} {mode} | {stage}', 'A pre/post matrix, batch metadata or feature manifest is missing')
        else:
            z = res[stage]
            labs = pd.Series(z['labels'], dtype=str)
            groups = sorted(labs.unique())
            cm = plt.get_cmap('tab20', max(len(groups), 2))
            for i, g in enumerate(groups):
                mask = labs.to_numpy() == g
                ax.scatter(z['scores'][mask, 0], z['scores'][mask, 1], s=30, color=cm(i), alpha=0.78, edgecolor='white', linewidth=0.3, label=g)
            ax.set_xlabel(f"PC1 ({z['exp'][0]:.1f}%)")
            ax.set_ylabel(f"PC2 ({z['exp'][1]:.1f}%)")
            ax.set_title(f"{m} {mode} | {stage}\nBatch R²={z['r2']:.3f}; features={z['features']}", loc='left')
            if c == 1:
                ax.legend(title='Batch', frameon=False, fontsize=10, title_fontsize=10, bbox_to_anchor=(1.02, 1), loc='upper left')
        panel(ax, letters[li])
        li += 1
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
fig.suptitle('Supplementary Figure S3-1 | Mode-specific metabolomics batch correction', fontsize=16, y=0.997)
fig.tight_layout(h_pad=2.2, w_pad=2)
savefig(fig, 'Supplementary_Figure_S3_1_metabolomics_batch_correction')
plt.close(fig)
R = {m: run_pca(X[m]) for m in ['KO', 'Blood', 'Urine', 'Radiomics']}
R['Species'] = run_species_pcoa(X['Species'])
score_rows = []
exp_rows = []
ori_rows = []
effect_all = []
for m in MODALITIES:
    z = R[m]
    for sid, v in zip(common, z['scores']):
        score_rows.append({'Sample ID': sid, 'Modality': m, 'Method': z['method'], 'Axis 1': v[0], 'Axis 2': v[1], 'Axis 3': v[2]})
    for a, v in enumerate(z['explained'], 1):
        exp_rows.append({'Modality': m, 'Method': z['method'], 'Axis': a, 'Explained variance (%)': v})
    for q in z['orientation']:
        ori_rows.append({'Modality': m, 'Method': z['method'], **q})
    if m == 'Species':
        for _, q in z['correlations'].iterrows():
            effect_all.append({'Modality': m, 'Axis': int(q.Axis), 'Feature': q.Feature, 'Effect': q.Effect, 'Effect type': 'Spearman rho', 'P value': q['P value'], 'FDR': q.FDR})
    else:
        for i, f in enumerate(z['features']):
            for a in range(N_AXES):
                effect_all.append({'Modality': m, 'Axis': a + 1, 'Feature': f, 'Effect': z['effects'][i, a], 'Effect type': 'PCA loading', 'P value': np.nan, 'FDR': np.nan})
scores = pd.DataFrame(score_rows)
expl = pd.DataFrame(exp_rows)
orient = pd.DataFrame(ori_rows)
effects = pd.DataFrame(effect_all)
scores.to_csv(OUT / get_value('science.tables.result_079', 'result_079.csv'), index=False, encoding='utf-8-sig')
expl.to_csv(OUT / get_value('science.tables.result_080', 'result_080.csv'), index=False, encoding='utf-8-sig')
orient.to_csv(OUT / get_value('science.tables.result_081', 'result_081.csv'), index=False, encoding='utf-8-sig')
effects.to_csv(OUT / get_value('science.tables.result_082', 'result_082.csv.gz'), index=False, compression='gzip', encoding='utf-8-sig')
R['Species']['relative'].to_csv(OUT / get_value('science.tables.result_083', 'result_083.csv.gz'), compression='gzip', encoding='utf-8-sig')
R['Species']['correlations'].to_csv(OUT / get_value('science.tables.result_084', 'result_084.csv.gz'), index=False, compression='gzip', encoding='utf-8-sig')
top = []
for m in MODALITIES:
    for a in range(1, N_AXES + 1):
        q = effects.loc[(effects.Modality == m) & (effects.Axis == a)].copy()
        q['Absolute effect'] = q.Effect.abs()
        top.extend(q.nlargest(TOP_N, 'Absolute effect').to_dict('records'))
top = pd.DataFrame(top)
top.to_csv(OUT / get_value('science.tables.result_085', 'result_085.csv'), index=False, encoding='utf-8-sig')
fig, axs = plt.subplots(2, 3, figsize=(16.5, 10.6))
axs = axs.ravel()
letters = list('ABCDEF')
for k, m in enumerate(MODALITIES):
    ax = axs[k]
    z = R[m]
    s = z['scores']
    zz = StandardScaler().fit_transform(s[:, :3])
    far = np.argsort((zz ** 2).sum(1))[-3:]
    ax.scatter(s[:, 0], s[:, 1], s=25, color=COLORS[m], alpha=0.72, edgecolor='white', linewidth=0.3)
    for i in far:
        ax.text(s[i, 0], s[i, 1], ' ' + common[i], fontsize=9.5, va='center')
    prefix = 'PCoA' if m == 'Species' else 'PC'
    ax.set_xlabel(f"{prefix}1 ({z['explained'][0]:.1f}%)")
    ax.set_ylabel(f"{prefix}2 ({z['explained'][1]:.1f}%)")
    ax.set_title(f"{m} | {z['method']}", loc='left')
    ax.axhline(0, color='#D9D9D9', lw=0.6)
    ax.axvline(0, color='#D9D9D9', lw=0.6)
    panel(ax, letters[k])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
ax = axs[5]
pv = expl.pivot(index='Modality', columns='Axis', values='Explained variance (%)').reindex(MODALITIES)
xp = np.arange(5)
w = 0.23
for a in [1, 2, 3]:
    ax.bar(xp + (a - 2) * w, pv[a], width=w, label=f'Axis {a}', alpha=0.82)
ax.set_xticks(xp)
ax.set_xticklabels(MODALITIES, rotation=20, ha='right')
ax.set_ylabel('Explained variance (%)')
ax.set_title('Variance captured by first three axes', loc='left')
ax.legend(frameon=False, ncol=3, fontsize=10)
panel(ax, 'F')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.suptitle('Supplementary Figure S3-2 | Outcome-independent modality structure', fontsize=16, y=1.01)
fig.tight_layout(h_pad=2.2, w_pad=2)
savefig(fig, 'Supplementary_Figure_S3_2_low_dimensional_structure')
plt.close(fig)
fig, axs = plt.subplots(5, 3, figsize=(18, 19))
letters = [chr(65 + i) for i in range(15)]
li = 0
for r, m in enumerate(MODALITIES):
    for c, a in enumerate([1, 2, 3]):
        ax = axs[r, c]
        q = top.loc[(top.Modality == m) & (top.Axis == a)].sort_values('Effect')
        yp = np.arange(len(q))
        barcolors = ['#6D8EAD' if v >= 0 else '#B88A72' for v in q.Effect]
        ax.barh(yp, q.Effect, color=barcolors, alpha=0.86)
        ax.set_yticks(yp)
        ax.set_yticklabels([short(f) for f in q.Feature])
        ax.axvline(0, color='#555', lw=0.7)
        ax.set_xlabel('Spearman ρ' if m == 'Species' else 'Signed PCA loading')
        ax.set_title(f"{m} | {('PCoA' if m == 'Species' else 'PC')}{a}", loc='left')
        for y, (_, v) in zip(yp, q.iterrows()):
            text = f'{v.Effect:.2f}' + (f'; q={v.FDR:.2g}' if m == 'Species' and pd.notna(v.FDR) else '')
            off = 0.015 if v.Effect >= 0 else -0.015
            ax.text(v.Effect + off, y, text, va='center', ha='left' if v.Effect >= 0 else 'right', fontsize=10)
        lim = max(0.05, q.Effect.abs().max() * 1.35)
        ax.set_xlim(-lim, lim)
        panel(ax, letters[li], x=-0.22)
        li += 1
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
fig.suptitle('Supplementary Figure S3-2 | Major drivers of the first three modality axes', fontsize=16, y=0.998)
fig.tight_layout(h_pad=2.2, w_pad=2.7)
savefig(fig, 'Supplementary_Figure_S3_2_major_axis_drivers')
plt.close(fig)
summary = {'Output directory': str(OUT), 'Common internal samples': len(common), 'Expected internal samples': EXPECTED_N, 'Species prevalence threshold': SPECIES_PREVALENCE, 'Radiomics ICC available': False, 'Radiomics raw/ICC filtering log available': False, 'Radiomics handling': 'Current model-ready matrix only; no upstream values inferred', 'Batch correction': 'Blood and Urine only; POS and NEG separately', 'Axis orientation': 'Outcome-independent; largest absolute loading/correlation anchor'}
(OUT / get_value('science.tables.metrics_045', 'metrics_045.json')).write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT / get_value('science.tables.result_086', 'result_086.txt')).write_text(get_value('science.messages.status_022', 'Supplementary Figure S3-1 / S3-2 (QC / preprocessing support)\n\nRadiomics limitations: ICC results, raw-feature counts and ICC-filtering logs are currently unavailable. Only the final model-ready feature count and feature-family composition are reported; no upstream values are inferred.\n\nBatch analysis: Blood and Urine only. POS and NEG use their respective batch identifiers to compare results before and after ComBat. No batches are fabricated for KO, Species or Radiomics.\n\nMain outputs:\n- Supplementary_Figure_S3_1_QC_and_feature_retention.*\n- Supplementary_Figure_S3_1_metabolomics_batch_correction.*\n- Supplementary_Figure_S3_2_low_dimensional_structure.*\n- Supplementary_Figure_S3_2_major_axis_drivers.*\n- input\n- input\n- input\n- input\n- input\n- input\n'), encoding='utf-8')

print(json.dumps(summary, ensure_ascii=False, indent=2))

